# FC Mold G5 - Data Analysis - TATA Ijmulden


**Goal**
- Identify stable casting sequences
- Remove sensor artifacts
- Relate mold level stability to process parameters

**Data**

CC23 has two strands: 23_5 and 23_6.
- dtExpert , data experts logged with a freq of 1Hz, it includes currents of EMBR parts, and casting parameters
- boExpert, BreakOut experts, logged with a freq of 2Hz, it consists of the raw FBG temperature measurements and some casting parameters

**Last updated**: 2025-06-xx


In [0]:
%pip install mpl-scatter-density
%pip install astropy

In [0]:
# Databricks notebook: PySpark + DBFS

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import time

from scipy.ndimage import median_filter
from pathlib import Path

import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
from matplotlib.colors import LinearSegmentedColormap
from astropy.visualization import LogStretch
from astropy.visualization.mpl_normalize import ImageNormalize
import plotly.express as px

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType, LongType, IntegerType, DoubleType, FloatType

from astropy.visualization import LogStretch
from astropy.visualization.mpl_normalize import ImageNormalize

norm = ImageNormalize(vmin=0., vmax=1000, stretch=LogStretch())

# --------------------------------------------------------------------
# Helper to recursively list files in DBFS and split boExpert / dtExpert
# --------------------------------------------------------------------
def add_plain_timestamp(df):
    """
    Ensure df has a 'plainTimeStamp' column of TimestampType, built from
    one of: plainTimeStamp, dt_timestamp_utc, TIMESTAMP.
    """
    schema = df.schema
    cols = df.columns

    # --- Case 1: plainTimeStamp already in data ---
    if "plainTimeStamp" in cols:
        dt = schema["plainTimeStamp"].dataType
        if isinstance(dt, StringType):
            df = df.withColumn("plainTimeStamp", F.to_timestamp("plainTimeStamp"))
        # if it's already TimestampType, do nothing
        return df

    # --- Case 2: dt_timestamp_utc exists (CSV case) ---
    if "dt_timestamp_utc" in cols:
        dt = schema["dt_timestamp_utc"].dataType

        if isinstance(dt, StringType):
            df = df.withColumn("plainTimeStamp", F.to_timestamp("dt_timestamp_utc"))

        elif isinstance(dt, TimestampType):
            df = df.withColumn("plainTimeStamp", F.col("dt_timestamp_utc"))

        elif isinstance(dt, (LongType, IntegerType, DoubleType, FloatType)):
            # numeric epoch: seconds vs milliseconds heuristic
            col = F.col("dt_timestamp_utc")
            df = df.withColumn(
                "plainTimeStamp",
                F.from_unixtime(
                    F.when(col > 1e12, col / 1000).otherwise(col)
                ).cast("timestamp")
            )

        return df

    # --- Case 3: TIMESTAMP exists (parquet case you showed) ---
    if "TIMESTAMP" in cols:
        dt = schema["TIMESTAMP"].dataType

        if isinstance(dt, TimestampType):
            df = df.withColumn("plainTimeStamp", F.col("TIMESTAMP"))

        elif isinstance(dt, (LongType, IntegerType, DoubleType, FloatType)):
            col = F.col("TIMESTAMP")
            df = df.withColumn(
                "plainTimeStamp",
                F.from_unixtime(
                    F.when(col > 1e12, col / 1000).otherwise(col)
                ).cast("timestamp")
            )

        return df

    # --- Fallback: no time column found ---
    return df

def get_expert_files(folder_path: str):
    """
    Databricks/DBFS version.

    folder_path: DBFS or mounted path, e.g. "dbfs:/mnt/tata_fcmoldg5/data/1393"
    Returns:
      bo_expert_files, dt_expert_files  (both lists of full DBFS paths)
    """
    bo_expert_files = []
    dt_expert_files = []

    def walk(path: str):
        for fi in dbutils.fs.ls(path):
            # In Databricks, directories usually end with '/'
            if fi.path.endswith('/'):
                walk(fi.path)
            else:
                name = fi.name
                if 'boExpert' in name:
                    bo_expert_files.append(fi.path)
                elif 'dtExpert' in name:
                    dt_expert_files.append(fi.path)

    walk(folder_path)
    return bo_expert_files, dt_expert_files


def load_expert_files(file_paths):
        """
        file_paths: list of DBFS paths (parquet and/or csv)
        Returns a Spark DataFrame with a proper 'plainTimeStamp' column if possible.
        """
        parquet_files = [f for f in file_paths if f.lower().endswith(".parquet")]
        csv_files     = [f for f in file_paths if f.lower().endswith(".csv")]

        if parquet_files:
            df = spark.read.parquet(*parquet_files)
        elif csv_files:
            df = (
                spark.read
                    .option("header", True)
                    .option("inferSchema", True)
                    .csv(csv_files)
            )
        else:
            return None

        df = add_plain_timestamp(df)
        return df

# Functions to convert units

from pyspark.sql import functions as F

def convert_units(df):
    """
    Convert units in the given Spark DataFrame:
      - castingSpeed:  (m/s → m/min) multiply by 60
      - Mold Level:    (m → mm)      multiply by 1000
      - Argon flows:   (m^3/s → L/min?) multiply by 60000

    Only applies conversions to columns that exist.
    """

    conversions = {
        "castingSpeed": 60,
        "Mold Level": 1000,
        "Mold Level Sensor Left": 1000,
        "Mold Level Sensor Right": 1000,
        "Argon Flow SEN": 60000,
        "Argon Flow Stopper": 60000 #,
        #"SENImmersionDepth": 100
    }

    df_conv = df

    for col, factor in conversions.items():
        if col in df.columns:
            df_conv = df_conv.withColumn(col, F.col(col) * factor)

    return df_conv



In [0]:

%fs ls dbfs:/FileStore/TATA_IJmuiden_CC23/data/


In [0]:
# Example base path in DBFS (adjust to your environment!)
strand_23_6_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/strand_6"

bo_files_23_6, dt_files_23_6 = get_expert_files(strand_23_6_path)


In [0]:
dt_files_23_6

## Load boExpert dtExpert files

In [0]:
# bo_files_23_6, dt_files_23_6 already built by get_expert_files(...)

df_boExpert_23_6 = load_expert_files(bo_files_23_6)
df_dtExpert_23_6 = load_expert_files(dt_files_23_6)

# Sanity checks
from pyspark.sql import functions as F

df_boExpert_23_6.select(
    F.count("*").alias("bo_total"),
    F.count("plainTimeStamp").alias("bo_plainTimeStamp_not_null")
).show()

df_dtExpert_23_6.select(
    F.count("*").alias("dt_total"),
    F.count("plainTimeStamp").alias("dt_plainTimeStamp_not_null")
).show()

df_boExpert_23_6.select("plainTimeStamp").show(5, False)
df_dtExpert_23_6.select("plainTimeStamp").show(5, False)


In [0]:
df_boExpert_23_6.printSchema()


In [0]:
df_boExpert_23_6.count()

In [0]:
df_boExpert_23_6.display()

## Join using plainTimeStamp

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

key_col = "plainTimeStamp"

# Identify numeric vs non-numeric columns (excluding the key)
numeric_cols = [
    f.name for f in df_boExpert_23_6.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != key_col
]

non_numeric_cols = [
    f.name for f in df_boExpert_23_6.schema.fields
    if not isinstance(f.dataType, NumericType) and f.name != key_col
]

print("Numeric columns to average:", numeric_cols)
print("Non-numeric columns (take first):", non_numeric_cols)

# Build aggregation expressions
agg_exprs = (
    [F.avg(c).alias(c) for c in numeric_cols] +
    [F.first(c, ignorenulls=True).alias(c) for c in non_numeric_cols]
)

# Aggregate bo on plainTimeStamp
df_boExpert_23_6_agg = (
    df_boExpert_23_6
    .groupBy(key_col)
    .agg(*agg_exprs)
)

# Sanity check: should now equal bo_distinct
print("df_boExpert_23_6_agg rows:", df_boExpert_23_6_agg.count())


In [0]:
from pyspark.sql import functions as F

# What columns exist and their types?
print(df_boExpert_23_6.dtypes)

# Non-null counts
df_boExpert_23_6.select(
    F.count("*").alias("bo_total"),
    F.count("plainTimeStamp").alias("plainTimeStamp_not_null"),
    F.count(F.when(F.col("plainTimeStamp").isNull(), 1)).alias("plainTimeStamp_nulls")
).show()

# Peek at *all* potential time columns
time_cols = [c for c, t in df_boExpert_23_6.dtypes if 'time' in c.lower() or 'timestamp' in c.lower()]
print("Time-like columns:", time_cols)
df_boExpert_23_6.select(*time_cols).show(20, truncate=False)


In [0]:
df_boExpert_23_6.count()

In [0]:
df_boExpert_23_6_agg.count()

In [0]:
df_dtExpert_23_6.display()

In [0]:
df_23_6_spark = (
    df_boExpert_23_6_agg.alias("bo")
    .join(df_dtExpert_23_6.alias("dt"), on=key_col, how="inner")
)

print("joined rows:", df_23_6_spark.count())


df_23_6_spark.cache()
df_23_6_spark.count()


In [0]:
print("bo rows:", df_boExpert_23_6.count())
print("dt rows:", df_dtExpert_23_6.count())
print("joined rows:", df_23_6_spark.count())


In [0]:
from pyspark.sql import functions as F

bo_total = df_boExpert_23_6.count()
dt_total = df_dtExpert_23_6.count()

bo_distinct = df_boExpert_23_6.select("plainTimeStamp").distinct().count()
dt_distinct = df_dtExpert_23_6.select("plainTimeStamp").distinct().count()

print("bo_total     :", bo_total)
print("bo_distinct  :", bo_distinct)
print("bo duplicates:", bo_total - bo_distinct)

print("dt_total     :", dt_total)
print("dt_distinct  :", dt_distinct)
print("dt duplicates:", dt_total - dt_distinct)


In [0]:
df_23_6_spark.count()

In [0]:
metadata_path = "dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv"
df_metadata_spark = spark.read.csv('dbfs:/FileStore/TATA_IJmuiden_CC23/data/Castings_TSN_2025_April_May_merged.csv', header=True, inferSchema=True, sep=";")
display(df_metadata_spark)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# =====================================================================
# Range-join sensor data with metadata to add Quality casting labels
# Key fixes:
#   - Filter metadata for Strand 6 only (prevents duplicates from
#     matching both Strand 5 & 6 overlapping time ranges)
#   - Direct timestamp comparison (session TZ = UTC, both sources
#     are in the same reference frame)
#   - Drop intermediate metadata columns after join
#
# Note: 'Datetime start last heat' is the START of the last heat,
# not the end of casting, so coverage is ~37% (underestimates actual
# casting duration).
# =====================================================================

row_count_before = df_23_6_spark.count()
print(f"Before metadata join: {row_count_before:,} rows")

# Drop Quality casting if it already exists (idempotent re-runs)
for c in ["Quality casting", "Datetime start first heat", "Datetime start last heat"]:
    if c in df_23_6_spark.columns:
        df_23_6_spark = df_23_6_spark.drop(c)

# Cast PlainTimeStamp to proper timestamp type
df_23_6_spark = df_23_6_spark.withColumn(
    "PlainTimeStamp",
    F.col("PlainTimeStamp").cast("timestamp")
)

# Prepare metadata: filter Strand 6 only
df_meta_strand6 = (
    df_metadata_spark
    .filter(F.col("Strand ID") == 6)
    .select(
        F.col("Datetime start first heat").cast("timestamp").alias("_meta_start"),
        F.col("Datetime start last heat").cast("timestamp").alias("_meta_end"),
        "Quality casting"
    )
    .filter(F.col("_meta_start").isNotNull() & F.col("_meta_end").isNotNull())
)

print(f"Metadata records for Strand 6: {df_meta_strand6.count()}")

# Range join: sensor timestamp within metadata casting period
join_cond = (
    (F.col("PlainTimeStamp") >= F.col("_meta_start")) &
    (F.col("PlainTimeStamp") <= F.col("_meta_end"))
)

df_23_6_spark = (
    df_23_6_spark
    .join(broadcast(df_meta_strand6), on=join_cond, how="left")
    .drop("_meta_start", "_meta_end")
)

row_count_after = df_23_6_spark.count()
matched = df_23_6_spark.filter(F.col("Quality casting").isNotNull()).count()

print(f"After metadata join:  {row_count_after:,} rows")
print(f"Row count change:     {row_count_after - row_count_before:+,} (should be 0)")
print(f"Quality casting matched: {matched:,} ({100*matched/row_count_after:.1f}%)")

if row_count_after != row_count_before:
    print(f"\n\u26a0\ufe0f  WARNING: Row count changed! Possible overlapping metadata periods.")
else:
    print(f"\n\u2713 No duplicates introduced by metadata join")

In [0]:
display(df_23_6_spark)

In [0]:
from pyspark.sql import functions as F

df_metadata_spark.select(
    "Datetime start first heat",
    "Datetime start last heat",
    F.col("Datetime start first heat").cast("timestamp").alias("first_cast"),
    F.col("Datetime start last heat").cast("timestamp").alias("last_cast")
).show(20, truncate=False)


In [0]:
from pyspark.sql import functions as F

def parse_ts_with_tz(colname):
    # example: "2025-05-01 21:56:21+02:00" -> "2025-05-01 21:56:21"
    return F.to_timestamp(
        F.regexp_replace(F.col(colname), r"\+\d{2}:\d{2}$", ""),
        "yyyy-MM-dd HH:mm:ss"
    )

df_metadata_fixed = (
    df_metadata_spark
    .withColumn("ts_start", parse_ts_with_tz("Datetime start first heat"))
    .withColumn("ts_end",   parse_ts_with_tz("Datetime start last heat"))
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# 1) Convert PlainTimeStamp (UTC) into local time to match metadata
df_23_6_local = df_23_6_spark.withColumn(
    "PlainTimeStamp_local",
    F.from_utc_timestamp(F.col("PlainTimeStamp"), "Europe/Amsterdam")
)

# Drop "Quality casting" if it exists to avoid duplicate columns in the join
if "Quality casting" in df_23_6_local.columns:
    df_23_6_local = df_23_6_local.drop("Quality casting")

# 2) Parse metadata timestamps and FILTER FOR STRAND 6 ONLY
df_metadata_fixed = (
    df_metadata_spark
    .filter(F.col("Strand ID") == 6)  # ← CRITICAL: Filter for Strand 6 only
    .withColumn("ts_start", F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ts_end",   F.to_timestamp("Datetime start last heat",  "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull())
)

print(f"Metadata records for Strand 6: {df_metadata_fixed.count()}")

# 3) Range join condition in SAME timezone reference (local)
join_condition = (
    (df_23_6_local["PlainTimeStamp_local"] >= df_metadata_fixed["ts_start"]) &
    (df_23_6_local["PlainTimeStamp_local"] <= df_metadata_fixed["ts_end"])
)

# 4) Join (broadcast metadata if small)
df_23_6_joined = (
    df_23_6_local
    .join(
        broadcast(df_metadata_fixed.select("ts_start", "ts_end", "Quality casting", "Casting ID")),
        on=join_condition,
        how="left"
    )
)

# 5) (Optional) drop helper cols
df_23_6_joined = df_23_6_joined.drop("ts_start", "ts_end")

print(f"\nJoin results:")
df_23_6_joined.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("Quality casting").isNotNull().cast("int")).alias("matched_rows"),
    F.countDistinct("Casting ID").alias("unique_castings")
).show()

In [0]:
df_23_6_joined.select(
    F.count("*").alias("rows"),
    F.sum(F.col("Quality casting").isNotNull().cast("int")).alias("matched_rows")
).show()

df_23_6_joined.select("PlainTimeStamp", "PlainTimeStamp_local", "Quality casting") \
    .where(F.col("Quality casting").isNotNull()) \
    .show(20, truncate=False)


In [0]:
%python
display(
    df_23_6_spark.groupBy(df_23_6_spark["Quality casting"]).count()
)

In [0]:
from pyspark.sql import functions as F

# Get statistics on null vs non-null Quality casting
quality_stats = df_23_6_spark.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("Quality casting").isNull(), 1).otherwise(0)).alias("null_quality"),
    F.sum(F.when(F.col("Quality casting").isNotNull(), 1).otherwise(0)).alias("non_null_quality")
).collect()[0]

print("="*70)
print("QUALITY CASTING DISTRIBUTION ANALYSIS")
print("="*70)
print(f"Total rows: {quality_stats['total_rows']:,}")
print(f"Null Quality casting: {quality_stats['null_quality']:,} ({100*quality_stats['null_quality']/quality_stats['total_rows']:.2f}%)")
print(f"Non-null Quality casting: {quality_stats['non_null_quality']:,} ({100*quality_stats['non_null_quality']/quality_stats['total_rows']:.2f}%)")
print("="*70)

# Get time range for null vs non-null
print("\nTIME RANGE COMPARISON:")
print("-"*70)

# Time range for NULL quality casting
null_time_range = df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    F.min("PlainTimeStamp").alias("first_null"),
    F.max("PlainTimeStamp").alias("last_null")
).collect()[0]

print(f"\nNull Quality casting time range:")
print(f"  First: {null_time_range['first_null']}")
print(f"  Last: {null_time_range['last_null']}")

# Time range for NON-NULL quality casting
non_null_time_range = df_23_6_spark.filter(F.col("Quality casting").isNotNull()).select(
    F.min("PlainTimeStamp").alias("first_non_null"),
    F.max("PlainTimeStamp").alias("last_non_null")
).collect()[0]

print(f"\nNon-null Quality casting time range:")
print(f"  First: {non_null_time_range['first_non_null']}")
print(f"  Last: {non_null_time_range['last_non_null']}")
print("="*70)

In [0]:
from pyspark.sql import functions as F

# Get metadata casting period ranges
metadata_range = df_metadata_spark.select(
    F.min(F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss")).alias("first_casting_start"),
    F.max(F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss")).alias("last_casting_end")
).collect()[0]

print("="*70)
print("METADATA CASTING PERIODS")
print("="*70)
print(f"First casting starts: {metadata_range['first_casting_start']}")
print(f"Last casting ends: {metadata_range['last_casting_end']}")
print("="*70)

# Get overall data time range
data_range = df_23_6_spark.select(
    F.min("PlainTimeStamp").alias("data_start"),
    F.max("PlainTimeStamp").alias("data_end")
).collect()[0]

print("\nOVERALL DATA TIME RANGE")
print("="*70)
print(f"Data starts: {data_range['data_start']}")
print(f"Data ends: {data_range['data_end']}")
print("="*70)

# Check if data extends beyond casting periods
print("\nANALYSIS:")
print("-"*70)
if data_range['data_start'] < metadata_range['first_casting_start']:
    print(f"⚠️  Data starts BEFORE first casting period")
    print(f"   Gap: {(metadata_range['first_casting_start'] - data_range['data_start']).total_seconds() / 3600:.2f} hours")
else:
    print(f"✓ Data starts within or after casting periods")

if data_range['data_end'] > metadata_range['last_casting_end']:
    print(f"⚠️  Data ends AFTER last casting period")
    print(f"   Gap: {(data_range['data_end'] - metadata_range['last_casting_end']).total_seconds() / 3600:.2f} hours")
else:
    print(f"✓ Data ends within or before casting periods")
print("="*70)

In [0]:
from pyspark.sql import functions as F, Window

# Get casting periods sorted by time
df_casting_periods_sorted = df_metadata_spark.select(
    F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss").alias("ts_start"),
    F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss").alias("ts_end"),
    "Casting ID",
    "Strand ID"
).filter(
    F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull()
).orderBy("ts_start")

# Calculate gaps between consecutive casting periods
window_spec = Window.orderBy("ts_start")

df_casting_gaps = df_casting_periods_sorted.withColumn(
    "prev_end",
    F.lag("ts_end").over(window_spec)
).withColumn(
    "gap_hours",
    (F.unix_timestamp("ts_start") - F.unix_timestamp("prev_end")) / 3600
).filter(
    F.col("gap_hours").isNotNull() & (F.col("gap_hours") > 0)
)

print("="*70)
print("GAPS BETWEEN CASTING PERIODS")
print("="*70)
print(f"Number of gaps between castings: {df_casting_gaps.count()}")
print("\nTop 10 largest gaps:")
df_casting_gaps.select(
    "prev_end",
    "ts_start",
    "gap_hours",
    "Casting ID",
    "Strand ID"
).orderBy(F.desc("gap_hours")).show(10, truncate=False)

# Calculate total gap time
total_gap_hours = df_casting_gaps.select(F.sum("gap_hours")).collect()[0][0]
print(f"\nTotal gap time between castings: {total_gap_hours:.2f} hours")
print("="*70)

In [0]:
from pyspark.sql import functions as F

# Sample some null Quality casting records to see their timestamps
print("="*70)
print("SAMPLE RECORDS WITH NULL QUALITY CASTING")
print("="*70)
print("\nFirst 20 records with null Quality casting:")
df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    "PlainTimeStamp",
    "Quality casting"
).orderBy("PlainTimeStamp").show(20, truncate=False)

print("\nLast 20 records with null Quality casting:")
df_23_6_spark.filter(F.col("Quality casting").isNull()).select(
    "PlainTimeStamp",
    "Quality casting"
).orderBy(F.desc("PlainTimeStamp")).show(20, truncate=False)
print("="*70)

## Analysis Summary: Null Quality Casting Values

### Key Findings:

**1. Distribution:**
* **45.74%** of measurements (324,079 rows) have **NULL** Quality casting
* **54.26%** of measurements (384,376 rows) have **non-null** Quality casting

**2. Temporal Analysis:**

**Null Quality Casting Time Range:**
* First: `2025-04-14 01:09:46`
* Last: `2025-05-16 05:36:25`
* Spans throughout the entire dataset

**Non-null Quality Casting Time Range:**
* First: `2025-04-13 22:59:26`
* Last: `2025-04-30 10:09:43`
* Limited to specific casting periods

**3. Root Cause:**

⚠️ **YES - Null values are from measurements OUTSIDE casting periods:**

* **Metadata casting periods:** April 13, 21:14:33 → May 5, 05:53:52
* **Actual data range:** April 13, 22:59:26 → **May 16, 05:36:25**
* **Data extends 263.71 hours (11 days) AFTER the last casting period**

**4. Explanation:**

The null `Quality casting` values occur because:

1. **Between casting periods:** 26 gaps totaling 410.73 hours between consecutive castings
   - Largest gap: 78.6 hours (April 18 → April 21)
   - During these gaps, the MEX system continues recording data but no casting is active

2. **After last casting:** 263.71 hours of data after May 5, 05:53:52
   - System continues logging even though no casting metadata exists
   - This is the majority of null values

3. **Before first casting:** Minimal (data starts within casting periods)

### Recommendation:

For your **mold level fluctuation analysis**, you should:
* **Filter out null Quality casting rows** - they represent non-production periods
* **Focus on sequences with valid Quality casting** - these are actual casting operations
* **Use Quality casting as a grouping variable** for sequence-based analysis

In [0]:
import plotly.graph_objects as go
import pandas as pd
from pyspark.sql import functions as F

# Sample data for visualization (to avoid memory issues)
sample_rate = max(1, df_23_6_spark.count() // 10000)

df_sample = df_23_6_spark.sample(fraction=1.0/sample_rate, seed=42).select(
    "PlainTimeStamp",
    F.when(F.col("Quality casting").isNull(), "No Casting (NULL)").otherwise("Active Casting").alias("status"),
    "Quality casting"
).orderBy("PlainTimeStamp").toPandas()

# Create figure
fig = go.Figure()

# Add traces for null and non-null
for status in df_sample['status'].unique():
    df_status = df_sample[df_sample['status'] == status]
    color = 'red' if status == 'No Casting (NULL)' else 'green'
    
    fig.add_trace(go.Scatter(
        x=df_status['PlainTimeStamp'],
        y=[1] * len(df_status),
        mode='markers',
        marker=dict(color=color, size=3, opacity=0.5),
        name=status,
        hovertemplate='%{x}<br>Status: ' + status + '<extra></extra>'
    ))

# Add metadata casting period boundaries
metadata_periods = df_metadata_spark.select(
    F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss").alias("start"),
    F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss").alias("end"),
    "Casting ID"
).filter(F.col("start").isNotNull()).toPandas()

for idx, row in metadata_periods.iterrows():
    fig.add_vrect(
        x0=row['start'], x1=row['end'],
        fillcolor='lightblue', opacity=0.1,
        layer="below", line_width=0
    )

fig.update_layout(
    title=dict(
        text='Temporal Distribution: Null vs Non-Null Quality Casting',
        font=dict(size=16, color='darkblue')
    ),
    xaxis_title='Timeline',
    yaxis_title='Data Presence',
    height=500,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.15,
        xanchor="center",
        x=0.5
    ),
    yaxis=dict(showticklabels=False, range=[0.5, 1.5]),
    hovermode='closest'
)

fig.show()

print("\n" + "="*70)
print("VISUALIZATION EXPLANATION")
print("="*70)
print("• GREEN dots: Measurements during active casting (with Quality casting)")
print("• RED dots: Measurements outside casting periods (NULL Quality casting)")
print("• Light blue bands: Metadata-defined casting periods")
print("\nNotice: Red dots appear in gaps between blue bands and after the last band")
print("="*70)

## Features engineering

In [0]:
from pyspark.sql import functions as F
import functools

# =====================================================================
# 1. MENISCUS TEMPERATURE (raw FBG averages across mold width)
#    meniscus_bff_c01-c20: 20 sensors on fixed face
#    meniscus_bfl_c01-c20: 20 sensors on loose face
#    Computed: row-wise mean, then FF-LF difference, and range (max-min)
# =====================================================================
bff_cols = [f"meniscus_bff_c{i:02d}" for i in range(1, 21)]
bfl_cols = [f"meniscus_bfl_c{i:02d}" for i in range(1, 21)]

# Row-wise mean (counting only non-null sensors)
bff_sum = functools.reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0)) for c in bff_cols])
bff_cnt = functools.reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in bff_cols])
bfl_sum = functools.reduce(lambda a, b: a + b, [F.coalesce(F.col(c), F.lit(0)) for c in bfl_cols])
bfl_cnt = functools.reduce(lambda a, b: a + b, [F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in bfl_cols])

df_features = df_23_6_spark_unitConverted \
    .withColumn("meniscus_bff_avg", F.when(bff_cnt > 0, bff_sum / bff_cnt)) \
    .withColumn("meniscus_bfl_avg", F.when(bfl_cnt > 0, bfl_sum / bfl_cnt)) \
    .withColumn("meniscus_FF_LF_asymmetry", F.col("meniscus_bff_avg") - F.col("meniscus_bfl_avg"))

# Temperature spread across width (max - min of 20 sensors per face)
df_features = df_features \
    .withColumn("meniscus_bff_range", F.greatest(*[F.col(c) for c in bff_cols]) - F.least(*[F.col(c) for c in bff_cols])) \
    .withColumn("meniscus_bfl_range", F.greatest(*[F.col(c) for c in bfl_cols]) - F.least(*[F.col(c) for c in bfl_cols]))

# =====================================================================
# 2. MENISCUS FLATNESS — Chebyshev Polynomial Coefficients (X1-X4)
#    From 4th order Chebyshev fit to meniscus height profile:
#      f(x) = X4*T4(x) + X3*T3(x) + X2*T2(x) + X1*T1(x) + X0
#    _1 = X1 (tilt/asymmetry), _2 = X2 (curvature/standing wave)
#    _3 = X3 (3rd order asymmetry), _4 = X4 (4th order ripple)
#    BFF = broad/fixed face, BFL = loose face
# =====================================================================
df_features = df_features \
    .withColumn("cheb_X1_bff", F.col("meniscusFlatness_bff_1")) \
    .withColumn("cheb_X2_bff", F.col("meniscusFlatness_bff_2")) \
    .withColumn("cheb_X3_bff", F.col("meniscusFlatness_bff_3")) \
    .withColumn("cheb_X4_bff", F.col("meniscusFlatness_bff_4")) \
    .withColumn("cheb_X1_bfl", F.col("meniscusFlatness_bfl_1")) \
    .withColumn("cheb_X2_bfl", F.col("meniscusFlatness_bfl_2")) \
    .withColumn("cheb_X3_bfl", F.col("meniscusFlatness_bfl_3")) \
    .withColumn("cheb_X4_bfl", F.col("meniscusFlatness_bfl_4"))

# Absolute values (magnitude of shape deformation regardless of direction)
df_features = df_features \
    .withColumn("abs_cheb_X1_bff", F.abs(F.col("cheb_X1_bff"))) \
    .withColumn("abs_cheb_X2_bff", F.abs(F.col("cheb_X2_bff"))) \
    .withColumn("abs_cheb_X1_bfl", F.abs(F.col("cheb_X1_bfl"))) \
    .withColumn("abs_cheb_X2_bfl", F.abs(F.col("cheb_X2_bfl")))

# BFF - BFL differences per coefficient (fixed-face vs loose-face shape)
df_features = df_features \
    .withColumn("cheb_X1_FF_LF_diff", F.col("cheb_X1_bff") - F.col("cheb_X1_bfl")) \
    .withColumn("cheb_X2_FF_LF_diff", F.col("cheb_X2_bff") - F.col("cheb_X2_bfl"))

# =====================================================================
# 3. EMBR LEFT-RIGHT CURRENT IMBALANCE
# =====================================================================
df_features = df_features \
    .withColumn("EMBR_DC_Bottom_LR_diff",
                F.col("EMBR Current DC Left Bottom") - F.col("EMBR Current DC Right Bottom")) \
    .withColumn("EMBR_AC_Master_LR_diff",
                F.col("EMBR Current AC Left Master") - F.col("EMBR Current AC Right Master")) \
    .withColumn("EMBR_DC_Master_LR_diff",
                F.col("EMBR Current DC Left Master") - F.col("EMBR Current DC Right Master"))

# =====================================================================
# 4. MOLD LEVEL LEFT-RIGHT ASYMMETRY
# =====================================================================
df_features = df_features \
    .withColumn("ML_LR_asymmetry",
                F.col("Mold Level Sensor Left") - F.col("Mold Level Sensor Right")) \
    .withColumn("ML_LR_abs_asymmetry",
                F.abs(F.col("Mold Level Sensor Left") - F.col("Mold Level Sensor Right")))

# =====================================================================
# 5. COLUMN NORMALISATION — ensure consistent naming downstream
# =====================================================================
if "PlainTimeStamp" in df_features.columns and "plainTimeStamp" not in df_features.columns:
    df_features = df_features.withColumnRenamed("PlainTimeStamp", "plainTimeStamp")

print("Feature engineering complete.")
print(f"Total rows: {df_features.count():,}")
print(f"Total columns: {len(df_features.columns)}")

# Verify all key columns propagated from upstream
for c in ["plainTimeStamp", "Quality casting", "castingSpeed", "meniscus_bff_avg", "cheb_X2_bff", "ML_LR_asymmetry"]:
    status = "\u2713" if c in df_features.columns else "\u2717 MISSING"
    print(f"  {status} {c}")

# Quick stats on Chebyshev features
cheb_cols = ["cheb_X1_bff", "cheb_X2_bff", "cheb_X3_bff", "cheb_X4_bff",
             "cheb_X1_bfl", "cheb_X2_bfl", "cheb_X3_bfl", "cheb_X4_bfl",
             "abs_cheb_X1_bff", "abs_cheb_X2_bff"]
df_features.select(cheb_cols).describe().show(truncate=False)

In [0]:

df_23_6_spark_unitConverted = convert_units(df_23_6_spark)
#df_23_6_spark_unitConverted = convert_argon_flow_to_l_per_min(df_23_6_spark_unitConverted)
display(df_23_6_spark_unitConverted)

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Sort by timestamp and calculate time difference between consecutive rows
window_spec = Window.orderBy("PlainTimeStamp")

df_time_gaps = (
    df_23_6_spark_unitConverted
    .select("PlainTimeStamp")
    .withColumn("prev_timestamp", F.lag("PlainTimeStamp").over(window_spec))
    .withColumn(
        "time_diff_seconds",
        F.unix_timestamp("PlainTimeStamp") - F.unix_timestamp("prev_timestamp")
    )
    .filter(F.col("time_diff_seconds").isNotNull())
)

# Show statistics about time gaps
df_time_gaps.select(
    F.min("time_diff_seconds").alias("min_gap_sec"),
    F.max("time_diff_seconds").alias("max_gap_sec"),
    F.avg("time_diff_seconds").alias("avg_gap_sec"),
    F.expr("percentile(time_diff_seconds, 0.5)").alias("median_gap_sec"),
    F.expr("percentile(time_diff_seconds, 0.95)").alias("p95_gap_sec")
).show()

In [0]:
# Define threshold for "significant" gap (e.g., > 5 minutes = 300 seconds)
threshold_seconds = 300

df_significant_gaps = (
    df_time_gaps
    .filter(F.col("time_diff_seconds") > threshold_seconds)
    .withColumn("time_diff_minutes", F.col("time_diff_seconds") / 60)
    .withColumn("time_diff_hours", F.col("time_diff_seconds") / 3600)
    .orderBy(F.desc("time_diff_seconds"))
)

print(f"Found {df_significant_gaps.count()} gaps larger than {threshold_seconds} seconds ({threshold_seconds/60} minutes)")
display(df_significant_gaps.select(
    "prev_timestamp",
    "PlainTimeStamp",
    "time_diff_seconds",
    "time_diff_minutes",
    "time_diff_hours"
).limit(20))

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Convert to pandas for plotting
df_gaps_pd = df_time_gaps.select("PlainTimeStamp", "time_diff_seconds").toPandas()

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Time Series Gap Analysis', fontsize=16, fontweight='bold')

# 1. Histogram of time gaps (log scale)
ax1 = axes[0, 0]
ax1.hist(df_gaps_pd['time_diff_seconds'], bins=100, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Time Gap (seconds)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Time Gaps')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# 2. Time gaps over time
ax2 = axes[0, 1]
ax2.scatter(df_gaps_pd['PlainTimeStamp'], df_gaps_pd['time_diff_seconds'], 
            alpha=0.5, s=1)
ax2.set_xlabel('Timestamp')
ax2.set_ylabel('Time Gap (seconds)')
ax2.set_title('Time Gaps Over Time')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# 3. Cumulative distribution of gaps
ax3 = axes[1, 0]
sorted_gaps = np.sort(df_gaps_pd['time_diff_seconds'])
cumulative = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps) * 100
ax3.plot(sorted_gaps, cumulative, linewidth=2)
ax3.set_xlabel('Time Gap (seconds)')
ax3.set_ylabel('Cumulative Percentage (%)')
ax3.set_title('Cumulative Distribution of Time Gaps')
ax3.set_xscale('log')
ax3.grid(True, alpha=0.3)
ax3.axhline(y=95, color='r', linestyle='--', label='95th percentile')
ax3.legend()

# 4. Box plot of time gaps (filtered to show detail)
ax4 = axes[1, 1]
# Filter out extreme outliers for better visualization
q99 = df_gaps_pd['time_diff_seconds'].quantile(0.99)
filtered_gaps = df_gaps_pd[df_gaps_pd['time_diff_seconds'] <= q99]['time_diff_seconds']
ax4.boxplot(filtered_gaps, vert=True)
ax4.set_ylabel('Time Gap (seconds)')
ax4.set_title(f'Box Plot of Time Gaps (up to 99th percentile: {q99:.1f}s)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/time_gaps_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal records analyzed: {len(df_gaps_pd):,}")
print(f"Mean gap: {df_gaps_pd['time_diff_seconds'].mean():.2f} seconds")
print(f"Median gap: {df_gaps_pd['time_diff_seconds'].median():.2f} seconds")
print(f"Max gap: {df_gaps_pd['time_diff_seconds'].max():.2f} seconds ({df_gaps_pd['time_diff_seconds'].max()/3600:.2f} hours)")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Get significant gaps for visualization
df_sig_gaps_pd = df_significant_gaps.select(
    "prev_timestamp", 
    "PlainTimeStamp", 
    "time_diff_hours"
).toPandas()

# Create timeline visualization
fig, ax = plt.subplots(figsize=(16, 6))

# Plot the continuous timeline
if len(df_gaps_pd) > 0:
    # Sample data points for timeline (every 100th point to avoid overcrowding)
    sample_indices = range(0, len(df_gaps_pd), max(1, len(df_gaps_pd) // 1000))
    timeline_sample = df_gaps_pd.iloc[sample_indices]
    
    ax.scatter(timeline_sample['PlainTimeStamp'], 
               [1] * len(timeline_sample), 
               alpha=0.3, s=10, c='blue', label='Data points')

# Mark significant gaps
if len(df_sig_gaps_pd) > 0:
    for idx, row in df_sig_gaps_pd.iterrows():
        ax.axvspan(row['prev_timestamp'], row['PlainTimeStamp'], 
                   alpha=0.3, color='red')
        # Add text label for large gaps
        if row['time_diff_hours'] > 1:
            mid_time = row['prev_timestamp'] + (row['PlainTimeStamp'] - row['prev_timestamp']) / 2
            ax.text(mid_time, 1.1, f"{row['time_diff_hours']:.1f}h", 
                   rotation=90, ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Timestamp', fontsize=12)
ax.set_ylabel('Data Presence', fontsize=12)
ax.set_title(f'Time Series Timeline with Gaps > {threshold_seconds/60} minutes (Red Regions)', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.5)
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.xticks(rotation=45, ha='right')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.3, label='Data points'),
    Patch(facecolor='red', alpha=0.3, label=f'Gaps > {threshold_seconds/60} min')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
fig.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/timeline_with_gaps.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization shows {len(df_sig_gaps_pd)} significant gaps in the timeline")

In [0]:
from pyspark.sql import functions as F

# Parse metadata timestamps and filter valid records
df_casting_periods = (
    df_metadata_spark
    .withColumn("ts_start", F.to_timestamp("Datetime start first heat", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ts_end", F.to_timestamp("Datetime start last heat", "yyyy-MM-dd HH:mm:ss"))
    .filter(F.col("ts_start").isNotNull() & F.col("ts_end").isNotNull())
    .select(
        "Casting ID",
        "Strand ID",
        "ts_start",
        "ts_end",
        "Quality casting",
        "Heat Count"
    )
    .orderBy("ts_start")
)

print(f"Total casting periods: {df_casting_periods.count()}")
display(df_casting_periods.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Join gaps with casting periods to classify them
df_gaps_classified = df_significant_gaps.alias("gaps").join(
    broadcast(df_casting_periods.alias("cast")),
    (
        (F.col("gaps.prev_timestamp") >= F.col("cast.ts_start")) &
        (F.col("gaps.prev_timestamp") <= F.col("cast.ts_end")) &
        (F.col("gaps.PlainTimeStamp") >= F.col("cast.ts_start")) &
        (F.col("gaps.PlainTimeStamp") <= F.col("cast.ts_end"))
    ),
    how="left"
).select(
    F.col("gaps.prev_timestamp"),
    F.col("gaps.PlainTimeStamp"),
    F.col("gaps.time_diff_hours"),
    F.col("cast.Casting ID").alias("casting_id"),
    F.col("cast.Strand ID").alias("strand_id"),
    F.col("cast.Quality casting").alias("quality"),
    F.when(F.col("cast.Casting ID").isNotNull(), "Within Casting")
     .otherwise("Between Castings").alias("gap_type")
)

print("\nGap Classification:")
df_gaps_classified.groupBy("gap_type").agg(
    F.count("*").alias("count"),
    F.sum("time_diff_hours").alias("total_hours"),
    F.avg("time_diff_hours").alias("avg_hours")
).show()

print("\nGaps WITHIN casting periods (potential issues):")
display(
    df_gaps_classified
    .filter(F.col("gap_type") == "Within Casting")
    .orderBy(F.desc("time_diff_hours"))
)

In [0]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from datetime import datetime
import os

# Convert casting periods to pandas
df_casting_pd = df_casting_periods.toPandas()
df_gaps_class_pd = df_gaps_classified.toPandas()

# Filter for Strand 6 only (Strand 5 data not loaded yet)
df_casting_pd = df_casting_pd[df_casting_pd['Strand ID'] == 5].reset_index(drop=True)

# ---- helpers to restrict events to Strand-6 casting windows ----
# Build IntervalIndex of Strand-6 casting periods
casting_intervals = pd.IntervalIndex.from_arrays(
    df_casting_pd["ts_start"],
    df_casting_pd["ts_end"],
    closed="both"
)

def in_any_casting_window(ts_series):
    """Return boolean mask: timestamp is inside any Strand-6 casting interval."""
    # get_indexer gives -1 when not contained in any interval
    return casting_intervals.get_indexer(ts_series) >= 0

def gap_overlaps_any_casting(row):
    """True if [prev_timestamp, PlainTimeStamp] overlaps ANY Strand-6 casting window."""
    a = row["prev_timestamp"]
    b = row["PlainTimeStamp"]
    if pd.isna(a) or pd.isna(b):
        return False
    # overlap test with all intervals: overlap exists if max(starts) < min(ends)
    # here do it simply: does either end fall inside, or does interval cover a casting boundary
    if in_any_casting_window(pd.Series([a]))[0] or in_any_casting_window(pd.Series([b]))[0]:
        return True
    # also catch gaps spanning across a casting window without endpoints inside
    return ((df_casting_pd["ts_start"] <= b) & (df_casting_pd["ts_end"] >= a)).any()




# Ensure timestamp columns are datetime objects (not Timedelta)
for col in ['ts_start', 'ts_end']:
    if col in df_casting_pd.columns:
        df_casting_pd[col] = pd.to_datetime(df_casting_pd[col])

for col in ['prev_timestamp', 'PlainTimeStamp']:
    if col in df_gaps_class_pd.columns:
        df_gaps_class_pd[col] = pd.to_datetime(df_gaps_class_pd[col])

if 'PlainTimeStamp' in df_gaps_pd.columns:
    df_gaps_pd['PlainTimeStamp'] = pd.to_datetime(df_gaps_pd['PlainTimeStamp'])

# Create output directory if it doesn't exist
output_dir = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/'
os.makedirs(output_dir, exist_ok=True)

# ========================================================================
# FIGURE 1: Casting Periods as Horizontal Timeline (Gantt-style)
# ========================================================================

# Prepare data for Gantt chart using plotly express
df_gantt = df_casting_pd.copy()
df_gantt['Casting_Label'] = df_gantt['Casting ID'].apply(lambda x: f"Casting {int(x)}")
df_gantt['Duration_hours'] = (df_gantt['ts_end'] - df_gantt['ts_start']).dt.total_seconds() / 3600

# Create Gantt chart using plotly express timeline
fig1 = px.timeline(
    df_gantt,
    x_start='ts_start',
    x_end='ts_end',
    y='Casting_Label',
    color_discrete_sequence=['steelblue'],
    title='Casting Periods Timeline - Strand 6',
    labels={'Casting_Label': 'Casting ID'},
    hover_data={
        'Quality casting': True,
        'Heat Count': True,
        'ts_start': '|%Y-%m-%d %H:%M',
        'ts_end': '|%Y-%m-%d %H:%M',
        'Duration_hours': ':.1f',
        'Casting_Label': False
    }
)

# Add quality labels alternating above and below bars
for idx, row in df_gantt.iterrows():
    # Calculate middle of the bar for x position
    mid_time = row['ts_start'] + (row['ts_end'] - row['ts_start']) / 2
    
    # Alternate position: even indices above, odd indices below
    if idx % 2 == 0:
        # Position above the bar
        y_offset = 25  # pixels above
        arrow_direction = -15  # arrow points down to bar
    else:
        # Position below the bar
        y_offset = -25  # pixels below
        arrow_direction = 15  # arrow points up to bar
    
    fig1.add_annotation(
        x=mid_time,
        y=row['Casting_Label'],
        text=row['Quality casting'],
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        arrowcolor='gray',
        ax=0,  # No horizontal offset
        ay=arrow_direction,  # Vertical arrow pointing to bar
        font=dict(size=10, color='darkblue', weight='bold'),
        xanchor='center',
        yanchor='middle',
        yshift=y_offset,  # Shift text up or down
        bgcolor='rgba(255, 255, 255, 0.8)',  # White background for readability
        bordercolor='lightgray',
        borderwidth=1,
        borderpad=3
    )

fig1.update_layout(
    xaxis_title='Date and Time',
    yaxis_title='Casting ID',
    height=max(600, len(df_casting_pd) * 25),
    showlegend=False,
    hovermode='closest',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    xaxis=dict(
        showgrid=True,
        gridcolor='white',
        gridwidth=1,
        type='date'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='white',
        gridwidth=1,
        autorange='reversed'
    ),
    title_font=dict(size=18, color='darkblue'),
    margin=dict(t=100, b=80)  # Add top and bottom margins for labels
)

# Update bar appearance
fig1.update_traces(
    marker=dict(opacity=0.7, line=dict(color='darkblue', width=1))
)

fig1.show()

# Save Figure 1
fig1_path = os.path.join(output_dir, 'casting_periods_timeline_strand6.html')
fig1.write_html(fig1_path)
print(f"\n✓ Figure 1 saved to: {fig1_path}")

print(f"\n{'='*70}")
print(f"FIGURE 1: Casting Periods Overview (Strand 6)")
print(f"{'='*70}")
print(f"Total casting periods: {len(df_casting_pd)}")
print(f"Date range: {df_casting_pd['ts_start'].min().strftime('%Y-%m-%d')} to {df_casting_pd['ts_end'].max().strftime('%Y-%m-%d')}")
print(f"Total casting time: {(df_casting_pd['ts_end'] - df_casting_pd['ts_start']).sum().total_seconds() / 3600:.1f} hours")


# df_gaps_pd must exist; if not, skip
if "PlainTimeStamp" in df_gaps_pd.columns:
    df_gaps_pd = df_gaps_pd.copy()
    df_gaps_pd["PlainTimeStamp"] = pd.to_datetime(df_gaps_pd["PlainTimeStamp"], errors="coerce")

    # keep only points inside Strand-6 casting windows
    df_gaps_pd = df_gaps_pd[in_any_casting_window(df_gaps_pd["PlainTimeStamp"])].reset_index(drop=True)

df_gaps_class_pd = df_gaps_class_pd.copy()

# Ensure datetimes (again safe)
df_gaps_class_pd["prev_timestamp"] = pd.to_datetime(df_gaps_class_pd["prev_timestamp"], errors="coerce")
df_gaps_class_pd["PlainTimeStamp"] = pd.to_datetime(df_gaps_class_pd["PlainTimeStamp"], errors="coerce")

# Keep only gaps that overlap Strand-6 casting windows
df_gaps_class_pd = df_gaps_class_pd[
    df_gaps_class_pd.apply(gap_overlaps_any_casting, axis=1)
].reset_index(drop=True)

# OPTIONAL: recompute gaps BETWEEN Strand-6 castings from df_casting_pd
df_casting_sorted = df_casting_pd.sort_values("ts_start").reset_index(drop=True)

between_rows = []
for i in range(1, len(df_casting_sorted)):
    prev_end = df_casting_sorted.loc[i-1, "ts_end"]
    cur_start = df_casting_sorted.loc[i, "ts_start"]
    if cur_start > prev_end:
        between_rows.append({
            "prev_timestamp": prev_end,
            "PlainTimeStamp": cur_start,
            "time_diff_hours": (cur_start - prev_end).total_seconds() / 3600.0,
            "gap_type": "Between Castings",
            "casting_id": None
        })

df_between_strand6 = pd.DataFrame(between_rows)



# ========================================================================
# FIGURE 2: Data Points and Gaps Timeline
# ========================================================================
fig2 = go.Figure()

# Add casting period background bands (without annotations to avoid timestamp arithmetic issues)
for idx, row in df_casting_pd.iterrows():
    fig2.add_vrect(
        x0=row['ts_start'], x1=row['ts_end'],
        fillcolor='lightblue', opacity=0.25,
        layer="below", line_width=0
    )

# Add gaps between castings (orange) and within castings (red)
if len(df_gaps_class_pd) > 0:
    #gaps_between = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Between Castings']
    gaps_between = df_between_strand6
    gaps_within = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Within Casting']
    
    # Add gap rectangles for gaps between castings
    for idx, row in gaps_between.iterrows():
        fig2.add_trace(
            go.Scatter(
                x=[row['prev_timestamp'], row['PlainTimeStamp'], row['PlainTimeStamp'], row['prev_timestamp'], row['prev_timestamp']],
                y=[0.5, 0.5, 1.5, 1.5, 0.5],
                fill='toself',
                fillcolor='orange',
                opacity=0.4,
                line=dict(width=0),
                mode='lines',
                showlegend=False,
                hovertemplate=f"<b>Gap Between Castings</b><br>" +
                             f"Duration: {row['time_diff_hours']:.1f}h<br>" +
                             f"From: {row['prev_timestamp'].strftime('%Y-%m-%d %H:%M')}<br>" +
                             f"To: {row['PlainTimeStamp'].strftime('%Y-%m-%d %H:%M')}<extra></extra>"
            )
        )
    
    # Add gap rectangles for gaps within castings
    for idx, row in gaps_within.iterrows():
        fig2.add_trace(
            go.Scatter(
                x=[row['prev_timestamp'], row['PlainTimeStamp'], row['PlainTimeStamp'], row['prev_timestamp'], row['prev_timestamp']],
                y=[0.5, 0.5, 1.5, 1.5, 0.5],
                fill='toself',
                fillcolor='red',
                opacity=0.6,
                line=dict(width=0),
                mode='lines',
                showlegend=False,
                hovertemplate=f"<b>⚠️ Gap WITHIN Casting (Data Loss!)</b><br>" +
                             f"Duration: {row['time_diff_hours']:.1f}h<br>" +
                             f"Casting: {int(row['casting_id']) if pd.notna(row['casting_id']) else 'Unknown'}<br>" +
                             f"From: {row['prev_timestamp'].strftime('%Y-%m-%d %H:%M')}<br>" +
                             f"To: {row['PlainTimeStamp'].strftime('%Y-%m-%d %H:%M')}<extra></extra>"
            )
        )

# Add data points (sampled)
if len(df_gaps_pd) > 0:
    sample_indices = range(0, len(df_gaps_pd), max(1, len(df_gaps_pd) // 3000))
    timeline_sample = df_gaps_pd.iloc[sample_indices]
    
    fig2.add_trace(
        go.Scatter(
            x=timeline_sample['PlainTimeStamp'],
            y=[1] * len(timeline_sample),
            mode='markers',
            marker=dict(color='green', size=2, opacity=0.3),
            name='Data points',
            hovertemplate='Data point at: %{x}<extra></extra>'
        )
    )

# Add legend items
fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='lightblue', opacity=0.25, line=dict(color='steelblue', width=1)),
        name='Casting period',
        showlegend=True
    )
)

fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='red', opacity=0.6),
        name='⚠️ Gap WITHIN casting (data loss)',
        showlegend=True
    )
)

fig2.add_trace(
    go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=12, color='orange', opacity=0.4),
        name='Gap BETWEEN castings (expected)',
        showlegend=True
    )
)

fig2.update_layout(
    title=dict(
        text='Data Collection Timeline with Gaps - Strand 6',
        font=dict(size=18, color='darkblue')
    ),
    xaxis_title='Date and Time',
    yaxis_title='Data Presence',
    height=600,
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.12,
        xanchor="center",
        x=0.5,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="black",
        borderwidth=1
    ),
    hovermode='closest',
    yaxis=dict(
        showticklabels=False,
        range=[0.4, 1.6],
        showgrid=False
    ),
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgray',
        gridwidth=0.5,
        type='date'
    ),
    plot_bgcolor='white'
)

fig2.show()

# Save Figure 2
fig2_path = os.path.join(output_dir, 'data_quality_timeline_strand6.html')
fig2.write_html(fig2_path)
print(f"\n✓ Figure 2 saved to: {fig2_path}")

print(f"\n{'='*70}")
print(f"FIGURE 2: Data Quality Analysis (Strand 6)")
print(f"{'='*70}")
if len(df_gaps_class_pd) > 0:
    gaps_within = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Within Casting']
    gaps_between = df_gaps_class_pd[df_gaps_class_pd['gap_type'] == 'Between Castings']
    print(f"✓ Data points analyzed: {len(df_gaps_pd):,}")
    print(f"⚠️  Gaps WITHIN castings: {len(gaps_within)} (RED - data collection issues)")
    print(f"✓ Gaps BETWEEN castings: {len(gaps_between)} (ORANGE - expected downtime)")
    if len(gaps_within) > 0:
        total_loss_hours = gaps_within['time_diff_hours'].sum()
        print(f"⚠️  Total data loss during casting: {total_loss_hours:.1f} hours")
else:
    print(f"✓ No significant gaps found")
print(f"{'='*70}\n")

print(f"\n{'='*70}")
print(f"SAVED FILES")
print(f"{'='*70}")
print(f"Figure 1: {fig1_path}")
print(f"Figure 2: {fig2_path}")
print(f"\nAccess via: https://<databricks-instance>/files/TATAIjmulden_FCMoldG5/figures/")
print(f"{'='*70}\n")

In [0]:
display(dbutils.fs.ls('dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/'))

In [0]:
from pyspark.sql import functions as F

# Find data points that fall outside any casting period
df_data_with_casting = (
    df_23_6_spark_unitConverted
    .select("PlainTimeStamp")
    .alias("data")
    .join(
        broadcast(df_casting_periods.alias("cast")),
        (
            (F.col("data.PlainTimeStamp") >= F.col("cast.ts_start")) &
            (F.col("data.PlainTimeStamp") <= F.col("cast.ts_end"))
        ),
        how="left"
    )
)

# Count matched vs unmatched
matching_stats = df_data_with_casting.select(
    F.count("*").alias("total_data_points"),
    F.sum(F.when(F.col("Casting ID").isNotNull(), 1).otherwise(0)).alias("matched_to_casting"),
    F.sum(F.when(F.col("Casting ID").isNull(), 1).otherwise(0)).alias("unmatched_orphan_data")
).collect()[0]

print("\n" + "="*60)
print("DATA-METADATA MISMATCH ANALYSIS")
print("="*60)
print(f"\nTotal data points: {matching_stats['total_data_points']:,}")
print(f"Matched to casting periods: {matching_stats['matched_to_casting']:,} ({100*matching_stats['matched_to_casting']/matching_stats['total_data_points']:.2f}%)")
print(f"Orphan data (outside any casting): {matching_stats['unmatched_orphan_data']:,} ({100*matching_stats['unmatched_orphan_data']/matching_stats['total_data_points']:.2f}%)")

# Show time range of orphan data
orphan_data = df_data_with_casting.filter(F.col("Casting ID").isNull())
if orphan_data.count() > 0:
    orphan_range = orphan_data.select(
        F.min("PlainTimeStamp").alias("first_orphan"),
        F.max("PlainTimeStamp").alias("last_orphan")
    ).collect()[0]
    
    print(f"\nOrphan data time range:")
    print(f"  First: {orphan_range['first_orphan']}")
    print(f"  Last: {orphan_range['last_orphan']}")

# Check for casting periods with no data
print("\n" + "-"*60)
print("Casting periods with data coverage:")
print("-"*60)

casting_coverage = (
    df_casting_periods.alias("cast")
    .join(
        df_23_6_spark_unitConverted.select("PlainTimeStamp").alias("data"),
        (
            (F.col("data.PlainTimeStamp") >= F.col("cast.ts_start")) &
            (F.col("data.PlainTimeStamp") <= F.col("cast.ts_end"))
        ),
        how="left"
    )
    .groupBy("Casting ID", "Strand ID", "ts_start", "ts_end", "Quality casting")
    .agg(
        F.count("PlainTimeStamp").alias("data_point_count"),
        F.min("PlainTimeStamp").alias("first_data"),
        F.max("PlainTimeStamp").alias("last_data")
    )
    .withColumn(
        "coverage_status",
        F.when(F.col("data_point_count") == 0, "NO DATA")
         .when(F.col("first_data") > F.col("ts_start"), "LATE START")
         .when(F.col("last_data") < F.col("ts_end"), "EARLY END")
         .otherwise("FULL COVERAGE")
    )
    .orderBy("ts_start")
)

display(casting_coverage)

# Summary by coverage status
print("\nCoverage Summary:")
casting_coverage.groupBy("coverage_status").count().orderBy("coverage_status").show()

In [0]:
# Create detailed mismatch report for further investigation
mismatch_report = casting_coverage.filter(
    F.col("coverage_status") != "FULL COVERAGE"
).select(
    "Casting ID",
    "Strand ID",
    "ts_start",
    "ts_end",
    "Quality casting",
    "data_point_count",
    "first_data",
    "last_data",
    "coverage_status"
).withColumn(
    "expected_duration_hours",
    (F.unix_timestamp("ts_end") - F.unix_timestamp("ts_start")) / 3600
).withColumn(
    "data_start_delay_hours",
    F.when(
        F.col("first_data").isNotNull(),
        (F.unix_timestamp("first_data") - F.unix_timestamp("ts_start")) / 3600
    ).otherwise(None)
).withColumn(
    "data_end_early_hours",
    F.when(
        F.col("last_data").isNotNull(),
        (F.unix_timestamp("ts_end") - F.unix_timestamp("last_data")) / 3600
    ).otherwise(None)
)

print(f"\nCastings with data mismatches: {mismatch_report.count()}")
display(mismatch_report.orderBy("ts_start"))

# Save report
import os
report_path = "/dbfs/FileStore/TATAIjmulden_FCMoldG5/reports/casting_data_mismatch_report.csv"
os.makedirs(os.path.dirname(report_path), exist_ok=True)
mismatch_report.toPandas().to_csv(report_path, index=False)
print(f"\nMismatch report saved to: {report_path}")

In [0]:
df_23_6_spark_clean = (
    df_23_6_spark_unitConverted
        .filter(F.col("castingSpeed") >= 0.5)
        .filter(F.col("SENImmersionDepth").between(0.1, 0.26))
        #.filter(F.col("moldWidth").isNotNull())
)


In [0]:
print("Before:", df_23_6_spark_unitConverted.count())
print("After :", df_23_6_spark_clean.count())


## Visualization: Select small subset for plotting

In [0]:
import os
fig_dir = "dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/"
os.makedirs(fig_dir, exist_ok=True)

In [0]:
cols_for_plot = [
    "PlainTimeStamp",
    "castingSpeed",
    "moldWidth",
    "SENImmersionDepth",
    'Mold Level', 
    'Mold Level Sensor Left', 
    'Mold Level Sensor Right',
    'Argon Flow SEN', 'Argon Flow Stopper',
    'EMBR Current DC Left Bottom',
    'EMBR Current AC Left Master',
    'EMBR Current DC Left Master',
    'Quality casting'
]

df_23_6_small = (
    df_23_6_spark_clean
       .select(*cols_for_plot)
        .dropna(subset=["castingSpeed", "moldWidth", "SENImmersionDepth"])
        # Optional if datasets are huge:
        # .sample(False, 0.1, seed=42)
        # .limit(500000)
)

df_23_6 = df_23_6_small.toPandas()
df_23_6 = df_23_6.rename(columns={"PlainTimeStamp": "plainTimeStamp"})

In [0]:
display(df_23_6_small)

In [0]:
df_23_6_small.printSchema()

In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# --- Create subplots container ---
fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Casting Speed Distribution",
        "SENID Distribution",
        "Mold Width Distribution",
        "Argon Flow SEN+Stopper Distribution"
    ]
)

# --- Subplot 1: Casting Speed ---
h1 = px.histogram(df_23_6, x='castingSpeed')
fig1.add_trace(h1.data[0], row=1, col=1)
fig1.update_xaxes(title_text="Casting Speed [m/min]", row=1, col=1)
fig1.update_yaxes(title_text="Count", row=1, col=1)

# --- Subplot 2: SEN Immersion Depth ---
h2 = px.histogram(df_23_6, x='SENImmersionDepth')#,nbins=165)
fig1.add_trace(h2.data[0], row=1, col=2)
fig1.update_xaxes(title_text="SENImmersionDepth [m]", row=1, col=2)
fig1.update_yaxes(title_text="Count", row=1, col=2)

# --- Subplot 3: Mold Width ---
h3 = px.histogram(df_23_6, x='moldWidth')
fig1.add_trace(h3.data[0], row=2, col=1)
fig1.update_xaxes(title_text="Mold Width [m]", row=2, col=1)
fig1.update_yaxes(title_text="Count", row=2, col=1)

# --- Subplot 4: Argon Flow (SEN + Stopper) ---
argon_total = df_23_6[['Argon Flow SEN', 'Argon Flow Stopper']].sum(axis=1)
h4 = px.histogram(x=argon_total)
fig1.add_trace(h4.data[0], row=2, col=2)
fig1.update_xaxes(title_text="Argon Flow SEN+STOPPER [l/min]", row=2, col=2)
fig1.update_yaxes(title_text="Count", row=2, col=2)

# --- Final Layout ---
fig1.update_layout(
    height=900,
    width=1100,
    title_text="Strand 23-6 — Parameter Distributions",
    showlegend=False
)

fig1.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/cc_parameters_histograms.html")
fig1.show()




In [0]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplot grid (2 rows × 2 columns)
fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Casting Speed Over Time",
        "SEN Immersion Depth Over Time",
        "Mold Width Over Time",
        "Argon Flow (SEN + Stopper) Over Time"
    ]
)

# 1️⃣ Casting Speed
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['castingSpeed'],
        mode='lines'
    ),
    row=1, col=1
)

# 2️⃣ SEN Immersion Depth
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['SENImmersionDepth'],
        mode='lines'
    ),
    row=1, col=2
)

# 3️⃣ Mold Width
fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=df_23_6['moldWidth'],
        mode='lines'
    ),
    row=2, col=1
)

# 4️⃣ Argon Flow = SEN + Stopper
argon_total = df_23_6[['Argon Flow SEN', 'Argon Flow Stopper']].sum(axis=1)

fig2.add_trace(
    go.Scatter(
        x=df_23_6['plainTimeStamp'],
        y=argon_total,
        mode='lines'
    ),
    row=2, col=2
)

# Axis labels
fig2.update_xaxes(title_text="Timestamp", row=1, col=1)
fig2.update_xaxes(title_text="Timestamp", row=1, col=2)
fig2.update_xaxes(title_text="Timestamp", row=2, col=1)
fig2.update_xaxes(title_text="Timestamp", row=2, col=2)

fig2.update_yaxes(title_text="Casting Speed [m/min]", row=1, col=1)
fig2.update_yaxes(title_text="SEN Immersion Depth [m]", row=1, col=2)
fig2.update_yaxes(title_text="Mold Width [m]", row=2, col=1)
fig2.update_yaxes(title_text="Argon Flow [l/min]", row=2, col=2)

# Figure layout
fig2.update_layout(
    height=1000,
    width=1200,
    title_text="Strand 23-6 — Time Series of Key Parameters",
    showlegend=False
)
fig2.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/cc_parameters_timeseriesPlots.html")
fig2.show()


In [0]:
df_23_6.shape

In [0]:
from matplotlib.colors import LogNorm# "Viridis-like" colormap with white background

white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
    (0, '#ffffff'),
    (1e-20, '#440053'),
    (0.2, '#404388'),
    (0.4, '#2a788e'),
    (0.6, '#21a784'),
    (0.8, '#78d151'),
    (1, '#fde624'),
], N=256)

def using_mpl_scatter_density(fig, x, y, labels):
    ax = fig.add_subplot(1, 1, 1, projection='scatter_density')
    density = ax.scatter_density(x, y, cmap=white_viridis, norm=LogNorm(vmin=1,vmax=5000), dpi=12)
    fig.colorbar(density, label=labels)

fig3 = plt.figure(tight_layout = True)
using_mpl_scatter_density(fig3, df_23_6.moldWidth, df_23_6.castingSpeed, labels='CC snapshot counts')
plt.xlabel('mold Width, [m]')
plt.ylabel('casting Speed, [m/min]')
fig3.savefig("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/width_vs_speed_density_strd_23_6.png")
plt.show()


In [0]:
# Plotly line plot for Mold Level
fig4 = px.line(
    df_23_6,
    y=['Mold Level', 'Mold Level Sensor Left', 'Mold Level Sensor Right'],
    x=  'plainTimeStamp',
    title='Mold Level Over Time'
    )
fig4.update_layout(xaxis_title='plainTimeStamp', yaxis_title='Mold Level [m]')
fig4.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/mold_level_plot.html")
fig4.show()

In [0]:
%fs ls dbfs:/FileStore/TATAIjmulden_FCMoldG5/figures/

In [0]:
# after any sampling or filtering
df_23_6_sampled = (
    df_23_6
      .sample(frac=0.25, random_state=42)   # or whatever sampling you used
      .sort_values("plainTimeStamp")        # VERY important
      .reset_index(drop=True)
)
df_23_6_sampled.shape

In [0]:
df_23_6_sampled.head()

In [0]:
df_23_6_sampled.columns

In [0]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.dropna(subset=["plainTimeStamp"]).sort_values("plainTimeStamp").reset_index(drop=True)

# gap-compressed x-axis
df["t_idx"] = np.arange(len(df))

# Argon average (use .sum(axis=1) if you want total instead)
df["Argon Flow AVG"] = df[["Argon Flow SEN", "Argon Flow Stopper"]].mean(axis=1)

# Variables to plot (one subplot each)
series_specs = [
    ("castingSpeed",        "Casting speed"),
    ("moldWidth",           "Mold width"),
    ("SENImmersionDepth",   "SEN immersion depth"),
    ("Mold Level",          "Mold level"),
    ("Argon Flow AVG",      "Argon flow (avg of SEN+Stopper)"),
]

nrows = len(series_specs)

fig = make_subplots(
    rows=nrows, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=[title for _, title in series_specs],
)

for r, (col, _) in enumerate(series_specs, start=1):
    fig.add_trace(
        go.Scatter(
            x=df["t_idx"],
            y=df[col],
            mode="lines",
            customdata=df[["plainTimeStamp"]].to_numpy(),
            hovertemplate=(
                "idx=%{x}<br>"
                "time=%{customdata[0]}<br>"
                f"{col}=%{{y}}<extra></extra>"
            ),
            name=col,
            showlegend=False,
        ),
        row=r, col=1
    )
    fig.update_yaxes(title_text=col, row=r, col=1)

# Sparse readable timestamp ticks
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig.update_xaxes(
    title_text="Time (gaps removed; hover shows real timestamp)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
    row=nrows, col=1
)

fig.update_layout(
    title="Strand#23_6 – Casting parameters (sampled and gap-compressed)",
    height=260 * nrows,
    width=1200,
)

# Save & show
fig.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_casting_parameters_subplots.html")
fig.show()


In [0]:
import pandas as pd
import plotly.express as px

df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig = px.line(
    df,
    x="t_idx",
    y=[
        "castingSpeed",
        "moldWidth",
       # "SENImmersionDepth",
       # 'Mold Level'
    ],
    title="Strand#23_6: Casting speed and Mold width (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)
# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_Vc_and_width_parameters.html"
)

fig.show()


In [0]:
df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig7 = px.line(
    df,
    x="t_idx",
    y=[
        "Mold Level", 
        "Mold Level Sensor Left", 
        "Mold Level Sensor Right"
    ],
    title="Strand#23_6: Mold Level (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig7.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig7.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)
fig7.update_layout(yaxis_title="Mold Level [mm]")
fig7.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/Argon_plot.html")
fig7.show()

In [0]:


df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig8 = px.line(
    df,
    x="t_idx",
    y=[
        'Argon Flow SEN', 
        'Argon Flow Stopper'
    ],
    title="Strand#23_6: Argon flow (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig8.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig8.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)

fig8.show()

In [0]:


df = df_23_6_sampled.copy()
df["plainTimeStamp"] = pd.to_datetime(df["plainTimeStamp"], errors="coerce")
df = df.sort_values("plainTimeStamp").dropna(subset=["plainTimeStamp"])

# Create a continuous axis (sample index)
df["t_idx"] = range(len(df))

fig8 = px.line(
    df,
    x="t_idx",
    y=[
        "EMBR Current DC Left Bottom",
        "EMBR Current AC Left Master",
        "EMBR Current DC Left Master",
    ],
    title="Strand#23_6: FC Mold currents (gap-compressed timeline)",
)

# Make hover show the real timestamp
fig8.update_traces(
    customdata=df[["plainTimeStamp"]].to_numpy(),
    hovertemplate="idx=%{x}<br>time=%{customdata[0]}<br>value=%{y}<extra></extra>",
)

# Replace x ticks with timestamps (sparse, readable)
tick_step = max(1, len(df) // 10)
tick_vals = df["t_idx"].iloc[::tick_step]
tick_text = df["plainTimeStamp"].dt.strftime("%b %d\n%H:%M").iloc[::tick_step]

fig8.update_xaxes(
    title_text="Sequence index (gaps removed)",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=tick_text,
)

fig8.show()


In [0]:
%python
import plotly.express as px

columns_to_plot = [
    "castingSpeed",
    "moldWidth",
    "SENImmersionDepth",
    "Mold Level",
    "Argon Flow SEN",
    "Argon Flow Stopper",
    "EMBR Current DC Left Bottom",
    "EMBR Current AC Left Master",
    "EMBR Current DC Left Master"
]

# Reshape to long format
df_long = df_23_6.melt(
    id_vars=["plainTimeStamp"],
    value_vars=columns_to_plot,
    var_name="Parameter",
    value_name="Value"
)

fig = px.line(
    df_long,
    x="plainTimeStamp",
    y="Value",
    color="Parameter",
    facet_col="Parameter",
    facet_col_wrap=4,
    labels={"plainTimeStamp": "Time", "Value": "Value"},
    color_discrete_sequence=["blue"],
    facet_col_spacing=0.05,
    facet_row_spacing=0.1,
    category_orders={"plainTimeStamp": df_long["plainTimeStamp"].astype(str).unique().tolist()}
)


fig.update_layout(showlegend=False)

fig.write_html("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand1_ALL_signals_SEN.html")
fig.show()

## Data grouping

In [0]:
# Function to round values with custom logic
def custom_rounding(series, round_up=True):
    rounded_values = []
    for i in range(len(series)):
        if i > 0 and abs(series[i] - series[i-1]) <= 0.01: # CASTING_SPEED rounding difference
            rounded_value = rounded_values[-1]
        else:
            if round_up:
                rounded_value = round(series[i] * 100 + 0.5) / 100
            else:
                rounded_value = round(series[i] * 100 - 0.5) / 100
        rounded_values.append(rounded_value)
    return rounded_values
  

def filter_and_group_sequences(df, status_column, speed_column, status_value=1, diff_threshold=0.1, min_length=300):
    """
    Filters the DataFrame based on a status column and groups sequences of rows
    where the difference between consecutive values in a specified column is
    less than or equal to a threshold, retaining only groups with a minimum length.

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        status_column (str): The column to filter by status.
        speed_column (str): The column to group by consecutive differences.
        status_value (int): The value to filter the status column. Default is 1.
        diff_threshold (float): The maximum allowed difference between consecutive values. Default is 0.1.
        min_length (int): The minimum length of a valid group. Default is 300.

    Returns:
        pd.DataFrame: A DataFrame containing valid groups.
    """
    # Step 1: Filter the DataFrame where the status column matches the specified value
    filtered_df = df[df[status_column] == status_value].reset_index(drop=True)

    # Step 2: Identify groups based on the difference in the speed column
    # Create a group identifier where the difference between consecutive rows is > diff_threshold
    filtered_df['group'] = (filtered_df[speed_column].diff().abs() > diff_threshold).cumsum()

    # Step 3: Group by the 'group' column and filter groups with at least the minimum length
    valid_groups = (
        filtered_df.groupby('group')
        .filter(lambda x: len(x) >= min_length)
        .reset_index(drop=True)
    )

    # Drop the temporary 'group' column if not needed
    valid_groups = valid_groups.drop(columns=['group'])

    return valid_groups

# Segment the time series into continuous chunks based on gaps in plainTimeStamp

def segment_by_time_gaps(df, tcol="plainTimeStamp", max_gap_seconds=5):
    """
    Split a time-series DataFrame into continuous segments based on time gaps.

    A new segment starts whenever the time difference between consecutive rows
    exceeds `max_gap_seconds`.

    Returns:
        df_seg: df with tcol as datetime and a new 'segment_id' column.
    """
    df_seg = df.copy()

    # ensure datetime and sort by time
    df_seg[tcol] = pd.to_datetime(df_seg[tcol])
    df_seg = df_seg.sort_values(tcol)

    # time difference in seconds
    dt = df_seg[tcol].diff().dt.total_seconds().fillna(0)

    # cumsum over "gap" indicators → segment IDs: 0,1,2,...
    df_seg["segment_id"] = (dt > max_gap_seconds).cumsum()

    return df_seg



# Function to identify groups of consecutive values within the difference threshold using a sliding window of step 1
def identify_sequences(df,
                       Vc_column,
                       window_size,
                       Vc_threshold,
                       Curr_columns=None,
                       Curr_threshold=None):
    """
    Sliding window with step 1.

    A window is classified as NORMAL if:
      - all values of Vc_column in the window are within Vc_threshold
        of the first value; AND
      - for every column in Curr_columns (if provided), all consecutive
        differences are < Curr_threshold.

    Otherwise the window is ABNORMAL.

    Returns:
        normal_groups: list of lists of indices (NORMAL windows)
        abnormal_groups: list of lists of indices (ABNORMAL windows)
    """
    normal_groups = []
    abnormal_groups = []

    i = 0
    n = len(df)

    while i <= n - window_size:
        window_df = df.iloc[i:i + window_size]
        window_idx = window_df.index

        # 1) main condition on casting speed
        Vc_window = window_df[Vc_column]
        cond_vc = np.all(np.abs(Vc_window - Vc_window.iloc[0]) <= Vc_threshold)

        # 2) additional conditions on current columns (if provided)
        cond_curr = True
        if cond_vc and Curr_columns is not None and Curr_threshold is not None:
            for col in Curr_columns:
                if col not in window_df.columns:
                    cond_curr = False
                    break
                series = window_df[col]
                # check consecutive differences
                if np.any(np.abs(series.diff().dropna()) >= Curr_threshold):
                    cond_curr = False
                    break

        is_normal = cond_vc and cond_curr

        if is_normal:
            normal_groups.append(window_idx.tolist())
            # skip past this whole stable window
            i += window_size
        else:
            abnormal_groups.append(window_idx.tolist())
            # move start forward by ONE sample and try again
            i += 1

    return normal_groups, abnormal_groups

def identify_sequences_segmented(df,
                                 Vc_column,
                                 window_size,
                                 Vc_threshold,
                                 Curr_columns=None,
                                 Curr_threshold=None,
                                 tcol="plainTimeStamp",
                                 max_gap_seconds=5,
                                 min_segment_len=None):
    """
    Apply identify_sequences within continuous time segments so that
    sliding windows never cross large time gaps.

    Args:
        df: original DataFrame
        Vc_column: name of casting speed column
        window_size: window length in samples
        Vc_threshold: threshold for "stable" speed
        Curr_columns: list of additional columns to check
        Curr_threshold: threshold for consecutive differences in those columns
        tcol: timestamp column
        max_gap_seconds: max allowed gap inside a segment
        min_segment_len: if set, skip segments shorter than this length
                         (often set to window_size)

    Returns:
        normal_groups_all: list of index lists (global indices)
        abnormal_groups_all: list of index lists (global indices)
    """
    df_seg = segment_by_time_gaps(df, tcol=tcol, max_gap_seconds=max_gap_seconds)

    normal_groups_all = []
    abnormal_groups_all = []

    for seg_id, seg_df in df_seg.groupby("segment_id"):
        if min_segment_len is not None and len(seg_df) < min_segment_len:
            continue

        n_g, a_g = identify_sequences(
            seg_df,
            Vc_column=Vc_column,
            window_size=window_size,
            Vc_threshold=Vc_threshold,
            Curr_columns=Curr_columns,
            Curr_threshold=Curr_threshold,
        )

        normal_groups_all.extend(n_g)
        abnormal_groups_all.extend(a_g)

    return normal_groups_all, abnormal_groups_all


In [0]:
df_23_6.head(5)

In [0]:
df_FCMold_ON_str23_6 = df_23_6[(df_23_6['EMBR Current AC Left Master'] != 0) & (df_23_6['EMBR Current DC Left Master'] != 0) & (df_23_6['EMBR Current DC Left Bottom'] != 0)].reset_index(drop=True)

In [0]:
(df_23_6.shape, df_FCMold_ON_str23_6.shape)

In [0]:
df_FCMold_ON_str23_6['castingSpeed'] =  custom_rounding(df_FCMold_ON_str23_6.castingSpeed, 2)

In [0]:
df_FCMold_ON_str23_6.head(10)

In [0]:
# Example usage
window_size = 300  # 6 min window size
Vc_column_str5 = 'castingSpeed'
Vc_column_str6 = 'castingSpeed'
Vc_threshold = 0.1
Curr_columns_str5 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_columns_str6 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_threshold = 50

In [0]:

Vc_column_str6 = "castingSpeed"

normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = identify_sequences_segmented(
    df_FCMold_ON_str23_6,
    Vc_column=Vc_column_str6,
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns_str6,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5,          # treat gaps >5s as new segments
    min_segment_len=window_size # ignore segments shorter than one window
)

len(normal_groups_ON_str23_6), len(abnormal_groups_ON_str23_6)

In [0]:
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import pandas as pd
import math

df = df_FCMold_ON_str23_6
Vc_col = Vc_column_str6
window_size_vis = 300
max_windows = 12
Vc_threshold = Vc_threshold

# ---------- 1) Restrict DF to FIRST 30 MIN ONLY & SORT ----------
if "plainTimeStamp" in df.columns:
    tcol = "plainTimeStamp"
elif "plainTimestamp" in df.columns:
    tcol = "plainTimestamp"
else:
    raise ValueError("Need timestamp column.")

ts_full = df[tcol]
start_ts = ts_full.iloc[0]
end_ts_2h = start_ts + pd.Timedelta(hours=0.5)

mask_2h = (ts_full >= start_ts) & (ts_full <= end_ts_2h)
df_chunk = (
    df.loc[mask_2h]
      .copy()
      .sort_values(tcol)
      .reset_index(drop=True)
)

ts = df_chunk[tcol]
Vc = df_chunk[Vc_col]
y_all = df_chunk["castingSpeed"]

n = len(df_chunk)
idx_all = np.arange(n)

# ---------- 2) Sliding-window search on chunk ----------
windows = []

i = 0
while i <= n - window_size_vis and len(windows) < max_windows:
    window = Vc.iloc[i:i + window_size_vis]
    diffs = np.abs(window - window.iloc[0])

    bad_offsets = np.where(diffs > Vc_threshold)[0]
    if len(bad_offsets) > 0:
        first_bad = bad_offsets[0]
        break_idx = i + first_bad
        cond = False
    else:
        break_idx = None
        cond = True

    start = i
    end = i + window_size_vis
    windows.append((start, end, break_idx, cond))

    if cond:
        i += window_size_vis
    else:
        i = break_idx + 1

n_windows = len(windows)
n_cols = 2
n_rows = max(1, math.ceil(n_windows / n_cols))

# ---------- 3) Plotly figure with 2 columns ----------
titles = [
    f"Window {k} (start={start}, end={end-1}) — {'NORMAL' if cond else 'ABNORMAL'}"
    for k, (start, end, break_idx, cond) in enumerate(windows)
]
# pad titles if last row not full
if len(titles) < n_rows * n_cols:
    titles += [""] * (n_rows * n_cols - len(titles))

fig = sp.make_subplots(
    rows=n_rows,
    cols=n_cols,
    shared_xaxes=True,
    vertical_spacing=0.06,
    horizontal_spacing=0.08,
    subplot_titles=titles,
)

for idx, (start, end, break_idx, cond) in enumerate(windows):
    row = idx // n_cols + 1
    col = idx % n_cols + 1

    first_subplot = (idx == 0)
    normal_subplot = (idx == n_windows - 1)

    gray_y = y_all.copy()

    in_window = (idx_all >= start) & (idx_all < end)
    blue_mask = np.zeros(n, dtype=bool)
    black_mask = np.zeros(n, dtype=bool)
    red_mask = np.zeros(n, dtype=bool)

    if cond:
        blue_mask[in_window] = True
        gray_y[blue_mask] = np.nan
    else:
        if break_idx is not None:
            black_mask[(idx_all >= start) & (idx_all < break_idx)] = True
            red_mask[(idx_all >= break_idx) & (idx_all < end)] = True

        gray_y[in_window] = np.nan

    # gray trace
    fig.add_trace(
        go.Scatter(
            x=ts,
            y=gray_y,
            mode="lines",
            line=dict(color="gray", width=1),
            name="castingSpeed, m/min" if first_subplot else None,
            showlegend=first_subplot,
        ),
        row=row,
        col=col,
    )

    # blue: full normal windows
    if blue_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(blue_mask, np.nan),
                mode="lines",
                line=dict(color="blue", width=3),
                name="Normal window", # if first_subplot else None,
                showlegend=normal_subplot,
            ),
            row=row,
            col=col,
        )

    # black: pre-break stable region in abnormal windows
    if black_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(black_mask, np.nan),
                mode="lines",
                line=dict(color="black", width=2),
                name="normal region" if first_subplot else None,
                showlegend=first_subplot,
            ),
            row=row,
            col=col,
        )

    # red: abnormal region
    if red_mask.any():
        fig.add_trace(
            go.Scatter(
                x=ts,
                y=y_all.where(red_mask, np.nan),
                mode="lines",
                line=dict(color="red", width=3),
                name="abnormal region" if first_subplot else None,
                showlegend=first_subplot,
            ),
            row=row,
            col=col,
        )

# layout
fig.update_layout(
    height=260 * n_rows,
    width=1300,
    title="Data grouping - Sliding Window Visualization",
)

# x-axis label only on bottom row
for c in range(1, n_cols + 1):
    fig.update_xaxes(title_text="Time", row=n_rows, col=c)

fig.show()


In [0]:
# Populate df_result with sequence's statistics
def generate_seq_statistics(df, sequences, seq_condition):
    df_result = pd.DataFrame()
    cnt = 0

    for group in sequences:
        if max(group) >= len(df):
            raise IndexError("positional indexers are out-of-bounds")
        
        temp_df = df.iloc[group]

        temp_df = temp_df.reset_index(drop=True)

        # Populate df_results with Statistics
        df_result.loc[cnt, 'Seq_Name'] = ','.join(['Seq#'+'%d' %(cnt)]) #', '.join(temp_df['filename'].apply(lambda x: os.path.basename(x)).unique())[25:29]+'_%d' %(cnt)]

        timestamps = temp_df['plainTimeStamp'].to_list()

        if timestamps:
            df_result.loc[cnt, 'Seq_time_Start'] = timestamps[0]
            #if len(timestamps) > (result_reduced[0][1] - 1):
            #    df_result.loc[cnt, 'Seq_time_End'] = timestamps[result_reduced[0][1] - 1]
            #else:
            df_result.loc[cnt, 'Seq_time_End'] = timestamps[-1]  # Use the last timestamp if the index is out of range
        else:
            df_result.loc[cnt, 'Seq_time_Start'] = None
            df_result.loc[cnt, 'Seq_time_End'] = None

        df_result.loc[cnt, 'SEN_avg [mm]'] = temp_df['SENImmersionDepth'].mean()
        df_result.loc[cnt, 'SEN_std [mm]'] = temp_df['SENImmersionDepth'].std()

       
        # Compute statistics
        df_result.loc[cnt,'CASTING_SPEED_avg [m/min]'] = temp_df['castingSpeed'].mean()
        df_result.loc[cnt,'CASTING_SPEED_std [m/min]'] = temp_df['castingSpeed'].std()
        df_result.loc[cnt,'MOLD_WIDTH_avg [m]'] = temp_df['moldWidth'].mean()
        df_result.loc[cnt,'MOLD_WIDTH_std [m]'] = temp_df['moldWidth'].std()
        df_result.loc[cnt,'min DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].min()
        df_result.loc[cnt,'mean DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].mean()
        df_result.loc[cnt,'max DC Current, Low [A]'] = temp_df['EMBR Current DC Left Bottom'].max()
        df_result.loc[cnt,'min DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].min()
        df_result.loc[cnt,'mean DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].mean()
        df_result.loc[cnt,'max DC Current, Up [A]'] = temp_df['EMBR Current DC Left Master'].max()
        df_result.loc[cnt,'min AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].min()
        df_result.loc[cnt,'mean AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].mean()
        df_result.loc[cnt,'max AC Current, Up [A]'] = temp_df['EMBR Current AC Left Master'].max()
        df_result.loc[cnt, 'MOLD_LEVEL_avg [mm]'] = temp_df['Mold Level'].mean()
        df_result.loc[cnt, 'MOLD_LEVEL_std [mm]'] = temp_df['Mold Level'].std()
        df_result.loc[cnt, 'ArFlow_avg [NL/min]'] = (temp_df['Argon Flow SEN'] + temp_df['Argon Flow Stopper']).mean()
        df_result.loc[cnt, 'ArFlow_std [NL/min]'] = (temp_df['Argon Flow SEN'] + temp_df['Argon Flow Stopper']).std()
        #df_result.loc[cnt, 'SEN_avg [m]'] = temp_df['SEN'].mean()
        #df_result.loc[cnt, 'SEN_std [m]'] = temp_df['SEN'].std()

        
        df_result.loc[cnt, 'Seq_Condition'] = seq_condition
        cnt +=1
        
    return df_result



In [0]:
df_seq_normal_ON_str23_6 = generate_seq_statistics(df_FCMold_ON_str23_6, normal_groups_ON_str23_6, 'normal')

In [0]:
df_seq_normal_ON_str23_6.shape

In [0]:
df_seq_normal_ON_str23_6.head()

In [0]:
def plot_MLsigma_per_sequence(df_seq, df_FCstatus, str_MLsignal, str_label):
    # Remove underscores from the 'Seq_Name' column values
    df_seq['Seq_Name'] = df_seq['Seq_Name'].str.replace('_', '', regex=False)

    for seq in df_seq['Seq_Name'].to_list():
        # Extract the sequence info - statistics - 
        seq_info = df_seq[df_seq['Seq_Name'] == seq]
        # Remove underscores from the 'Seq_Name' column values
        #seq_info['Seq_Name'] = seq_info['Seq_Name'].str.replace('_', '', regex=False)

        # Get the start and end time as strings
        start_time = seq_info['Seq_time_Start'].iloc[0]
        end_time = seq_info['Seq_time_End'].iloc[0]

        # Slice the DataFrame based on the 'time' column
        df_temp = df_FCstatus[(df_FCstatus['plainTimeStamp'] >= start_time) & (df_FCstatus['plainTimeStamp'] <= end_time)]

        #df_ON_seq6 = df_FCMold_ON.iloc[normal_groups_ON[21]]
        # Calculate mean, min, and max values
        mean_value = df_temp[str_MLsignal].mean()
        min_value = df_temp[str_MLsignal].min()
        max_value = df_temp[str_MLsignal].max()

        # Create the line plot
        fig11 = px.line(
            df_temp,
            x="plainTimeStamp",
            y=[str_MLsignal],
            title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                str_label, seq, start_time, end_time,
                seq_info['CASTING_SPEED_avg [m/min]'].values[0],
                seq_info['min AC Current, Up [A]'].values[0],
                seq_info['mean DC Current, Up [A]'].values[0],
                seq_info['mean DC Current, Low [A]'].values[0]
            ),
)
        fig11.update_layout(title_font_size=14,  # Set the title font size
            yaxis_title="Mold Level [mm]"  # Add y-axis label
        )
        # Add horizontal lines for mean, min, and max
        fig11.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
        fig11.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
        fig11.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

        # Save the plot as a PNG file
        # Sanitize time strings for filename
        safe_start_time = start_time.strftime('%Y-%m-%d_%H-%M-%S')
        safe_end_time = end_time.strftime('%Y-%m-%d_%H-%M-%S')
       
        #fig2.write_html("../reports/figures/data_grouping/window_size_5min/%s_%s_%s_%s_Vc=%.2f_AC_up=%d_DC_up=%d_DC_low=%d.html" % (
        #        str_label, seq, safe_start_time, safe_end_time,
        #        seq_info['CASTING_SPEED_avg [m/min]'].values[0],
        #        seq_info['min AC Current, Up [A]'].values[0],
        #        seq_info['mean DC Current, Up [A]'].values[0],
        #        seq_info['mean DC Current, Low [A]'].values[0]))#, width=1200, height=600, scale=2)
        # Show the plot
        #fig11.show()

In [0]:
#plot_MLsigma_per_sequence(df_seq_normal_ON_str23_6, 
#                            df_FCMold_ON_str23_6, 
#                            str_MLsignal = "Mold Level", 
#                            str_label = 'Strd23_6')

In [0]:
df_seq_normal_ON_str23_6.head()

In [0]:
df_seq_normal_ON_str23_6.shape[0]

In [0]:
for i in range(5): #df_seq_normal_ON_str23_6.shape[0]):
    print(df_seq_normal_ON_str23_6.iloc[i]['Seq_Name'])
    # Select the 
    sequence = i

    df_ON_seq23_6 = df_FCMold_ON_str23_6.iloc[normal_groups_ON_str23_6[sequence]]
    # Calculate mean, min, and max values
    mean_value = df_ON_seq23_6["Mold Level"].mean()
    min_value = df_ON_seq23_6["Mold Level"].min()
    max_value = df_ON_seq23_6["Mold Level"].max()

    #metadata
    str_label = 'strd23_6'
    seq = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_Name']
    start_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_Start']
    end_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_End']
    Vc_seq = df_seq_normal_ON_str23_6.iloc[sequence]['CASTING_SPEED_avg [m/min]']
    ACup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['min AC Current, Up [A]']
    DCup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Up [A]']
    DClow_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Low [A]']

    # Crete the line plot
    fig10 = px.line(
        df_ON_seq23_6,
        x="plainTimeStamp",
        y=["Mold Level"],
        title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                    str_label, seq, start_time, end_time,
                    Vc_seq,
                    ACup_seq,
                    DCup_seq,
                    DClow_seq
                ),
            )

    # Add horizontal lines for mean, min, and max
    fig10.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
    fig10.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
    fig10.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

    # Save the plot as a PNG file
    #fig2.write_html("../reports/figures/data_grouping/window_size_5min/strand1_seq21_mold_level.html")#, width=1200, height=600, scale=2)
    # Show the plot
    fig10.show()

In [0]:
df_seq_normal_ON_str23_6.columns

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_ON_str23_6

# --- Helper to build each scatter with Seq_Name in hover only ---
def make_scatter(df, x_col, x_label):
    fig_px = px.scatter(
        df,
        x=x_col,
        y="MOLD_LEVEL_std [mm]",
        hover_data=["Seq_Name"],     # show Seq_Name in hover
        labels={
            x_col: x_label,
            "MOLD_LEVEL_std [mm]": "Mold Level, [mm]"
        }
    )

    trace = fig_px.data[0]
    trace.text = None  # <-- remove text labels on points
    trace.hovertemplate = (
        "Seq_Name: %{customdata[0]}<br>"
        f"{x_label}: %{{x}}<br>"
        "Mold Level, [mm]: %{y}<extra></extra>"
    )

    return trace

# --- Create 2x2 subplot layout ---
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Vc vs Mold Level",
        "Mold Width vs Mold Level",
        "SEN vs Mold Level",
        "Argon Flow vs Mold Level"
    ]
)

# 1) Casting speed vs Mold Level
fig.add_trace(
    make_scatter(df, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]"),
    row=1, col=1
)

# 2) Mold width vs Mold Level
fig.add_trace(
    make_scatter(df, "MOLD_WIDTH_avg [m]", "Mold Width Avg [m]"),
    row=1, col=2
)

# 3) SEN vs Mold Level
fig.add_trace(
    make_scatter(df, "SEN_avg [mm]", "SEN Avg [mm]"),
    row=2, col=1
)

# 4) Argon Flow vs Mold Level
fig.add_trace(
    make_scatter(df, "ArFlow_avg [NL/min]", "Ar Flow Avg [NL/min]"),
    row=2, col=2
)

# --- Add horizontal threshold lines ---
for r in [1, 2]:
    for c in [1, 2]:
        fig.add_hline(
            y=2,
            line_dash="dash",
            line_color="green",
            annotation_text="ML fluctuation = 2 mm",
            annotation_position="top left",
            row=r, col=c
        )

# --- Axis titles ---
fig.update_xaxes(title_text="Casting Speed Avg [m/min]", row=1, col=1)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=1, col=1)

fig.update_xaxes(title_text="Mold Width Avg [m]",        row=1, col=2)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=1, col=2)

fig.update_xaxes(title_text="SEN Avg [mm]",              row=2, col=1)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=2, col=1)

fig.update_xaxes(title_text="Ar Flow Avg [NL/min]",      row=2, col=2)
fig.update_yaxes(title_text="Mold Level, [mm]",         row=2, col=2)

# --- Layout ---
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Correlations vs Mold Level",
    showlegend=False
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_cc_parameters_and_ML_correlations.html"
)
fig.show()


In [0]:
sequence = 355

df_ON_seq23_6 = df_FCMold_ON_str23_6.iloc[normal_groups_ON_str23_6[sequence]]
    # Calculate mean, min, and max values
mean_value = df_ON_seq23_6["Mold Level"].mean()
min_value = df_ON_seq23_6["Mold Level"].min()
max_value = df_ON_seq23_6["Mold Level"].max()

    #metadata
str_label = 'strd23_6'
seq = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_Name']
start_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_Start']
end_time = df_seq_normal_ON_str23_6.iloc[sequence]['Seq_time_End']
Vc_seq = df_seq_normal_ON_str23_6.iloc[sequence]['CASTING_SPEED_avg [m/min]']
ACup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['min AC Current, Up [A]']
DCup_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Up [A]']
DClow_seq = df_seq_normal_ON_str23_6.iloc[sequence]['mean DC Current, Low [A]']

    # Crete the line plot
fig11 = px.line(
    df_ON_seq23_6,
    x="plainTimeStamp",
    y=["Mold Level"],
    title="%s_%s_%s_%s, Vc=%.2f, AC_up=%d, DC_up=%d, DC_low=%d" % (
                 str_label, seq, start_time, end_time,
                Vc_seq,
                ACup_seq,
                DCup_seq,
                DClow_seq
            ),
        )

    # Add horizontal lines for mean, min, and max
fig11.add_hline(y=mean_value, line_dash="dash", line_color="green", annotation_text="Mean", annotation_position="top left")
fig11.add_hline(y=min_value, line_dash="dot", line_color="blue", annotation_text="Min", annotation_position="bottom left")
fig11.add_hline(y=max_value, line_dash="dot", line_color="red", annotation_text="Max", annotation_position="top right")

    # Save the plot as a PNG file
    #fig2.write_html("../reports/figures/data_grouping/window_size_5min/strand1_seq21_mold_level.html")#, width=1200, height=600, scale=2)
    # Show the plot
fig11.show()

In [0]:
df_seq_normal_ON_str23_6.columns

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Colormap where low values are visible (no white-on-white)
ml_cmap = LinearSegmentedColormap.from_list(
    'ml_sigma_cmap',
    [
        (0.0, '#440053'),   # dark purple  (very low σ)
        (0.3, '#404388'),
        (0.6, '#2a788e'),
        (0.8, '#21a784'),
        (0.95, '#78d151'),
        (1.0, '#fde624'),   # yellow       (very high σ)
    ],
    N=256
)

ml_std = df_seq_normal_ON_str23_6['MOLD_LEVEL_std [mm]'].to_numpy()
ml_std_max = np.nanmax(ml_std)

norm = TwoSlopeNorm(vmin=0, vcenter=2, vmax=ml_std_max)

x = df_seq_normal_ON_str23_6['MOLD_WIDTH_avg [m]'].to_numpy()
y = df_seq_normal_ON_str23_6['CASTING_SPEED_avg [m/min]'].to_numpy()

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x = x[mask]
y = y[mask]
ml_std = ml_std[mask]

above_thr = ml_std > 2.0  # bad region

fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter: all points colored by σ, threshold-aware colormap
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge so they stand out
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=60,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.95,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('MOLD_LEVEL_std [mm]')

ax.set_xlabel('Mold Width avg [m]')
ax.set_ylabel('Casting Speed avg [m/min]')
ax.set_title('Mold Width vs Casting Speed colored by Mold Level σ')

# Optional: visual threshold line in colorbar label
# (the TwoSlopeNorm already makes 2 mm visually special)
plt.show()



In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Colormap where low values are visible (no white-on-white)
ml_cmap = LinearSegmentedColormap.from_list(
    'ml_sigma_cmap',
    [
        (0.0, '#440053'),   # dark purple  (very low σ)
        (0.3, '#404388'),
        (0.6, '#2a788e'),
        (0.8, '#21a784'),
        (0.95, '#78d151'),
        (1.0, '#fde624'),   # yellow       (very high σ)
    ],
    N=256
)

ml_std = df_seq_normal_ON_str23_6['MOLD_LEVEL_std [mm]'].to_numpy()
ml_std_max = np.nanmax(ml_std)

norm = TwoSlopeNorm(vmin=0, vcenter=2, vmax=ml_std_max)

x = df_seq_normal_ON_str23_6['SEN_avg [mm]'].to_numpy()
y = df_seq_normal_ON_str23_6['ArFlow_avg [NL/min]'].to_numpy()

mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x = x[mask]
y = y[mask]
ml_std = ml_std[mask]

above_thr = ml_std > 2.0  # bad region

fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter: all points colored by σ, threshold-aware colormap
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge so they stand out
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=60,
    edgecolor="black",
    linewidth=0.7,
    alpha=0.95,
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label('MOLD_LEVEL_std [mm]')

ax.set_xlabel('SEN_avg [m]')
ax.set_ylabel('ArFlow_avg [NL/min]')
#ax.set_title('Mold Width vs Casting Speed colored by Mold Level σ')

# Optional: visual threshold line in colorbar label
# (the TwoSlopeNorm already makes 2 mm visually special)
plt.show()



In [0]:
ml_colorscale = [
    [0.0,  "#440053"],  # dark purple – very low σ
    [0.3,  "#404388"],
    [0.6,  "#2a788e"],
    [0.8,  "#21a784"],
    [0.95, "#78d151"],
    [1.0,  "#fde624"],  # yellow – very high σ
]


import plotly.graph_objects as go
import numpy as np
import pandas as pd

df = df_seq_normal_ON_str23_6

ml_std = df["MOLD_LEVEL_std [mm]"]
vmin = 0.0
vmax = float(ml_std.max())
threshold = 2.0

# Custom colorscale similar to the Matplotlib one
ml_colorscale = [
    [0.0,  "#440053"],
    [0.3,  "#404388"],
    [0.6,  "#2a788e"],
    [0.8,  "#21a784"],
    [0.95, "#78d151"],
    [1.0,  "#fde624"],
]

# Split data into "normal" (σ ≤ 2 mm) and "abnormal" (σ > 2 mm)
df_normal   = df[ml_std <= threshold]
df_abnormal = df[ml_std >  threshold]

fig = go.Figure()

# ---- NORMAL trace (σ ≤ 2 mm) ----
fig.add_trace(
    go.Scatter3d(
        x=df_normal['max AC Current, Up [A]'],
        y=df_normal['max DC Current, Low [A]'],
        z=df_normal['max DC Current, Up [A]'],
        mode='markers',
        marker=dict(
            size=4,
            color=df_normal["MOLD_LEVEL_std [mm]"],
            colorscale=ml_colorscale,
            cmin=vmin,
            cmax=vmax,
            coloraxis="coloraxis",
            opacity=0.8,
        ),
        name="σ ≤ 2 mm",
        hovertemplate=(
            "AC Up: %{x:.2f} A<br>"
            "DC Low: %{y:.2f} A<br>"
            "DC Up: %{z:.2f} A<br>"
            "ML σ: %{marker.color:.2f} mm<extra>Normal</extra>"
        ),
    )
)

# ---- ABNORMAL trace (σ > 2 mm), highlighted with black edge ----
fig.add_trace(
    go.Scatter3d(
        x=df_abnormal['max AC Current, Up [A]'],
        y=df_abnormal['max DC Current, Low [A]'],
        z=df_abnormal['max DC Current, Up [A]'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_abnormal["MOLD_LEVEL_std [mm]"],
            colorscale=ml_colorscale,
            cmin=vmin,
            cmax=vmax,
            coloraxis="coloraxis",
            opacity=0.95,
            line=dict(color="black", width=1),
        ),
        name="σ > 2 mm",
        hovertemplate=(
            "AC Up: %{x:.2f} A<br>"
            "DC Low: %{y:.2f} A<br>"
            "DC Up: %{z:.2f} A<br>"
            "ML σ: %{marker.color:.2f} mm<extra>Abnormal</extra>"
        ),
    )
)

# ---- Shared coloraxis (single colorbar) ----
fig.update_layout(
    coloraxis=dict(
        colorscale=ml_colorscale,
        cmin=vmin,
        cmax=vmax,
        colorbar=dict(
            title="MOLD_LEVEL σ [mm]",
            thickness=15,
            len=0.8,
            x=1.02,
            xanchor="left",
            y=0.5,
            yanchor="middle",
        ),
    ),
    scene=dict(
        xaxis_title="AC Current, Up (A)",
        yaxis_title="DC Current, Low (A)",
        zaxis_title="DC Current, Up (A)",
        aspectmode="cube",
        xaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        yaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        zaxis=dict(
            color='black',
            showline=True,
            linewidth=2,
            linecolor='black',
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            showbackground=True,
            backgroundcolor='white',
        ),
        camera=dict(eye=dict(x=1.2, y=1.2, z=1.2)),
    ),
    width=800,
    height=600,
    margin=dict(l=20, r=120, b=20, t=50),
    title="Strand #23_6 – FC Mold G5 Currents colored by Mold Level σ",
    title_x=0.5,
    autosize=False,
)

fig.show()


# Data grouping - Disturbance detector


In [0]:
import numpy as np
import pandas as pd

# ---------- detectors from our latest approach ----------
def has_slow_drift(df_seq, value_col="Mold Level_clean", time_col="plainTimeStamp",
                   min_run_s=60.0, min_amp_mm=10.0):
    if len(df_seq) < 3:
        return False
    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce").to_numpy()
    mask = np.isfinite(v)
    if mask.sum() < 3:
        return False
    t = t[mask]
    v = v[mask]

    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False

    dv = np.diff(v)
    sign = np.sign(dv)
    for i in range(1, len(sign)):
        if sign[i] == 0:
            sign[i] = sign[i-1]

    runs, start = [], 0
    for i in range(1, len(sign)):
        if sign[i] != sign[start]:
            runs.append((start, i-1))
            start = i
    runs.append((start, len(sign)-1))

    for i0, i1 in runs:
        if sign[i0] == 0:
            continue
        duration_s = (i1 - i0 + 1) * dt
        amp = abs(v[i1+1] - v[i0])
        if duration_s >= min_run_s and amp >= min_amp_mm:
            return True
    return False


def has_excursion_event_robust(df_seq, value_col="Mold Level", time_col="plainTimeStamp",
                               excursion_mm=8.0, min_duration_s=5.0):
    if len(df_seq) < 3:
        return False
    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce")

    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False

    baseline = v.median()
    exc = (v - baseline).abs() > excursion_mm
    exc = exc.fillna(False).to_numpy()

    max_run, cur = 0, 0
    for b in exc:
        if b:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 0

    return (max_run * dt) >= min_duration_s


def simple_spike_clean(series, median_win=5, jump_mm=5.0):
    s = pd.to_numeric(series, errors="coerce")
    med = s.rolling(median_win, center=True).median()
    spike = (med.diff().abs() > jump_mm)
    s2 = s.copy()
    s2[spike] = np.nan
    return s2.interpolate(limit_direction="both")

import numpy as np
import pandas as pd

def detect_transient_bump_dynamic(
    df_seq,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    # dynamic threshold controls
    k_amp=8.0,            # how many "noise sigmas" defines a bump
    min_amp_mm=6.0,       # absolute minimum bump amplitude
    k_return=2.5,         # return band in sigmas
    # duration controls
    min_duration_s=5.0,
    max_duration_s=180.0, # bump should resolve within this time (tune)
    baseline_win_s=20.0   # rolling median baseline window
):
    if len(df_seq) < 10:
        return False, {}

    t = pd.to_datetime(df_seq[time_col])
    v = pd.to_numeric(df_seq[value_col], errors="coerce")

    # sampling time
    dt = t.diff().dt.total_seconds().median()
    if not np.isfinite(dt) or dt <= 0:
        return False, {}

    v = v.reset_index(drop=True)
    t = t.reset_index(drop=True)

    # rolling median baseline (slowly varying)
    win = max(5, int(round(baseline_win_s / dt)))
    baseline = v.rolling(win, center=True, min_periods=max(3, win//2)).median()

    resid = v - baseline

    # robust noise estimate using MAD of first differences (handles drifting baseline)
    dv = resid.diff().dropna()
    mad = np.median(np.abs(dv - np.median(dv)))
    sigma = 1.4826 * mad if mad > 0 else np.nan

    if not np.isfinite(sigma) or sigma == 0:
        # fallback: MAD on residual itself
        r = resid.dropna()
        mad2 = np.median(np.abs(r - np.median(r)))
        sigma = 1.4826 * mad2 if mad2 > 0 else 0.0

    amp_thresh = max(min_amp_mm, k_amp * sigma)
    return_band = max(1.0, k_return * sigma)

    # bump mask
    bump = resid.abs() >= amp_thresh
    bump = bump.fillna(False).to_numpy()

    # find contiguous bump segments
    if not bump.any():
        return False, {"sigma": sigma, "amp_thresh": amp_thresh}

    # segment runs
    idx = np.where(bump)[0]
    runs = []
    s = idx[0]
    prev = idx[0]
    for ii in idx[1:]:
        if ii == prev + 1:
            prev = ii
        else:
            runs.append((s, prev))
            s = prev = ii
    runs.append((s, prev))

    # evaluate each run: duration + peak prominence + return to baseline
    for a, b in runs:
        dur = (b - a + 1) * dt
        if dur < min_duration_s or dur > max_duration_s:
            continue

        peak = np.nanmax(np.abs(resid.iloc[a:b+1]))
        if peak < amp_thresh:
            continue

        # "return" check: after event, do we get back close to baseline?
        after = resid.iloc[b+1:min(len(resid), b+1+int(round(max_duration_s/dt)))]
        if len(after) == 0:
            continue

        returned = np.nanmin(np.abs(after)) <= return_band
        if returned:
            info = {
                "sigma": sigma,
                "amp_thresh": amp_thresh,
                "return_band": return_band,
                "run_start_idx": a,
                "run_end_idx": b,
                "duration_s": dur,
                "peak_mm": float(peak),
            }
            return True, info

    return False, {"sigma": sigma, "amp_thresh": amp_thresh, "return_band": return_band}

def detect_excursion_event_robust(
    seg,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    excursion_mm=8.0,
    min_duration_s=5.0
):
    y = seg[value_col].values
    t = pd.to_datetime(seg[time_col]).values

    baseline = np.median(y)
    resid = y - baseline

    mask = np.abs(resid) >= excursion_mm

    if not mask.any():
        return False, {}

    # find contiguous runs
    idx = np.where(mask)[0]
    runs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)

    for run in runs:
        duration = (t[run[-1]] - t[run[0]]) / np.timedelta64(1, "s")
        if duration >= min_duration_s:
            return True, {
                "start_idx": int(run[0]),
                "end_idx": int(run[-1]),
                "duration_s": float(duration),
                "peak_mm": float(np.max(np.abs(resid[run]))),
            }

    return False, {}

def detect_slow_drift_event(
    seg,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    min_run_s=60.0,
    min_total_amp_mm=10.0
):
    """
    Detect a slow drift: a sustained monotonic trend with sufficient duration
    and total amplitude.
    """
    y = seg[value_col].values
    t = pd.to_datetime(seg[time_col]).values

    if len(y) < 3:
        return False, {}

    dy = np.diff(y)

    # Ignore near-zero steps (noise)
    eps = 0.05
    sign = np.sign(dy)
    sign[np.abs(dy) < eps] = 0

    run_start = None
    current_sign = 0

    for i in range(len(sign)):
        if sign[i] == 0:
            continue

        if current_sign == 0:
            # start a new run
            current_sign = sign[i]
            run_start = i
            continue

        if sign[i] != current_sign:
            # end current run
            run_end = i
            duration = (t[run_end] - t[run_start]) / np.timedelta64(1, "s")
            amplitude = abs(y[run_end] - y[run_start])

            if duration >= min_run_s and amplitude >= min_total_amp_mm:
                return True, {
                    "start_idx": int(run_start),
                    "end_idx": int(run_end),
                    "duration_s": float(duration),
                    "amplitude_mm": float(amplitude),
                    "direction": "up" if current_sign > 0 else "down",
                }

            # reset
            current_sign = sign[i]
            run_start = i

    return False, {}


def detect_high_variability_event(
    seg,
    value_col="Mold Level",
    time_col="plainTimeStamp",
    # thresholds
    range_mm=10.0,              # peak-to-peak range threshold
    frac_outside=0.10,          # fraction of samples outside band
    band_mm=4.0,                # band around baseline
):
    y = pd.to_numeric(seg[value_col], errors="coerce").to_numpy()
    if np.isfinite(y).sum() < 10:
        return False, {}

    baseline = np.nanmedian(y)
    resid = y - baseline

    # 1) peak-to-peak range
    ptp = np.nanmax(y) - np.nanmin(y)

    # 2) fraction outside baseline band
    frac = np.nanmean(np.abs(resid) >= band_mm)

    flag = (ptp >= range_mm) or (frac >= frac_outside)

    return bool(flag), {
        "baseline": float(baseline),
        "ptp_mm": float(ptp),
        "frac_outside": float(frac),
        "band_mm": float(band_mm),
    }


def _unique_quality(seg, col):
    if col not in seg.columns:
        return np.nan

    vals = (
        seg[col]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    if len(vals) == 0:
        return np.nan
    elif len(vals) == 1:
        return vals[0]
    else:
        return "MULTIPLE: " + " | ".join(sorted(vals))

# ---------- combined per-seq statistics + disturbance flags ----------
import numpy as np
import pandas as pd

def generate_seq_statistics_with_disturbance(
    df,
    sequences,
    seq_condition,
    # ---- column names (match your raw DF) ----
    tcol="plainTimeStamp",
    quality_col="Quality casting",
    vc_col="castingSpeed",
    mold_width_col="moldWidth",
    sen_col="SENImmersionDepth",
    mold_col="Mold Level",
    ar_sen_col="Argon Flow SEN",
    ar_stop_col="Argon Flow Stopper",
    dc_low_col="EMBR Current DC Left Bottom",
    dc_up_col="EMBR Current DC Left Master",
    ac_up_col="EMBR Current AC Left Master",
    # ---- disturbance thresholds ----
    excursion_mm=8.0,
    excursion_min_duration_s=5.0,
    slow_min_run_s=60.0,
    slow_min_amp_mm=10.0,
    # bump detector params (passed through)
    bump_kwargs=None,
    # variability detector params
    var_range_mm=10.0,
    var_band_mm=4.0,
    var_frac_outside=0.10,
    # ---- robustness ----
    enforce_required_cols=False,   # if True -> raise if any missing, else fill NaN
):
    bump_kwargs = bump_kwargs or {}

    # Work on a copy; ensure datetime once
    df2 = df.copy()
    df2[tcol] = pd.to_datetime(df2[tcol], errors="coerce")

    def _series(seg, c):
        if c in seg.columns:
            return seg[c]
        if enforce_required_cols:
            raise KeyError(f"Missing required column: {c}")
        return pd.Series([np.nan] * len(seg))

    def _mean(seg, c): return float(_series(seg, c).mean())
    def _std(seg, c):  return float(_series(seg, c).std(ddof=1))
    def _min(seg, c):  return float(_series(seg, c).min())
    def _max(seg, c):  return float(_series(seg, c).max())

    rows = []

    for cnt, group in enumerate(sequences):
        if len(group) == 0:
            continue
        if max(group) >= len(df2):
            raise IndexError("positional indexers are out-of-bounds")

        seg = df2.iloc[group].copy().reset_index(drop=True)

        # --- timestamps ---
        ts = seg[tcol].to_list()
        t_start = ts[0] if ts else None
        t_end   = ts[-1] if ts else None

        # --- disturbance detectors ---
        has_exc, exc_info = detect_excursion_event_robust(
            seg,
            value_col=mold_col,
            time_col=tcol,
            excursion_mm=excursion_mm,
            min_duration_s=excursion_min_duration_s
        )

        has_slow, slow_info = detect_slow_drift_event(
            seg,
            value_col=mold_col,
            time_col=tcol,
            min_run_s=slow_min_run_s,
            min_total_amp_mm=slow_min_amp_mm
        )

        has_bump, bump_info = detect_transient_bump_dynamic(
            seg,
            value_col=mold_col,
            time_col=tcol,
            **bump_kwargs
        )

        has_var, var_info = detect_high_variability_event(
            seg,
            value_col=mold_col,
            time_col=tcol,
            range_mm=var_range_mm,
            band_mm=var_band_mm,
            frac_outside=var_frac_outside
        )

        has_dist = bool(has_exc or has_slow or has_bump or has_var)

        # --- Argon total ---
        ar_total = _series(seg, ar_sen_col) + _series(seg, ar_stop_col)

        # --- Build row ---
        row = {
            # identifiers
            "Seq_Name": f"Seq#{cnt}",
            "Seq_time_Start": t_start,
            "Seq_time_End": t_end,
            "Seq_Condition": seq_condition,

            # SEN stats
            "SEN_avg [mm]": _mean(seg, sen_col),
            "SEN_std [mm]": _std(seg, sen_col),

            # casting speed stats
            "CASTING_SPEED_avg [m/min]": _mean(seg, vc_col),
            "CASTING_SPEED_std [m/min]": _std(seg, vc_col),

            # mold width stats
            "MOLD_WIDTH_avg [m]": _mean(seg, mold_width_col),
            "MOLD_WIDTH_std [m]": _std(seg, mold_width_col),

            # DC current low stats
            "min DC Current, Low [A]": _min(seg, dc_low_col),
            "mean DC Current, Low [A]": _mean(seg, dc_low_col),
            "max DC Current, Low [A]": _max(seg, dc_low_col),

            # DC current up stats
            "min DC Current, Up [A]": _min(seg, dc_up_col),
            "mean DC Current, Up [A]": _mean(seg, dc_up_col),
            "max DC Current, Up [A]": _max(seg, dc_up_col),

            # AC current up stats
            "min AC Current, Up [A]": _min(seg, ac_up_col),
            "mean AC Current, Up [A]": _mean(seg, ac_up_col),
            "max AC Current, Up [A]": _max(seg, ac_up_col),

            # mold level stats
            "MOLD_LEVEL_avg [mm]": _mean(seg, mold_col),
            "MOLD_LEVEL_std [mm]": _std(seg, mold_col),

            # argon flow stats
            "ArFlow_avg [NL/min]": float(ar_total.mean()) if len(ar_total) else np.nan,
            "ArFlow_std [NL/min]": float(ar_total.std(ddof=1)) if len(ar_total) else np.nan,

            # Steel Grade
            "Quality casting": _unique_quality(seg, quality_col),


            # disturbance flags
            "has_excursion_event": bool(has_exc),
            "has_slow_drift": bool(has_slow),
            "has_transient_bump": bool(has_bump),
            "has_high_variability": bool(has_var),
            "has_disturbance": bool(has_dist),

            # variability diagnostics
            "ptp_mm": var_info.get("ptp_mm", np.nan),
            "frac_outside_band": var_info.get("frac_outside", np.nan),

            # excursion diagnostics
            "excursion_peak_mm": exc_info.get("peak_mm", np.nan),
            "excursion_duration_s": exc_info.get("duration_s", np.nan),

            # slow drift diagnostics
            "slow_drift_amp_mm": slow_info.get("amplitude_mm", np.nan),
            "slow_drift_duration_s": slow_info.get("duration_s", np.nan),

            # bump diagnostics
            "bump_peak_mm": bump_info.get("peak_mm", np.nan),
            "bump_duration_s": bump_info.get("duration_s", np.nan),
            "sigma_noise": bump_info.get("sigma", np.nan),
        }

        rows.append(row)

    return pd.DataFrame(rows)


In [0]:
# 3) Usage
window_size = 300
Vc_threshold = 0.1
Vc_column_str6 = "castingSpeed"
Curr_columns_str6 = ['EMBR Current AC Left Master', 'EMBR Current DC Left Master', 'EMBR Current DC Left Bottom']
Curr_threshold = 50

#df_seq_str23_6, normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = (
#    identify_sequences_segmented_with_disturbance_flags(
#        df_FCMold_ON_str23_6,
#        Vc_column=Vc_column_str6,
#        window_size=window_size,
##        Vc_threshold=Vc_threshold,
#        Curr_columns=Curr_columns_str6,
#        Curr_threshold=Curr_threshold,
#        tcol="plainTimeStamp",
#        max_gap_seconds=5,
#        min_segment_len=window_size,
#        mold_col="Mold Level",
#        excursion_mm=8.0,
#        excursion_min_duration_s=5.0,
#        slow_min_run_s=60.0,
#        slow_min_amp_mm=10.0,
#    )
#)

#df_seq_str23_6.head()


In [0]:
df_FCMold_ON_str23_6.head(5)

In [0]:
Vc_column_str6 = "castingSpeed"

normal_groups_ON_str23_6, abnormal_groups_ON_str23_6 = identify_sequences_segmented(
    df_FCMold_ON_str23_6,
    Vc_column=Vc_column_str6,
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns_str6,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5,          # treat gaps >5s as new segments
    min_segment_len=window_size # ignore segments shorter than one window
)

len(normal_groups_ON_str23_6), len(abnormal_groups_ON_str23_6)

In [0]:
df_seq_normal_ON_str23_6 = generate_seq_statistics_with_disturbance(
    df=df_FCMold_ON_str23_6,
    sequences=normal_groups_ON_str23_6,
    seq_condition="ON",
    excursion_mm=8.0,
    excursion_min_duration_s=5.0,
    slow_min_run_s=60.0,
    slow_min_amp_mm=10.0
)


In [0]:
df_seq_normal_ON_str23_6.head(5)

In [0]:
def classify_disturbance(row):
    flags = []
    if row["has_excursion_event"]:
        flags.append("Excursion")
    if row["has_high_variability"]:
        flags.append("High variability")
    if row.get("has_transient_bump", False):
        flags.append("Transient bump")
    if row.get("has_slow_drift", False):
        flags.append("Slow drift")

    return "Normal" if not flags else " + ".join(flags)

def assign_disturbance_type(row):
    """
    Assign a human-readable disturbance type based on flags.
    Priority is intentional: excursion > slow drift > bump > variability > normal
    """

    flags = []

    if row.get("has_excursion_event", False):
        flags.append("Excursion")

    if row.get("has_slow_drift", False):
        flags.append("Slow drift")

    if row.get("has_transient_bump", False):
        flags.append("Transient bump")

    if row.get("has_high_variability", False):
        flags.append("High variability")

    if not flags:
        return "Normal"

    return " + ".join(flags)


df_seq_normal_ON_str23_6 = df_seq_normal_ON_str23_6.copy()
df_seq_normal_ON_str23_6["disturbance_type"] = (
    df_seq_normal_ON_str23_6.apply(assign_disturbance_type, axis=1)
)
df_seq_normal_ON_str23_6["disturbance_type"].value_counts()


In [0]:
df_seq_normal_ON_str23_6.head(5)

In [0]:

disturbance_order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + Transient bump + High variability",
]
disturbance_cols = [
    "has_disturbance",
    "has_excursion_event",
    "has_slow_drift",
    "has_transient_bump",
    "has_high_variability",
]


df_seq_normal_ON_str23_6["disturbance_type"] = pd.Categorical(
    df_seq_normal_ON_str23_6["disturbance_type"],
    categories=disturbance_order,
    ordered=True
)


df_seq_normal_ON_str23_6[disturbance_cols].sum().to_frame("count")


In [0]:
df_seq = df_seq_normal_ON_str23_6.copy()

# quick overlap table
pd.crosstab(df_seq["has_excursion_event"], df_seq["has_high_variability"], margins=True)


In [0]:
df_seq_normal_ON_str23_6["disturbance_type"].value_counts()


In [0]:
df_seq_normal_ON_str23_6["disturbance_type"].unique()


In [0]:
df_seq = df_seq_normal_ON_str23_6.copy()

def classify_disturbance(row):
    flags = []
    if row.get("has_excursion_event", False):   flags.append("Excursion")
    if row.get("has_slow_drift", False):        flags.append("Slow drift")
    if row.get("has_transient_bump", False):    flags.append("Transient bump")
    if row.get("has_high_variability", False):  flags.append("High variability")
    return "Normal" if not flags else " + ".join(flags)

df_seq["disturbance_type"] = df_seq.apply(classify_disturbance, axis=1)
df_seq["disturbance_type"].value_counts()


In [0]:
import plotly.express as px

fig = px.box(
    df_seq,
    x="CASTING_SPEED_avg [m/min]",
    y="MOLD_LEVEL_std [mm]",
    color="disturbance_type",
    points="all",
    hover_data=["Seq_Name", "disturbance_type"],
    title="Casting Speed vs Mold Level σ — by disturbance type",
    labels={
        "CASTING_SPEED_avg [m/min]": "Casting Speed Avg [m/min]",
        "MOLD_LEVEL_std [mm]": "Mold Level σ [mm]",
        "disturbance_type": "Disturbance type",
    },
)

# threshold
fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="green",
    annotation_text="σ = 2 mm (stability threshold)",
    annotation_position="top right",
)
fig.update_layout(
    boxmode="overlay",
    height=700,
    width=1150,

    legend=dict(
        title="Disturbance type",
        x=0.02,                 # inside, left
        y=0.98,                 # inside, top
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1,
        traceorder="reversed"   # reverse legend order
    )
)

fig.update_annotations(visible=False)
# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_casting_speed_and_MLsigma_correlations.html"
)
fig.show()


In [0]:
import pandas as pd
import plotly.express as px

#dfp = df_seq_normal_ON_str23_6.copy()
dfp = df_seq

# 1) Make sure disturbance_type exists (if already created, you can skip this block)
def assign_disturbance_type(row):
    flags = []
    if row.get("has_excursion_event", False):   flags.append("Excursion")
    if row.get("has_slow_drift", False):        flags.append("Slow drift")
    if row.get("has_transient_bump", False):    flags.append("Transient bump")
    if row.get("has_high_variability", False):  flags.append("High variability")
    return "Normal" if not flags else " + ".join(flags)

if "disturbance_type" not in dfp.columns:
    dfp["disturbance_type"] = dfp.apply(assign_disturbance_type, axis=1)

# 2) Round casting speed for grouping (same idea as your example)
dfp["CASTING_SPEED_avg_rounded"] = dfp["CASTING_SPEED_avg [m/min]"].round(1)

# Optional: enforce a clean legend order (adjust to what exists in your data)
disturbance_order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + High variability",
    "Excursion + Transient bump + High variability",
]
present = [c for c in disturbance_order if c in set(dfp["disturbance_type"])]
# add any unexpected categories at the end
present += [c for c in dfp["disturbance_type"].unique() if c not in present]

# 3) Boxplot with points overlay, colored by disturbance type
fig = px.box(
    dfp,
    x="CASTING_SPEED_avg_rounded",
    y="MOLD_LEVEL_std [mm]",
    color="disturbance_type",
    category_orders={"disturbance_type": present},
    points="all",
    hover_data=["Seq_Name", "disturbance_type"],
    labels={
        "CASTING_SPEED_avg_rounded": "Casting Speed Avg [m/min] (rounded)",
        "MOLD_LEVEL_std [mm]": "Mold Level σ [mm]",
        "disturbance_type": "Disturbance type",
    },
    title="Strand#23-6 — Casting Speed vs Mold Level σ (by disturbance type)",
)

# 4) Threshold line (σ = 2 mm)
fig.add_hline(
    y=2.0,
    line_dash="dash",
    line_color="green",
    annotation_text="Stability threshold (σ = 2 mm)",
    annotation_position="top right",
)

# 5) Layout + legend inside the plot
fig.update_layout(
    boxmode="overlay",
    height=700,
    width=1150,
    legend=dict(
        title="Disturbance type",
        x=0.02, y=0.98,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        bordercolor="black",
        borderwidth=1,
        traceorder="reversed",
    ),
)

# If you previously added annotations elsewhere, hide them
fig.update_annotations(visible=False)

fig.show()


In [0]:
df_seq_normal_process = df_seq_normal_ON_str23_6[~df_seq_normal_ON_str23_6["has_disturbance"]]


In [0]:
df_seq_normal_process.head(5)

In [0]:
df_seq_normal_process.to_csv("/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_seq_normal_summary.csv", index=False )


In [0]:
import numpy as np
import pandas as pd

df_seq = df_seq_normal_ON_str23_6.copy()

def classify_disturbance(row):
    flags = []
    if bool(row.get("has_excursion_event", False)):   flags.append("Excursion")
    if bool(row.get("has_high_variability", False)):  flags.append("High variability")
    if bool(row.get("has_transient_bump", False)):    flags.append("Transient bump")
    if bool(row.get("has_slow_drift", False)):        flags.append("Slow drift")
    return "Normal" if not flags else " + ".join(flags)

df_seq["disturbance_type"] = df_seq.apply(classify_disturbance, axis=1)

# pick one example index per type
# option A (simple): first occurrence
seq_idx_by_type = df_seq.groupby("disturbance_type").head(1).reset_index()["index"].to_dict()

# Better option B: pick the "strongest" example per type (more illustrative)
# Uncomment this if you want strongest by ptp_mm
# seq_idx_by_type = (
#     df_seq.sort_values(["disturbance_type", "ptp_mm"], ascending=[True, False])
#           .groupby("disturbance_type")
#           .head(1)
#           .reset_index()
#           .set_index("disturbance_type")["index"]
#           .to_dict()
# )

seq_idx_by_type


In [0]:
import plotly.graph_objects as go
import plotly.subplots as sp

# keys must be disturbance type NAMES (strings)
# e.g.:
seq_idx_by_type = {
     "Normal": 127,
     "High variability": 950,
     "Excursion": 956,
     "Excursion + Transient bump + High variability": 332
 }
disturbance_label_map = {
    0: "Normal",
    1: "High variability",
    2: "Excursion",
    3: "Excursion + Transient bump + High variability",
}

types = list(seq_idx_by_type.keys())


nrows = len(types)

fig = sp.make_subplots(
    rows=nrows,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=[
    f"Disturbance type: {disturbance_label_map.get(t, str(t))}  |  Sequence index: {seq_idx_by_type[t]}"
    for t in types
  ],

)

for r, dtype in enumerate(types, start=1):
    seq_idx = seq_idx_by_type[dtype]

    # raw sequence data
    df_seq_raw = df_FCMold_ON_str23_6.iloc[
        normal_groups_ON_str23_6[seq_idx]
    ].copy()

    mean_value = df_seq_raw["Mold Level"].mean()
    min_value  = df_seq_raw["Mold Level"].min()
    max_value  = df_seq_raw["Mold Level"].max()

    # Mold Level signal
    fig.add_trace(
        go.Scatter(
            x=df_seq_raw["plainTimeStamp"],
            y=df_seq_raw["Mold Level"],
            mode="lines",
            name=dtype,
            hovertemplate=(
                "Time=%{x}<br>"
                "Mold Level=%{y:.2f} mm<br>"
                f"Type={dtype}"
                "<extra></extra>"
            ),
        ),
        row=r,
        col=1,
    )

    # Mean / Min / Max reference lines
    fig.add_hline(
        y=mean_value,
        line_dash="dash",
        line_color="green",
        row=r,
        col=1,
    )
    fig.add_hline(
        y=min_value,
        line_dash="dot",
        line_color="blue",
        row=r,
        col=1,
    )
    fig.add_hline(
        y=max_value,
        line_dash="dot",
        line_color="red",
        row=r,
        col=1,
    )

    fig.update_yaxes(title_text="Mold Level [mm]", row=r, col=1)

fig.update_xaxes(title_text="Time", row=nrows, col=1)

fig.update_layout(
    height=320 * nrows,
    width=1200,
    title="Representative sequences per disturbance type",
    showlegend=False,
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_seq_groupings_by_disturbance_type.html")

fig.show()


In [0]:
import numpy as np
import matplotlib.pyplot as plt

dfp = df_seq_normal_ON_str23_6.copy()

# --- data vectors ---
x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()

# categorical color = disturbance type (must exist)
dtype = dfp["disturbance_type"].astype(str).to_numpy()

# filter finite
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x, y, ml_std, dtype = x[mask], y[mask], ml_std[mask], dtype[mask]

above_thr = ml_std > 2.0

# --- choose a stable order for legend (optional) ---
order = [
    "Normal",
    "High variability",
    "Transient bump",
    "Slow drift",
    "Excursion",
    "Excursion + High variability",
    "Excursion + Transient bump + High variability",
]
present = [c for c in order if c in set(dtype)]
# add any unexpected categories at the end
present += [c for c in np.unique(dtype) if c not in present]

# --- map categories -> colors using matplotlib default cycle ---
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_map = {cat: colors[i % len(colors)] for i, cat in enumerate(present)}
point_colors = np.array([color_map[c] for c in dtype])

# --- plot ---
fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) Base scatter (all points)
ax.scatter(
    x, y,
    c=point_colors,
    s=40,
    edgecolor="none",
    alpha=0.8,
)

# 2) Highlight σ > 2 mm with black edge (same colors, just outlined)
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=point_colors[above_thr],
    s=70,
    edgecolor="black",
    linewidth=0.9,
    alpha=0.95,
)

# --- legend (disturbance types) ---
handles = [
    plt.Line2D([0], [0], marker="o", color="none",
               markerfacecolor=color_map[cat],
               markersize=8, label=cat)
    for cat in present
]
ax.legend(
    handles=handles,
    title="Disturbance type",
    loc="best",
    frameon=True,
)

ax.set_xlabel("Mold Width avg [m]")
ax.set_ylabel("Casting Speed avg [m/min]")
ax.set_title("Mold Width vs Casting Speed colored by disturbance type\n(black outline: MOLD_LEVEL_std > 2 mm)")

# Save & show
fig.savefig(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_Vc_MoldWidth_sequences_colored.png")
plt.show()


In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# --- filter to NORMAL sequences only ---
dfp = df_seq_normal_ON_str23_6[
    df_seq_normal_ON_str23_6["disturbance_type"] == "Normal"
].copy()

# --- custom colormap (same as your reference) ---
ml_cmap = LinearSegmentedColormap.from_list(
    "ml_sigma_cmap",
    [
        (0.0, "#440053"),   # dark purple (very low σ)
        (0.3, "#404388"),
        (0.6, "#2a788e"),
        (0.8, "#21a784"),
        (0.95, "#78d151"),
        (1.0, "#fde624"),   # yellow (very high σ)
    ],
    N=256,
)

# --- data vectors ---
x = dfp["MOLD_WIDTH_avg [m]"].to_numpy()
y = dfp["CASTING_SPEED_avg [m/min]"].to_numpy()
ml_std = dfp["MOLD_LEVEL_std [mm]"].to_numpy()

# keep finite only
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(ml_std)
x, y, ml_std = x[mask], y[mask], ml_std[mask]

# threshold
above_thr = ml_std > 2.0
ml_std_max = np.nanmax(ml_std)

# threshold-aware normalization
norm = TwoSlopeNorm(vmin=0, vcenter=2.0, vmax=ml_std_max)

# --- plot ---
fig, ax = plt.subplots(figsize=(7, 5), tight_layout=True)

# 1) base scatter (Normal only, colored by σ)
sc = ax.scatter(
    x,
    y,
    c=ml_std,
    cmap=ml_cmap,
    norm=norm,
    s=40,
    edgecolor="none",
    alpha=0.85,
)

# 2) highlight σ > 2 mm with black outline
ax.scatter(
    x[above_thr],
    y[above_thr],
    c=ml_std[above_thr],
    cmap=ml_cmap,
    norm=norm,
    s=70,
    edgecolor="black",
    linewidth=0.9,
    alpha=0.95,
)

# colorbar
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("MOLD_LEVEL_std [mm]")
cbar.ax.axhline(2.0, color="black", linewidth=1)
cbar.ax.text(
    1.05, 2.0, "σ = 2 mm",
    transform=cbar.ax.get_yaxis_transform(),
    va="center"
)

# labels & title
ax.set_xlabel("Mold Width avg [m]")
ax.set_ylabel("Casting Speed avg [m/min]")
ax.set_title("Normal sequences only\nColored by Mold Level σ (black outline: σ > 2 mm)")

plt.show()


In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helpers
# -------------------------
def make_box_points_and_meanline(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace, mean_line_trace)
    """
    x_round_name = f"{x_col}__rounded"
    df = df.copy()
    df[x_round_name] = df[x_col].round(round_to)

    # -------------------------
    # Box trace
    # -------------------------
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        boxmean=False,
        marker=dict(color=color),
        line=dict(color=color),
        opacity=0.45,
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        )
    )

    # -------------------------
    # Scatter trace
    # -------------------------
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.8),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False
    )

    # -------------------------
    # Mean line (per bin)
    # -------------------------
    mean_df = (
        df.groupby(x_round_name, as_index=False)["MOLD_LEVEL_std [mm]"]
        .mean()
        .sort_values(x_round_name)
    )

    mean_line = go.Scatter(
        x=mean_df[x_round_name],
        y=mean_df["MOLD_LEVEL_std [mm]"],
        mode="lines+markers",
        line=dict(color="black", width=2),
        marker=dict(size=6),
        name="Mean trend",
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mean Mold Level σ: %{y:.3f} mm<extra></extra>"
        )
    )

    return box, scatter, mean_line


def make_box_and_points(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace) for a given x_col vs Mold Level std.
    Boxes are grouped by rounded x to create meaningful distributions.
    """
    x_round_name = f"{x_col}__rounded"
    df[x_round_name] = df[x_col].round(round_to)

    # Box trace (distribution per rounded bin)
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        name=None,
        boxmean="sd",
        marker=dict(color=color, opacity=0.85),
        line=dict(color=color, width=1),
        opacity=0.55,               # box opacity
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        )
    )

    # Scatter trace (points on top)
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.85),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False,
        name=None
    )

    return box, scatter

# -------------------------
# Subplot layout (2x2)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Vc vs Mold Level σ (Normal only)",
        "Mold Width vs Mold Level σ (Normal only)",
        "SEN vs Mold Level σ (Normal only)",
        "Argon Flow vs Mold Level σ (Normal only)",
    ]
)

panels = [
    (1, 1, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]"),
    (1, 2, "MOLD_WIDTH_avg [m]",        "Mold Width Avg [m]"),
    (2, 1, "SEN_avg [mm]",              "SEN Avg [mm]"),
    (2, 2, "ArFlow_avg [NL/min]",       "Ar Flow Avg [NL/min]"),
]

for r, c, xcol, xlabel in panels:
    box_tr, sc_tr, mean_tr = make_box_points_and_meanline(
    df, xcol, xlabel, round_to=1, color="#bdb76b"
    )

    fig.add_trace(box_tr, row=r, col=c)
    fig.add_trace(sc_tr, row=r, col=c)
    fig.add_trace(mean_tr, row=r, col=c)


    # axis titles
    fig.update_xaxes(title_text=f"{xlabel} (rounded)", row=r, col=c)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=r, col=c)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Normal process only: distributions (box + points)",
    boxmode="overlay",
    showlegend=False,
)

# Save & show
fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_cc_parameters_and_ML_correlations_NORMAL_only_boxpoints.html"
)
fig.show()


In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

YCOL = "MOLD_LEVEL_std [mm]"

# -------------------------
# Helpers
# -------------------------
def add_box_points_mean(fig, df, x_col, x_label, row, col, bin_size, color="#bdb76b"):
    """
    Box + points + mean line using coarse bins.
    bin_size is in the units of x_col (e.g., 0.1 m/min, 0.05 m).
    """
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    # Bin x into coarse categories (works better than rounding for continuous-ish vars)
    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # 1) Boxplot
    fig.add_trace(
        go.Box(
            x=d[x_bin],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.45,
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                f"Mold Level σ [mm]: %{{y:.3f}}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    # 2) Points
    fig.add_trace(
        go.Scatter(
            x=d[x_bin],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label} (bin): %{{x}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # 3) Mean trend per bin
    mean_df = (
        d.groupby(x_bin, as_index=False)[YCOL]
        .mean()
        .sort_values(x_bin)
    )

    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color="white", width=2),
            marker=dict(size=6, color="white"),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label} (binned, Δ={bin_size})", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_scatter_mean(fig, df, x_col, x_label, row, col, bin_size, point_color="#bdb76b", line_color="white"):
    """
    Scatter + mean trend line only (no box). Uses bins to stabilize the mean line.
    """
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # 1) Scatter (raw x, not binned) for truthful geometry
    fig.add_trace(
        go.Scatter(
            x=d[x_col],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=point_color, size=6, opacity=0.55),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.4g}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # 2) Mean trend (computed on binned x, plotted at binned x)
    mean_df = (
        d.groupby(x_bin, as_index=False)[YCOL]
        .mean()
        .sort_values(x_bin)
    )

    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label}", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


# -------------------------
# Subplots (2×2)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Casting speed: box + points + mean (best)",
        "Mold width: box + points + mean (coarse bins)",
        "SEN: scatter + mean trend (no box)",
        "Argon flow: scatter + mean trend (no box)",
    ]
)

# 1) Casting speed: discrete setpoints → box works well (bin ~ 0.1 m/min)
add_box_points_mean(
    fig, df,
    x_col="CASTING_SPEED_avg [m/min]",
    x_label="Casting Speed Avg [m/min]",
    row=1, col=1,
    bin_size=0.1
)

# 2) Mold width: coarse bins needed (e.g. 0.05 m)
add_box_points_mean(
    fig, df,
    x_col="MOLD_WIDTH_avg [m]",
    x_label="Mold Width Avg [m]",
    row=1, col=2,
    bin_size=0.05
)

# 3) SEN: continuous → no box, mean trend stabilised with small bins (e.g. 0.001 m ~ 1 mm)
# Your SEN column label says [mm] but values look like meters (0.16–0.22).
# If it's meters, 0.001 = 1 mm. If it's truly mm, change bin_size accordingly.
add_scatter_mean(
    fig, df,
    x_col="SEN_avg [mm]",
    x_label="SEN Avg",
    row=2, col=1,
    bin_size=0.001
)

# 4) Argon flow: continuous → no box, mean trend stabilised with bins (e.g. 0.5 NL/min)
add_scatter_mean(
    fig, df,
    x_col="ArFlow_avg [NL/min]",
    x_label="Ar Flow Avg [NL/min]",
    row=2, col=2,
    bin_size=0.5
)

# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1200,
    title="Strand #23-6 – Normal process only: recommended summaries",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_6_recommended_correlations_NORMAL_only.html"
)
fig.show()


In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()   # aggregated df (per-sequence)
YCOL = "MOLD_LEVEL_std [mm]"
QCOL = "Quality casting"

# -------------------------
# Helpers
# -------------------------
def add_box_points_mean(fig, df, x_col, x_label, row, col, bin_size, color="#bdb76b"):
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # Box
    fig.add_trace(
        go.Box(
            x=d[x_bin],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.45,
            showlegend=False,
        ),
        row=row, col=col
    )

    # Points
    fig.add_trace(
        go.Scatter(
            x=d[x_bin],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label} (bin): %{{x}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # Mean line per bin
    mean_df = d.groupby(x_bin, as_index=False)[YCOL].mean().sort_values(x_bin)
    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color="black", width=2),
            marker=dict(size=6, color="black"),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=f"{x_label} (binned, Δ={bin_size})", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_scatter_mean(fig, df, x_col, x_label, row, col, bin_size, point_color="#bdb76b", line_color="black"):
    d = df[[x_col, YCOL, "Seq_Name"]].dropna().copy()

    x_bin = f"{x_col}__bin"
    d[x_bin] = (np.round(d[x_col] / bin_size) * bin_size).astype(float)

    # Scatter (raw x)
    fig.add_trace(
        go.Scatter(
            x=d[x_col],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=point_color, size=6, opacity=0.55),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                f"{x_label}: %{{x:.4g}}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    # Mean trend (computed on binned x)
    mean_df = d.groupby(x_bin, as_index=False)[YCOL].mean().sort_values(x_bin)
    fig.add_trace(
        go.Scatter(
            x=mean_df[x_bin],
            y=mean_df[YCOL],
            mode="lines+markers",
            line=dict(color=line_color, width=2),
            marker=dict(size=6, color=line_color),
            showlegend=False,
            hovertemplate=(
                f"{x_label} (bin): %{{x}}<br>"
                "Mean Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text=x_label, row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)


def add_quality_box(fig, df, row, col, color="#bdb76b"):
    d = df[[QCOL, YCOL, "Seq_Name"]].dropna().copy()

    # clean category strings
    d[QCOL] = d[QCOL].astype(str).str.strip()

    # Box
    fig.add_trace(
        go.Box(
            x=d[QCOL],
            y=d[YCOL],
            marker=dict(color=color),
            line=dict(color=color, width=1),
            opacity=0.50,
            showlegend=False,
        ),
        row=row, col=col
    )

    # Points
    fig.add_trace(
        go.Scatter(
            x=d[QCOL],
            y=d[YCOL],
            mode="markers",
            marker=dict(color=color, size=7, opacity=0.80),
            customdata=d[["Seq_Name"]].to_numpy(),
            hovertemplate=(
                "Seq_Name: %{customdata[0]}<br>"
                "Quality casting: %{x}<br>"
                "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row, col=col
    )

    fig.update_xaxes(title_text="Quality casting", row=row, col=col)
    fig.update_yaxes(title_text="Mold Level σ [mm]", row=row, col=col)

# -------------------------
# Layout: 2 rows × 3 cols (5 panels used)
# -------------------------
fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
    subplot_titles=[
        "Casting speed",
        "Mold width",
        "Quality casting",
        "SEN",
        "Argon flow",
        ""  # empty cell (row 2 col 3)
    ]
)

# Row 1
add_box_points_mean(fig, df, "CASTING_SPEED_avg [m/min]", "Casting Speed Avg [m/min]", row=1, col=1, bin_size=0.1)
add_box_points_mean(fig, df, "MOLD_WIDTH_avg [m]",        "Mold Width Avg [m]",        row=1, col=2, bin_size=0.05)
#add_quality_box(fig, df, row=1, col=3)

# Row 2
#add_scatter_mean(fig, df, "SEN_avg [mm]",          "SEN Avg [m]",            row=2, col=1, bin_size=0.001)  # 1 mm bins = 0.001 m
add_quality_box(fig, df, row=2, col=1)
add_scatter_mean(fig, df, "ArFlow_avg [NL/min]",   "Ar Flow Avg [NL/min]",   row=2, col=2, bin_size=0.5)
quality_order = (
    df.groupby("Quality casting")["MOLD_LEVEL_std [mm]"]
      .median()
      .sort_values()
      .index
      .tolist()
)

fig.update_xaxes(
    categoryorder="array",
    categoryarray=quality_order,
    row=1, col=3
)


# -------------------------
# Layout
# -------------------------
fig.update_layout(
    height=900,
    width=1600,
    title="Strand #23-5 – Normal process only",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_5_recommended_correlations_NORMAL_only_plus_quality.html"
)

fig.show()


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helpers
# -------------------------
def make_box_and_points(df, x_col, x_label, round_to=1, color="#bdb76b"):
    """
    Returns (box_trace, scatter_trace) for a given x_col vs Mold Level std.
    Boxes are grouped by rounded x to create meaningful distributions.
    Points are plotted at the same rounded x positions for a clean overlay.
    """
    x_round_name = f"{x_col}__rounded"
    df[x_round_name] = df[x_col].round(round_to)

    # Box trace (distribution per rounded bin)
    box = go.Box(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        boxmean="sd",
        marker=dict(color=color, opacity=0.85),
        line=dict(color=color, width=1),
        opacity=0.55,
        showlegend=False,
        hovertemplate=(
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
    )

    # Scatter trace (points on top)
    scatter = go.Scatter(
        x=df[x_round_name],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.85),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label} (rounded): %{{x}}<br>"
            "Mold Level σ [mm]: %{y:.3f}<extra></extra>"
        ),
        showlegend=False,
    )

    return box, scatter, x_round_name

# -------------------------
# Subplot layout (1x3)
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[
        "mean DC Current, Low vs Mold Level σ (Normal only)",
        "mean DC Current, Up vs Mold Level σ (Normal only)",
        "mean AC Current, Up vs Mold Level σ (Normal only)",
    ],
)

panels = [
    (1, 1, "mean DC Current, Low [A]", "Mean DC Current, Low [A]", 0),  # round_to=0A
    (1, 2, "mean DC Current, Up [A]",  "Mean DC Current, Up [A]",  0),
    (1, 3, "mean AC Current, Up [A]",  "Mean AC Current, Up [A]",  0),
]

for r, c, xcol, xlabel, round_to in panels:
    box_tr, sc_tr, x_round_name = make_box_and_points(df, xcol, xlabel, round_to=round_to, color="#bdb76b")
    fig.add_trace(box_tr, row=r, col=c)
    fig.add_trace(sc_tr, row=r, col=c)

    fig.update_xaxes(title_text=f"{xlabel} (rounded)", row=r, col=c)

fig.update_yaxes(title_text="Mold Level σ [mm]", row=1, col=1)

fig.update_layout(
    height=520,
    width=1400,
    title="Strand #23-6 – Normal only: Mold Level σ vs mean currents (box + points)",
    boxmode="overlay",
    showlegend=False,
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/strand23_6_currents_vs_ML_correlations_NORMAL_only_boxpoints.html"
)

fig.show()


In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = df_seq_normal_process.copy()

# -------------------------
# Helper: pure scatter
# -------------------------
def make_scatter(df, x_col, x_label, color="#bdb76b"):
    return go.Scatter(
        x=df[x_col],
        y=df["MOLD_LEVEL_std [mm]"],
        mode="markers",
        marker=dict(
            size=7,
            color=color,
            opacity=0.75,
        ),
        customdata=df[["Seq_Name"]].to_numpy(),
        hovertemplate=(
            "Seq_Name: %{customdata[0]}<br>"
            f"{x_label}: %{{x:.2f}}<br>"
            "Mold Level σ: %{y:.3f} mm"
            "<extra></extra>"
        ),
        showlegend=False,
    )

# -------------------------
# Subplots (1 × 3)
# -------------------------
fig = make_subplots(
    rows=1, cols=3,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=[
        "mean DC Current, Low vs Mold Level σ (Normal only)",
        "mean DC Current, Up vs Mold Level σ (Normal only)",
        "mean AC Current, Up vs Mold Level σ (Normal only)",
    ],
)

panels = [
    (1, 1, "mean DC Current, Low [A]", "Mean DC Current, Low [A]"),
    (1, 2, "mean DC Current, Up [A]",  "Mean DC Current, Up [A]"),
    (1, 3, "mean AC Current, Up [A]",  "Mean AC Current, Up [A]"),
]

for r, c, xcol, xlabel in panels:
    fig.add_trace(
        make_scatter(df, xcol, xlabel),
        row=r, col=c
    )
    fig.update_xaxes(title_text=xlabel, row=r, col=c)

fig.update_yaxes(title_text="Mold Level σ [mm]", row=1, col=1)

fig.update_layout(
    height=480,
    width=1400,
    title="Strand #23-6 – Normal process only: Mold Level σ vs mean currents",
)

fig.write_html(
    "/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/"
    "strand23_6_currents_vs_ML_scatter_NORMAL_only.html"
)

fig.show()


In [0]:
import plotly.express as px

def plot_one_sequence(
    df_raw,
    df_seq,
    seq_idx,
    sequences,
    *,
    tcol="plainTimeStamp",
    value_col="Mold Level",
    title_prefix="",
):
    """
    Plot raw Mold Level time series for ONE sequence.

    Parameters
    ----------
    df_raw : pd.DataFrame
        Raw time-series dataframe (e.g. df_FCMold_ON_str23_6)
    df_seq : pd.DataFrame
        Aggregated per-sequence dataframe
        (output of generate_seq_statistics_with_disturbance)
    seq_idx : int
        Index of the sequence in df_seq / sequences
    sequences : list[list[int]]
        List of index groups returned by identify_sequences_segmented
    """

    # --- safety checks ---
    if seq_idx >= len(sequences):
        raise IndexError("seq_idx out of range")

    idxs = sequences[seq_idx]
    seg = df_raw.iloc[idxs].copy()

    # --- metadata ---
    meta = df_seq.iloc[seq_idx]

    title = (
        f"{title_prefix}"
        f"{meta['Seq_Name']} | "
        f"{meta['Seq_time_Start']} → {meta['Seq_time_End']}<br>"
        f"Vc={meta['CASTING_SPEED_avg [m/min]']:.2f}, "
        f"ML σ={meta['MOLD_LEVEL_std [mm]']:.2f} mm, "
        f"Disturbance={meta['has_disturbance']}"
    )

    # --- plot ---
    fig = px.line(
        seg,
        x=tcol,
        y=value_col,
        title=title,
        labels={
            tcol: "Time",
            value_col: "Mold Level [mm]",
        },
    )

    # reference lines
    fig.add_hline(
        y=seg[value_col].mean(),
        line_dash="dash",
        line_color="green",
        annotation_text="Mean",
        annotation_position="top left",
    )

    fig.add_hline(
        y=seg[value_col].min(),
        line_dash="dot",
        line_color="blue",
        annotation_text="Min",
        annotation_position="bottom left",
    )

    fig.add_hline(
        y=seg[value_col].max(),
        line_dash="dot",
        line_color="red",
        annotation_text="Max",
        annotation_position="top right",
    )

    fig.update_layout(
        height=400,
        width=1100,
        showlegend=False,
    )

    fig.show()


In [0]:
sequence = 127
plot_one_sequence(df_FCMold_ON_str23_6, df_seq, sequence, normal_groups_ON_str23_6)

# Root Cause Investigation: What Drives Mold Level Instability?

**Analytical approach:**
1. **Feature engineering** — Derive physics-based features: meniscus temperature asymmetry (FBG), Chebyshev flatness coefficients (X₁–X₄), EMBR L-R imbalance, mold level L-R asymmetry
2. **Raw correlation screening** — Rank all features by Spearman ρ with ML σ
3. **Confounding analysis** — EMBR currents are controlled setpoints (f(Vc, Width, SEN)), so partial correlations are needed to disentangle true predictors from confounded ones
4. **Validated deep dive** — Reconstruct meniscus profiles, test temporal causality for the confirmed top predictors

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

# =====================================================================
# 1. Select columns of interest (original + new derived + Chebyshev)
#    All columns now come directly from df_features (no workaround needed)
# =====================================================================
cols_analysis = [
    "plainTimeStamp", "castingSpeed", "moldWidth", "SENImmersionDepth",
    "Mold Level", "Mold Level Sensor Left", "Mold Level Sensor Right",
    "Argon Flow SEN", "Argon Flow Stopper",
    "EMBR Current DC Left Bottom", "EMBR Current AC Left Master",
    "EMBR Current DC Left Master",
    "EMBR Current DC Right Bottom", "EMBR Current AC Right Master",
    "EMBR Current DC Right Master",
    "superHeat", "tundishTemperature",
    "Quality casting",
    # --- Process context ---
    "SEN_type", "castingLength", "castMode",
    # --- Meniscus raw temperature features ---
    "meniscus_bff_avg", "meniscus_bfl_avg", "meniscus_FF_LF_asymmetry",
    "meniscus_bff_range", "meniscus_bfl_range",
    # --- Chebyshev flatness coefficients (X1-X4, BFF & BFL) ---
    "cheb_X1_bff", "cheb_X2_bff", "cheb_X3_bff", "cheb_X4_bff",
    "cheb_X1_bfl", "cheb_X2_bfl", "cheb_X3_bfl", "cheb_X4_bfl",
    "abs_cheb_X1_bff", "abs_cheb_X2_bff",
    "abs_cheb_X1_bfl", "abs_cheb_X2_bfl",
    "cheb_X1_FF_LF_diff", "cheb_X2_FF_LF_diff",
    # --- EMBR L-R imbalance ---
    "EMBR_DC_Bottom_LR_diff", "EMBR_AC_Master_LR_diff", "EMBR_DC_Master_LR_diff",
    # --- Mold Level L-R asymmetry ---
    "ML_LR_asymmetry", "ML_LR_abs_asymmetry",
]

# =====================================================================
# 2. Filter for steady-state
# =====================================================================
df_filtered = (
    df_features
    .select(cols_analysis)
    .filter(F.col("castingSpeed") >= 0.5)
    .filter(F.col("SENImmersionDepth").between(0.1, 0.26))
)

print(f"After steady-state filter: {df_filtered.count():,} rows")

# =====================================================================
# 3. Convert to pandas
# =====================================================================
df_pd = df_filtered.toPandas()
df_pd["plainTimeStamp"] = pd.to_datetime(df_pd["plainTimeStamp"])
df_pd = df_pd.sort_values("plainTimeStamp").reset_index(drop=True)

print(f"Pandas shape: {df_pd.shape}")

# =====================================================================
# 4. Filter FC Mold ON (non-zero EMBR currents)
# =====================================================================
df_fc_on = df_pd[
    (df_pd["EMBR Current AC Left Master"] != 0) &
    (df_pd["EMBR Current DC Left Master"] != 0) &
    (df_pd["EMBR Current DC Left Bottom"] != 0)
].reset_index(drop=True)

print(f"FC Mold ON: {df_fc_on.shape[0]:,} rows ({100*df_fc_on.shape[0]/df_pd.shape[0]:.1f}%)")

df_fc_on["castingSpeed"] = custom_rounding(df_fc_on["castingSpeed"], 2)

# =====================================================================
# 5. Identify sequences (5-min / 300 samples)
# =====================================================================
window_size = 300
Vc_threshold = 0.1
Curr_columns = ["EMBR Current AC Left Master", "EMBR Current DC Left Master", "EMBR Current DC Left Bottom"]
Curr_threshold = 50

normal_groups, abnormal_groups = identify_sequences_segmented(
    df_fc_on,
    Vc_column="castingSpeed",
    window_size=window_size,
    Vc_threshold=Vc_threshold,
    Curr_columns=Curr_columns,
    Curr_threshold=Curr_threshold,
    tcol="plainTimeStamp",
    max_gap_seconds=5
)

print(f"\nSequences: {len(normal_groups)} normal, {len(abnormal_groups)} abnormal")

In [0]:
import pandas as pd
import numpy as np

def generate_seq_statistics_extended(df, sequences, seq_condition):
    """Generate per-sequence statistics including Chebyshev flatness + new derived features."""
    records = []
    
    for cnt, group in enumerate(sequences):
        if max(group) >= len(df):
            continue
        
        t = df.iloc[group]
        
        # --- Helper for safe mode ---
        def safe_mode(series):
            m = series.dropna()
            if len(m) == 0:
                return None
            return m.mode().iloc[0]
        
        rec = {
            'Seq_Name': f'Seq#{cnt}',
            'Seq_time_Start': t['plainTimeStamp'].iloc[0],
            'Seq_time_End': t['plainTimeStamp'].iloc[-1],
            
            # --- Original features ---
            'CASTING_SPEED_avg [m/min]': t['castingSpeed'].mean(),
            'MOLD_WIDTH_avg [m]': t['moldWidth'].mean(),
            'SEN_avg [mm]': t['SENImmersionDepth'].mean(),
            'MOLD_LEVEL_avg [mm]': t['Mold Level'].mean(),
            'MOLD_LEVEL_std [mm]': t['Mold Level'].std(),
            'ArFlow_SEN_avg': t['Argon Flow SEN'].mean(),
            'ArFlow_Stopper_avg': t['Argon Flow Stopper'].mean(),
            'Quality casting': safe_mode(t['Quality casting']),
            
            # =============================================
            # PROCESS CONTEXT (previously underexplored)
            # =============================================
            'castingLength_avg': t['castingLength'].mean(),    # wear proxy: distance into cast
            'castingLength_std': t['castingLength'].std(),     # should be small within 5-min seq
            'SEN_type': safe_mode(t['SEN_type']),              # nozzle type (categorical)
            'castMode': safe_mode(t['castMode']),              # casting mode (categorical)
            
            # --- EMBR Left currents ---
            'EMBR_DC_Bottom_L_avg': t['EMBR Current DC Left Bottom'].mean(),
            'EMBR_AC_Master_L_avg': t['EMBR Current AC Left Master'].mean(),
            'EMBR_DC_Master_L_avg': t['EMBR Current DC Left Master'].mean(),
            
            # =============================================
            # CHEBYSHEV FLATNESS COEFFICIENTS (BFF)
            # X1=tilt, X2=curvature, X3=3rd order, X4=4th order
            # =============================================
            'cheb_X1_bff_mean': t['cheb_X1_bff'].mean(),
            'cheb_X1_bff_std': t['cheb_X1_bff'].std(),
            'cheb_X2_bff_mean': t['cheb_X2_bff'].mean(),
            'cheb_X2_bff_std': t['cheb_X2_bff'].std(),
            'cheb_X3_bff_mean': t['cheb_X3_bff'].mean(),
            'cheb_X4_bff_mean': t['cheb_X4_bff'].mean(),
            
            # Absolute values (shape deformation magnitude)
            'abs_cheb_X1_bff_mean': t['abs_cheb_X1_bff'].mean(),
            'abs_cheb_X2_bff_mean': t['abs_cheb_X2_bff'].mean(),
            
            # =============================================
            # CHEBYSHEV FLATNESS COEFFICIENTS (BFL)
            # =============================================
            'cheb_X1_bfl_mean': t['cheb_X1_bfl'].mean(),
            'cheb_X1_bfl_std': t['cheb_X1_bfl'].std(),
            'cheb_X2_bfl_mean': t['cheb_X2_bfl'].mean(),
            'cheb_X2_bfl_std': t['cheb_X2_bfl'].std(),
            'cheb_X3_bfl_mean': t['cheb_X3_bfl'].mean(),
            'cheb_X4_bfl_mean': t['cheb_X4_bfl'].mean(),
            
            'abs_cheb_X1_bfl_mean': t['abs_cheb_X1_bfl'].mean(),
            'abs_cheb_X2_bfl_mean': t['abs_cheb_X2_bfl'].mean(),
            
            # BFF-BFL differences per Chebyshev coefficient
            'cheb_X1_FF_LF_diff_mean': t['cheb_X1_FF_LF_diff'].mean(),
            'cheb_X2_FF_LF_diff_mean': t['cheb_X2_FF_LF_diff'].mean(),
            
            # =============================================
            # MENISCUS RAW TEMPERATURE
            # =============================================
            'meniscus_bff_avg_mean': t['meniscus_bff_avg'].mean(),
            'meniscus_bfl_avg_mean': t['meniscus_bfl_avg'].mean(),
            'meniscus_FF_LF_asym_mean': t['meniscus_FF_LF_asymmetry'].mean(),
            'meniscus_FF_LF_asym_std': t['meniscus_FF_LF_asymmetry'].std(),
            'meniscus_bff_range_mean': t['meniscus_bff_range'].mean(),
            'meniscus_bfl_range_mean': t['meniscus_bfl_range'].mean(),
            
            # =============================================
            # EMBR L-R CURRENT IMBALANCE
            # =============================================
            'EMBR_DC_Bottom_LR_mean': t['EMBR_DC_Bottom_LR_diff'].mean(),
            'EMBR_DC_Bottom_LR_std': t['EMBR_DC_Bottom_LR_diff'].std(),
            'EMBR_AC_Master_LR_mean': t['EMBR_AC_Master_LR_diff'].mean(),
            'EMBR_AC_Master_LR_std': t['EMBR_AC_Master_LR_diff'].std(),
            'EMBR_DC_Master_LR_mean': t['EMBR_DC_Master_LR_diff'].mean(),
            'EMBR_DC_Master_LR_std': t['EMBR_DC_Master_LR_diff'].std(),
            
            # =============================================
            # MOLD LEVEL L-R ASYMMETRY
            # =============================================
            'ML_LR_asym_mean': t['ML_LR_asymmetry'].mean(),
            'ML_LR_asym_std': t['ML_LR_asymmetry'].std(),
            'ML_LR_abs_asym_mean': t['ML_LR_abs_asymmetry'].mean(),
            
            # =============================================
            # THERMAL CONTEXT
            # =============================================
            'superHeat_avg': t['superHeat'].mean(),
            'tundishTemp_avg': t['tundishTemperature'].mean(),
            
            'Seq_Condition': seq_condition
        }
        records.append(rec)
    
    df_out = pd.DataFrame(records)
    
    # --- Parse Quality casting into steel grade family ---
    # Convention: first char or first digit+letter combo identifies the family
    df_out['quality_family'] = df_out['Quality casting'].apply(
        lambda x: x[0] if isinstance(x, str) and len(x) > 0 else None
    )
    
    return df_out

df_seq_ext = generate_seq_statistics_extended(df_fc_on, normal_groups, 'normal')
print(f"Sequences with extended features: {df_seq_ext.shape}")
print(f"\nNew columns added: castingLength_avg, castingLength_std, SEN_type, castMode, quality_family")
print(f"  SEN_type distribution:  {df_seq_ext['SEN_type'].value_counts().to_dict()}")
print(f"  castMode distribution:  {df_seq_ext['castMode'].value_counts().to_dict()}")
print(f"  quality_family:         {df_seq_ext['quality_family'].value_counts().to_dict()}")
print(f"  castingLength range:    [{df_seq_ext['castingLength_avg'].min():.1f}, {df_seq_ext['castingLength_avg'].max():.1f}] m")
df_seq_ext.head(3)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

target = 'MOLD_LEVEL_std [mm]'

# All numeric feature columns
exclude = ['Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'Seq_Condition', 'Quality casting', target]
feature_cols = [c for c in df_seq_ext.columns if c not in exclude and df_seq_ext[c].dtype in ['float64', 'int64']]

# Compute Pearson and Spearman correlations
corr_results = []
for col in feature_cols:
    valid = df_seq_ext[[col, target]].dropna()
    if len(valid) > 10:
        r_p, p_p = stats.pearsonr(valid[col], valid[target])
        r_s, p_s = stats.spearmanr(valid[col], valid[target])
        corr_results.append({
            'Feature': col, 'Pearson r': r_p, 'Pearson p': p_p,
            'Spearman rho': r_s, 'Spearman p': p_s,
            '|Spearman|': abs(r_s)
        })

df_corr = pd.DataFrame(corr_results).sort_values('|Spearman|', ascending=False)

# Print ranked correlations
print("="*95)
print(f"CORRELATION RANKING vs {target} (n={len(df_seq_ext)} sequences)")
print("="*95)
print(f"{'Feature':<35} {'Pearson r':>10} {'p-val':>10} {'Spearman rho':>12} {'p-val':>10}")
print("-"*95)
for _, row in df_corr.iterrows():
    sig_p = '***' if row['Pearson p'] < 0.001 else ('**' if row['Pearson p'] < 0.01 else ('*' if row['Pearson p'] < 0.05 else ''))
    sig_s = '***' if row['Spearman p'] < 0.001 else ('**' if row['Spearman p'] < 0.01 else ('*' if row['Spearman p'] < 0.05 else ''))
    print(f"{row['Feature']:<35} {row['Pearson r']:>+9.4f}{sig_p:<3} {row['Pearson p']:>8.1e}  {row['Spearman rho']:>+9.4f}{sig_s:<3} {row['Spearman p']:>8.1e}")
print("="*95)
print("Significance: *** p<0.001, ** p<0.01, * p<0.05")

# Heatmap: focused on new + Chebyshev features
heatmap_features = [
    'MOLD_LEVEL_std [mm]',
    # Chebyshev coefficients
    'cheb_X1_bff_mean', 'cheb_X2_bff_mean', 'cheb_X3_bff_mean', 'cheb_X4_bff_mean',
    'cheb_X1_bff_std', 'cheb_X2_bff_std',
    'abs_cheb_X1_bff_mean', 'abs_cheb_X2_bff_mean',
    'cheb_X1_FF_LF_diff_mean', 'cheb_X2_FF_LF_diff_mean',
    # ML asymmetry
    'ML_LR_asym_std', 'ML_LR_abs_asym_mean',
    # Meniscus raw
    'meniscus_FF_LF_asym_std', 'meniscus_bff_range_mean',
    # EMBR
    'EMBR_DC_Master_LR_mean', 'EMBR_AC_Master_LR_mean',
    # Thermal + process
    'superHeat_avg', 'ArFlow_SEN_avg',
    'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]',
]

corr_matrix = df_seq_ext[heatmap_features].corr(method='spearman')

fig, ax = plt.subplots(figsize=(18, 15))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

short_labels = [
    'ML_std', 
    'X1_bff', 'X2_bff', 'X3_bff', 'X4_bff',
    'X1_bff_std', 'X2_bff_std',
    '|X1|_bff', '|X2|_bff',
    'X1_FF-LF', 'X2_FF-LF',
    'ML_LR_std', 'ML_|LR|',
    'FF-LF_asym_std', 'bff_range',
    'EMBR_DC_LR', 'EMBR_AC_LR',
    'superHeat', 'ArFlow_SEN',
    'Vc', 'Width',
]

ax.set_xticks(range(len(heatmap_features)))
ax.set_yticks(range(len(heatmap_features)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

for i in range(len(heatmap_features)):
    for j in range(len(heatmap_features)):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='Spearman rho', shrink=0.8)
ax.set_title('Spearman Correlation: Chebyshev Flatness + New Features vs Mold Level Stability',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

target = 'MOLD_LEVEL_std [mm]'

# Top features organized by physics domain
plot_features = [
    # Row 1: Chebyshev variability (top predictors)
    ('cheb_X2_bff_std', 'X2 Standing Wave Std (BFF)', 2, '#9467bd'),
    ('cheb_X1_bff_std', 'X1 Tilt Std (BFF)', 2, '#9467bd'),
    ('cheb_X2_bfl_std', 'X2 Standing Wave Std (BFL)', 2, '#9467bd'),
    # Row 2: Chebyshev mean levels
    ('abs_cheb_X2_bff_mean', '|X2| Standing Wave Mag (BFF)', 1, '#8c564b'),
    ('abs_cheb_X1_bff_mean', '|X1| Tilt Magnitude (BFF)', 1, '#8c564b'),
    ('cheb_X2_bff_mean', 'X2 Standing Wave Mean (BFF)', 1, '#8c564b'),
    # Row 3: ML asymmetry + meniscus + process
    ('ML_LR_asym_std', 'ML L-R Asymmetry Std [mm]', 1, '#1f77b4'),
    ('meniscus_FF_LF_asym_std', 'Meniscus FF-LF Asym Std', 4, '#2ca02c'),
    ('ArFlow_SEN_avg', 'Argon Flow SEN [L/min]', 1, '#ff7f0e'),
]

rows, cols = 3, 3
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f[1] for f in plot_features],
    vertical_spacing=0.08, horizontal_spacing=0.06
)

for idx, (col_name, label, rnd, color) in enumerate(plot_features):
    r = idx // cols + 1
    c = idx % cols + 1
    
    valid = df_seq_ext[[col_name, target, 'Seq_Name']].dropna()
    
    fig.add_trace(
        go.Scatter(
            x=valid[col_name], y=valid[target],
            mode='markers',
            marker=dict(size=4, color=color, opacity=0.5),
            customdata=valid[['Seq_Name']].to_numpy(),
            hovertemplate=f'{label}: %{{x:.{rnd}f}}<br>ML \u03c3: %{{y:.3f}} mm<br>%{{customdata[0]}}<extra></extra>',
            showlegend=False
        ), row=r, col=c
    )
    
    # Trend line
    if len(valid) > 2:
        z = np.polyfit(valid[col_name], valid[target], 1)
        x_line = np.linspace(valid[col_name].min(), valid[col_name].max(), 100)
        fig.add_trace(
            go.Scatter(
                x=x_line, y=np.polyval(z, x_line),
                mode='lines', line=dict(color='red', width=2, dash='dash'),
                showlegend=False
            ), row=r, col=c
        )
    
    fig.update_xaxes(title_text=label, row=r, col=c, title_font_size=9)
    fig.update_yaxes(title_text='ML \u03c3 [mm]' if c == 1 else '', row=r, col=c)

fig.update_layout(
    height=950, width=1200,
    title_text='Chebyshev Flatness Coefficients & New Features vs Mold Level Stability (\u03c3)',
    title_font_size=15, title_x=0.5
)
fig.show()

print("Row 1 (purple): Chebyshev variability — top predictors")
print("Row 2 (brown): Chebyshev mean magnitudes")
print("Row 3 (mixed): ML asymmetry, meniscus temp, argon flow")

## Accounting for EMBR Setpoint Confounding

The raw correlation analysis above shows EMBR DC Master L with ρ = −0.37 against ML σ — suggesting higher EMBR current → better stability. **However**, EMBR currents are **controlled variables**: their setpoints are determined by casting speed, mold width, and SEN immersion depth. Therefore:
- Correlations between EMBR currents and ML σ are **confounded** by the same process parameters
- E.g., wider mold → lower EMBR current AND higher ML σ → spurious negative correlation
- The L-R current imbalances are also partially driven by process conditions

**Approach**: Compute **partial correlations** controlling for (Vc, mold width, SEN) to isolate which features have independent predictive power — and which are merely reflecting process parameter effects.

In [0]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

# =====================================================================
# Partial correlation: correlation of X with Y after removing
# the linear effect of confounders Z = [Vc, Width, SEN]
# =====================================================================
def partial_corr(df, x_col, y_col, control_cols, method='spearman'):
    """Compute partial correlation between x and y controlling for Z."""
    from sklearn.linear_model import LinearRegression
    valid = df[[x_col, y_col] + control_cols].dropna()
    if len(valid) < 20:
        return np.nan, np.nan
    
    Z = valid[control_cols].values
    
    # Residualize x and y: remove linear effect of Z
    reg_x = LinearRegression().fit(Z, valid[x_col].values)
    res_x = valid[x_col].values - reg_x.predict(Z)
    
    reg_y = LinearRegression().fit(Z, valid[y_col].values)
    res_y = valid[y_col].values - reg_y.predict(Z)
    
    if method == 'spearman':
        r, p = stats.spearmanr(res_x, res_y)
    else:
        r, p = stats.pearsonr(res_x, res_y)
    return r, p

# =====================================================================
# Compute both raw and partial correlations
# =====================================================================
target = 'MOLD_LEVEL_std [mm]'
controls = ['CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]', 'SEN_avg [mm]']

exclude = ['Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'Seq_Condition',
           'Quality casting', target] + controls
feature_cols = [c for c in df_seq_ext.columns
                if c not in exclude and df_seq_ext[c].dtype in ['float64', 'int64']]

results = []
for col in feature_cols:
    valid = df_seq_ext[[col, target] + controls].dropna()
    if len(valid) < 20:
        continue
    
    # Raw Spearman
    r_raw, p_raw = stats.spearmanr(valid[col], valid[target])
    
    # Partial (controlling for Vc, Width, SEN)
    r_partial, p_partial = partial_corr(df_seq_ext, col, target, controls, method='spearman')
    
    results.append({
        'Feature': col,
        'Raw ρ': r_raw,
        'Raw p': p_raw,
        'Partial ρ': r_partial,
        'Partial p': p_partial,
        '|Raw|': abs(r_raw),
        '|Partial|': abs(r_partial),
        'Δ': abs(r_partial) - abs(r_raw),  # positive = gained strength
    })

df_partial = pd.DataFrame(results).sort_values('|Partial|', ascending=False)

# =====================================================================
# Print comparison: Raw vs Partial
# =====================================================================
print("="*105)
print(f"RAW vs PARTIAL CORRELATION with {target}")
print(f"Partial: controlling for Vc, Mold Width, SEN Immersion Depth")
print("="*105)
print(f"{'Feature':<35} {'Raw ρ':>8} {'Partial ρ':>10} {'Change':>8} {'Interpretation':>30}")
print("-"*105)

for _, row in df_partial.head(25).iterrows():
    delta = row['Δ']
    if delta > 0.05:
        interp = '↑ STRONGER (independent)'
    elif delta < -0.05:
        interp = '↓ WEAKER (confounded)'
    else:
        interp = '≈ Similar'
    
    sig_r = '***' if row['Raw p'] < 0.001 else ('**' if row['Raw p'] < 0.01 else ('*' if row['Raw p'] < 0.05 else ''))
    sig_p = '***' if row['Partial p'] < 0.001 else ('**' if row['Partial p'] < 0.01 else ('*' if row['Partial p'] < 0.05 else ''))
    
    print(f"{row['Feature']:<35} {row['Raw ρ']:>+.4f}{sig_r:<3} {row['Partial ρ']:>+.4f}{sig_p:<3}  {delta:>+.4f}  {interp:>28}")

print("="*105)

# =====================================================================
# Visual: Raw vs Partial barplot
# =====================================================================
top_n = 20
df_plot = df_partial.head(top_n).copy()

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(top_n)
bar_h = 0.35

ax.barh(y_pos + bar_h/2, df_plot['Raw ρ'], bar_h, label='Raw Spearman ρ', color='#aec7e8', edgecolor='#1f77b4')
ax.barh(y_pos - bar_h/2, df_plot['Partial ρ'], bar_h, label='Partial ρ (ctrl: Vc, Width, SEN)', color='#ff9896', edgecolor='#d62728')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot['Feature'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Spearman ρ vs Mold Level σ', fontsize=12)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(loc='lower right', fontsize=11)
ax.set_title('Raw vs Partial Correlation with ML σ\n'
             'Partial: after removing linear effect of Vc, Mold Width, SEN',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/raw_vs_partial_correlations.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nKey insight: Features whose partial ρ DROPS significantly are confounded by process params.")
print("Features whose partial ρ HOLDS or INCREASES have independent predictive power.")

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# =====================================================================
# Stratified analysis: EMBR vs ML σ WITHIN process operating regimes
# If EMBR truly helps, higher current should → lower σ within each bin
# =====================================================================
df_strat = df_seq_ext.copy()

# Bin by mold width and casting speed (2×2 = 4 operating regimes)
df_strat['Vc_bin'] = pd.qcut(df_strat['CASTING_SPEED_avg [m/min]'], q=2, labels=['Low Vc', 'High Vc'])
df_strat['Width_bin'] = pd.qcut(df_strat['MOLD_WIDTH_avg [m]'], q=2, labels=['Narrow', 'Wide'])
df_strat['Regime'] = df_strat['Width_bin'].astype(str) + ' / ' + df_strat['Vc_bin'].astype(str)

print("Sequences per operating regime:")
print(df_strat['Regime'].value_counts().sort_index())
print()

# =====================================================================
# Within-regime correlations for key features vs ML σ
# =====================================================================
target = 'MOLD_LEVEL_std [mm]'
key_features = [
    ('cheb_X2_bff_std', 'X₂ BFF Std (shape)'),
    ('meniscus_FF_LF_asym_std', 'FF-LF Asymmetry Std'),
    ('ArFlow_SEN_avg', 'Argon SEN Flow'),
    ('EMBR_DC_Master_L_avg', 'EMBR DC Master L'),
    ('EMBR_DC_Bottom_L_avg', 'EMBR DC Bottom L'),
    ('EMBR_DC_Master_LR_mean', 'EMBR DC Master L-R Diff'),
]

regimes = sorted(df_strat['Regime'].unique())

print("="*110)
print(f"{'Feature':<28}", end="")
for reg in regimes:
    print(f" {reg:>18}", end="")
print(f"  {'Overall':>10}")
print("-"*110)

for col, label in key_features:
    print(f"{label:<28}", end="")
    for reg in regimes:
        subset = df_strat[df_strat['Regime'] == reg][[col, target]].dropna()
        if len(subset) >= 10:
            r, p = stats.spearmanr(subset[col], subset[target])
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
            print(f"  {r:>+.3f}{sig:>4}", end="")
        else:
            print(f"  {'N/A':>8}", end="")
    # Overall
    valid = df_strat[[col, target]].dropna()
    r_all, _ = stats.spearmanr(valid[col], valid[target])
    print(f"  {r_all:>+.3f}")

print("="*110)

# =====================================================================
# Visual: Scatter EMBR DC Master vs ML σ, colored by regime
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors_regime = {'Narrow / Low Vc': '#1f77b4', 'Narrow / High Vc': '#ff7f0e',
                 'Wide / Low Vc': '#2ca02c', 'Wide / High Vc': '#d62728'}

for ax, (col, label) in zip(axes, [
    ('EMBR_DC_Master_L_avg', 'EMBR DC Master L [A]'),
    ('cheb_X2_bff_std', 'X₂ BFF Std (Standing Wave Var.)'),
    ('ArFlow_SEN_avg', 'Argon SEN Flow'),
]):
    for reg in regimes:
        sub = df_strat[df_strat['Regime'] == reg]
        ax.scatter(sub[col], sub[target], alpha=0.4, s=15,
                   color=colors_regime.get(reg, 'gray'), label=reg)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('ML σ [mm]', fontsize=11)
    ax.legend(fontsize=8, title='Regime', title_fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f'{label} vs ML σ\n(colored by operating regime)', fontsize=11, fontweight='bold')

plt.suptitle('Stratified Analysis: Feature vs ML σ within Process Operating Regimes',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/stratified_embr_analysis.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nInterpretation:")
print("  If within-regime ρ ≈ 0 for EMBR but overall ρ < 0 → Simpson's paradox / confounding")
print("  If within-regime ρ remains strong for Chebyshev → genuinely independent predictor")

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

target = 'MOLD_LEVEL_std [mm]'

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# =====================================================================
# 1. SEN_type boxplot
# =====================================================================
ax = axes[0]
sen_order = [-1.0, 1.0, 2.0]
sen_labels = ['SEN -1\n(n=495)', 'SEN 1\n(n=679)', 'SEN 2\n(n=472)']
sen_colors = ['#e74c3c', '#f39c12', '#27ae60']

box_data = [df_seq_ext[df_seq_ext['SEN_type'] == s][target].dropna().values for s in sen_order]
bp = ax.boxplot(box_data, labels=sen_labels, patch_artist=True, 
                widths=0.6, showfliers=False, medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], sen_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
# Strip points
for i, (data, s) in enumerate(zip(box_data, sen_order)):
    x = np.random.normal(i+1, 0.06, len(data))
    ax.scatter(x, data, alpha=0.15, s=8, color=sen_colors[i], zorder=3)

H, p = stats.kruskal(*box_data)
ax.set_title(f'SEN Nozzle Type\nKruskal-Wallis H={H:.1f}, p={p:.1e}', fontsize=12, fontweight='bold')
ax.set_ylabel('ML \u03c3 [mm]', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# =====================================================================
# 2. Quality family boxplot  
# =====================================================================
ax = axes[1]
qf_order = ['1', '2', '3', '5']
qf_counts = [df_seq_ext[df_seq_ext['quality_family'] == q].shape[0] for q in qf_order]
qf_labels = [f'Family {q}\n(n={c})' for q, c in zip(qf_order, qf_counts)]
qf_colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']

box_data_q = [df_seq_ext[df_seq_ext['quality_family'] == q][target].dropna().values for q in qf_order]
bp2 = ax.boxplot(box_data_q, labels=qf_labels, patch_artist=True,
                 widths=0.6, showfliers=False, medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp2['boxes'], qf_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
for i, data in enumerate(box_data_q):
    x = np.random.normal(i+1, 0.06, len(data))
    ax.scatter(x, data, alpha=0.15, s=8, color=qf_colors[i], zorder=3)

H_q, p_q = stats.kruskal(*box_data_q)
ax.set_title(f'Steel Grade Family\nKruskal-Wallis H={H_q:.1f}, p={p_q:.1e}', fontsize=12, fontweight='bold')
ax.set_ylabel('ML \u03c3 [mm]', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# =====================================================================
# 3. Casting length scatter
# =====================================================================
ax = axes[2]
sc = ax.scatter(df_seq_ext['castingLength_avg'], df_seq_ext[target],
                c=df_seq_ext['SEN_type'], cmap='RdYlGn_r', alpha=0.4, s=15, edgecolors='none')
ax.set_xlabel('Casting Length [m] (wear proxy)', fontsize=11)
ax.set_ylabel('ML \u03c3 [mm]', fontsize=11)
ax.set_title(f'Casting Length vs ML \u03c3\n(colored by SEN type)', fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.colorbar(sc, ax=ax, label='SEN_type', shrink=0.8)

plt.suptitle('Newly Included Process Context Variables vs Mold Level Stability',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/new_process_context_vars.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("\nKey findings:")
print(f"  SEN_type=2 is the most stable nozzle (median \u03c3=0.85 mm vs 1.27 mm for SEN_type=-1)")
print(f"  Steel grade family 1 casts most stably (median \u03c3=0.77 mm vs 1.23 mm for family 5)")
print(f"  castingLength shows no clear wear-driven degradation pattern")

## Summary: Causal Structure After Accounting for EMBR Confounding

### The Problem
EMBR currents are **setpoints controlled by** casting speed, mold width, and SEN depth — not independent variables.  
Naive correlations (e.g., EMBR DC Master ρ = –0.37 with ML σ) mix the *effect of process conditions* with the *effect of EMBR itself*.

### Implied Causal DAG
```
Process Parameters (Vc, Width, SEN)
      ├───────────→ EMBR Setpoints ──────────────→ Meniscus Shape (Chebyshev) ──→ ML σ
      ├───────────→ Argon Flow ───────────────────→ Meniscus Shape (Chebyshev) ──→ ML σ
      └─────────────────────────────────────────────────── (direct physical effect) ───→ ML σ

SEN Nozzle Type (design) ────────────────────────────────→ Meniscus Shape ─────────────→ ML σ
Steel Grade (chemistry) ─────────────────────────────────→ Flow Behavior ──────────────→ ML σ
```

### Revised Feature Ranking (Partial Correlation, controlling for Vc, Width, SEN)

| Rank | Feature | Partial ρ | Raw ρ | Change | Interpretation |
|------|---------|-----------|--------|--------|----------------|
| 1 | **FF-LF Asymmetry Std** | **+0.61** | +0.63 | ≈0 | Truly independent — stable across all regimes |
| 2 | **Cheb X₂ BFF Std** | **+0.61** | +0.68 | −0.07 | Slightly confounded, but still dominant |
| 3 | **Cheb X₂ BFL Std** | **+0.58** | +0.66 | −0.08 | Same: shape variability is fundamental |
| 4 | **Cheb X₁ BFF Std** | **+0.56** | +0.64 | −0.07 | Tilt variability independent of process |
| 5 | **Argon SEN Flow** | **+0.51** | +0.53 | ≈0 | Independent effect: more argon → more instability |
| 6 | **SEN\_type** (categorical) | **−0.25** | −0.24 | ≈0 | **Independent of process params! SEN\_type=2 best** |
| 7 | EMBR DC Master L | −0.15 | −0.37 | **−0.22** | Mostly confounded by mold width |
| 8 | EMBR DC Bottom L | −0.05 | −0.31 | **−0.26** | Almost entirely confounded |

### Newly Included Categorical Variables

| Variable | Test | H / p-value | Key Finding |
|----------|------|-------------|-------------|
| **SEN\_type** | Kruskal-Wallis | H=106, p=8.7e-24 | **Type 2 most stable** (median σ=0.85 mm) vs Type -1 worst (1.27 mm) |
| **quality\_family** | Kruskal-Wallis | H=124, p=9.0e-27 | **Family 1 most stable** (median σ=0.77 mm) vs Family 5 worst (1.23 mm) |
| castingLength | Spearman | ρ=−0.016, n.s. | No wear-driven degradation pattern |
| castMode | — | constant=1.0 | No variance in FC Mold ON data |
| moldThickness\* | — | constant | No variance (single mold geometry) |
| moldWidth\_b/e | — | r=±1.0 with moldWidth | Perfectly redundant |

### Stratified Analysis: Simpson's Paradox for EMBR

| Feature | Narrow/High Vc | Narrow/Low Vc | Wide/High Vc | Wide/Low Vc |
|---------|---------------|--------------|-------------|------------|
| Cheb X₂ Std | **+0.58** | **+0.46** | **+0.76** | **+0.65** |
| EMBR DC Master | −0.43 | **−0.02** | **+0.16** | −0.29 |
| EMBR DC Bottom | −0.12 | **+0.23** | **+0.12** | −0.19 |

### Decision: Focus Downstream Analysis on Meniscus Shape

Given that **Chebyshev coefficients are the strongest, process-independent predictors**, the following sections deep-dive into the physics of meniscus shape: profile reconstruction, temporal dynamics, and causal analysis. **EMBR absolute current levels are omitted from further downstream analysis** as their correlations are primarily artifacts of process parameter confounding.

**New actionable insight**: SEN nozzle type and steel grade family are significant, independent moderators of ML stability — these should be considered as **stratification variables** in any operational optimization or model development.

## Deep Dive: Meniscus Shape as Leading Indicator of ML Instability

Having established that Chebyshev flatness coefficients (X₁–X₄) are the most robust predictors of mold level instability — independent of operating regime — we now investigate the physical mechanism:
1. **Profile reconstruction** — What does the meniscus shape actually look like across the mold width?
2. **Stable vs unstable comparison** — How do profiles differ between good and bad sequences?
3. **Temporal dynamics** — Does meniscus shape instability *precede* mold level excursions?
4. **Granger causality** — Formal statistical test for predictive causality

In [0]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================================
# Chebyshev polynomials T0..T4
# =====================================================================
def cheb_T0(z): return np.ones_like(z)
def cheb_T1(z): return z
def cheb_T2(z): return 2*z**2 - 1
def cheb_T3(z): return 4*z**3 - 3*z
def cheb_T4(z): return 8*z**4 - 8*z**2 + 1

def reconstruct_meniscus(X1, X2, X3, X4, n_points=100):
    """
    Reconstruct meniscus height profile f(x) from Chebyshev coefficients.
    f(x) = X1*T1(x) + X2*T2(x) + X3*T3(x) + X4*T4(x)
    x is normalized mold width position: [-1, +1]
    X0 (mean offset) is not available → profile is relative to mean.
    """
    x = np.linspace(-1, 1, n_points)
    fx = X1*cheb_T1(x) + X2*cheb_T2(x) + X3*cheb_T3(x) + X4*cheb_T4(x)
    return x, fx

# =====================================================================
# 1. Per-sequence: compute f(x) using average Chebyshev coefs
# =====================================================================
df = df_seq_ext.copy()
n_pts = 200
x_norm = np.linspace(-1, 1, n_pts)

# Store reconstructed profiles as arrays
bff_profiles = np.zeros((len(df), n_pts))
bfl_profiles = np.zeros((len(df), n_pts))

for i in range(len(df)):
    row = df.iloc[i]
    _, bff_fx = reconstruct_meniscus(
        row['cheb_X1_bff_mean'], row['cheb_X2_bff_mean'],
        row['cheb_X3_bff_mean'], row['cheb_X4_bff_mean'], n_pts
    )
    _, bfl_fx = reconstruct_meniscus(
        row['cheb_X1_bfl_mean'], row['cheb_X2_bfl_mean'],
        row['cheb_X3_bfl_mean'], row['cheb_X4_bfl_mean'], n_pts
    )
    bff_profiles[i] = bff_fx
    bfl_profiles[i] = bfl_fx

# Compute peak-to-peak amplitude of f(x) as a scalar metric
df['bff_profile_ptp'] = np.ptp(bff_profiles, axis=1)
df['bfl_profile_ptp'] = np.ptp(bfl_profiles, axis=1)
df['bff_profile_max'] = np.max(np.abs(bff_profiles), axis=1)

print(f"BFF profile peak-to-peak: mean={df['bff_profile_ptp'].mean():.2f}, "
      f"std={df['bff_profile_ptp'].std():.2f}, "
      f"range=[{df['bff_profile_ptp'].min():.2f}, {df['bff_profile_ptp'].max():.2f}]")

# =====================================================================
# 2. Classify sequences into stability bins
# =====================================================================
ml_std = df['MOLD_LEVEL_std [mm]']
q_low, q_high = ml_std.quantile(0.2), ml_std.quantile(0.8)

df['stability_group'] = pd.cut(
    ml_std,
    bins=[-np.inf, q_low, q_high, np.inf],
    labels=['Stable (bottom 20%)', 'Medium', 'Unstable (top 20%)']
)

print(f"\nStability thresholds: stable < {q_low:.3f} mm, unstable > {q_high:.3f} mm")
print(df['stability_group'].value_counts())

# =====================================================================
# 3. Plot average meniscus profiles by stability group
# =====================================================================
groups = ['Stable (bottom 20%)', 'Medium', 'Unstable (top 20%)']
colors = {'Stable (bottom 20%)': '#2ca02c', 'Medium': '#ff7f0e', 'Unstable (top 20%)': '#d62728'}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Fixed Face (BFF)', 'Loose Face (BFL)'],
    horizontal_spacing=0.08
)

for grp in groups:
    mask = df['stability_group'] == grp
    n = mask.sum()
    
    # BFF
    mean_bff = bff_profiles[mask].mean(axis=0)
    std_bff = bff_profiles[mask].std(axis=0)
    fig.add_trace(go.Scatter(
        x=x_norm, y=mean_bff, mode='lines',
        name=f'{grp} (n={n})',
        line=dict(color=colors[grp], width=3),
        legendgroup=grp,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=np.concatenate([x_norm, x_norm[::-1]]),
        y=np.concatenate([mean_bff + std_bff, (mean_bff - std_bff)[::-1]]),
        fill='toself', fillcolor=colors[grp], opacity=0.15,
        line=dict(width=0), showlegend=False, legendgroup=grp,
    ), row=1, col=1)
    
    # BFL
    mean_bfl = bfl_profiles[mask].mean(axis=0)
    std_bfl = bfl_profiles[mask].std(axis=0)
    fig.add_trace(go.Scatter(
        x=x_norm, y=mean_bfl, mode='lines',
        name=f'{grp} (n={n})',
        line=dict(color=colors[grp], width=3),
        legendgroup=grp, showlegend=False,
    ), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=np.concatenate([x_norm, x_norm[::-1]]),
        y=np.concatenate([mean_bfl + std_bfl, (mean_bfl - std_bfl)[::-1]]),
        fill='toself', fillcolor=colors[grp], opacity=0.15,
        line=dict(width=0), showlegend=False, legendgroup=grp,
    ), row=1, col=2)

fig.update_xaxes(title_text='Normalized mold width position x [-1, +1]', row=1, col=1)
fig.update_xaxes(title_text='Normalized mold width position x [-1, +1]', row=1, col=2)
fig.update_yaxes(title_text='Meniscus height deviation (a.u.)', row=1, col=1)
fig.update_yaxes(title_text='Meniscus height deviation (a.u.)', row=1, col=2)

fig.update_layout(
    height=500, width=1200,
    title_text='Reconstructed Meniscus Profile f(x) = X\u2081T\u2081 + X\u2082T\u2082 + X\u2083T\u2083 + X\u2084T\u2084<br>'
               '<sub>Averaged by mold level stability group (\u00b11\u03c3 bands shown)</sub>',
    title_font_size=14, title_x=0.5,
    legend=dict(x=0.01, y=0.99),
)
fig.show()

print("\nNote: X\u2080 (mean offset) not available \u2014 profiles show shape deviation from mean level.")

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =====================================================================
# Presentation-ready figure: all individual profiles + bold average
# For one stable and one unstable sequence (BFF)
# =====================================================================
df_s = df_seq_ext.sort_values('MOLD_LEVEL_std [mm]')
stable_seq_idx = df_s.head(30).sample(1, random_state=42).index[0]
unstable_seq_idx = df_s.tail(30).sample(1, random_state=42).index[0]

n_pts = 100
x_norm = np.linspace(-1, 1, n_pts)

def get_profiles(seq_idx):
    """Extract all per-timestamp profiles for a given sequence."""
    grp = normal_groups[seq_idx]
    seq_data = df_fc_on.iloc[grp]
    profiles = []
    for _, r in seq_data.iterrows():
        X1, X2, X3, X4 = r['cheb_X1_bff'], r['cheb_X2_bff'], r['cheb_X3_bff'], r['cheb_X4_bff']
        if np.isnan(X1) or np.isnan(X2):
            continue
        fx = X1*cheb_T1(x_norm) + X2*cheb_T2(x_norm) + X3*cheb_T3(x_norm) + X4*cheb_T4(x_norm)
        profiles.append(fx)
    return np.array(profiles)

prof_stable = get_profiles(stable_seq_idx)
prof_unstable = get_profiles(unstable_seq_idx)

sigma_s = df_seq_ext.iloc[stable_seq_idx]['MOLD_LEVEL_std [mm]']
sigma_u = df_seq_ext.iloc[unstable_seq_idx]['MOLD_LEVEL_std [mm]']
name_s = df_seq_ext.iloc[stable_seq_idx]['Seq_Name']
name_u = df_seq_ext.iloc[unstable_seq_idx]['Seq_Name']

# =====================================================================
# Matplotlib figure — PPT-friendly (high DPI, clean fonts)
# =====================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, profiles, sigma, name, color_line, color_avg, title_label in [
    (axes[0], prof_stable, sigma_s, name_s, '#a8d5e2', '#0b5394', 'Stable'),
    (axes[1], prof_unstable, sigma_u, name_u, '#f4cccc', '#cc0000', 'Unstable'),
]:
    # Plot all individual profiles
    for i in range(len(profiles)):
        ax.plot(x_norm, profiles[i], color=color_line, alpha=0.15, linewidth=0.5)
    
    # Bold average
    avg = profiles.mean(axis=0)
    ax.plot(x_norm, avg, color=color_avg, linewidth=3.5, label='Average profile', zorder=10)
    
    # ±1 std band
    std = profiles.std(axis=0)
    ax.fill_between(x_norm, avg - std, avg + std, color=color_avg, alpha=0.12, zorder=5)
    
    # Zero reference line
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
    
    # Formatting
    ax.set_title(f'{title_label}: {name} (ML \u03c3 = {sigma:.2f} mm)',
                 fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Normalized mold width  x', fontsize=12)
    ax.set_ylabel('Meniscus height deviation  (a.u.)', fontsize=12)
    ax.tick_params(labelsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Legend
    legend_elements = [
        Line2D([0], [0], color=color_line, alpha=0.5, lw=1.5, label=f'Individual profiles (n={len(profiles)})'),
        Line2D([0], [0], color=color_avg, lw=3.5, label='Average profile'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10, framealpha=0.9)

plt.suptitle('Reconstructed Meniscus Profile  f(x) = X\u2081T\u2081 + X\u2082T\u2082 + X\u2083T\u2083 + X\u2084T\u2084\n'
             'Fixed Face (BFF) — All timestamps within sequence',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

# Save high-resolution for PPT
fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/meniscus_profiles_stable_vs_unstable.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\nFigure saved to DBFS for download.")
print(f"Stable: {len(prof_stable)} profiles, ptp range: [{prof_stable.ptp(axis=1).min():.1f}, {prof_stable.ptp(axis=1).max():.1f}]")
print(f"Unstable: {len(prof_unstable)} profiles, ptp range: [{prof_unstable.ptp(axis=1).min():.1f}, {prof_unstable.ptp(axis=1).max():.1f}]")

In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import numpy as np

target = 'MOLD_LEVEL_std [mm]'

# =====================================================================
# 1. Peak-to-peak amplitude of f(x) vs Mold Level sigma
# =====================================================================
for face, col in [('BFF', 'bff_profile_ptp'), ('BFL', 'bfl_profile_ptp')]:
    r_s, p_s = stats.spearmanr(df[col].dropna(), df.loc[df[col].notna(), target])
    print(f"f(x) peak-to-peak ({face}) vs ML \u03c3: Spearman \u03c1 = {r_s:+.4f} (p = {p_s:.1e})")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'BFF: f(x) Peak-to-Peak vs ML \u03c3',
        'BFL: f(x) Peak-to-Peak vs ML \u03c3'
    ],
    horizontal_spacing=0.08
)

for i, (face, col, color) in enumerate([
    ('BFF', 'bff_profile_ptp', '#9467bd'),
    ('BFL', 'bfl_profile_ptp', '#8c564b')
]):
    valid = df[[col, target, 'Seq_Name']].dropna()
    r_s, _ = stats.spearmanr(valid[col], valid[target])
    
    fig.add_trace(go.Scatter(
        x=valid[col], y=valid[target],
        mode='markers',
        marker=dict(size=5, color=color, opacity=0.5),
        customdata=valid[['Seq_Name']].to_numpy(),
        hovertemplate=f'{face} ptp: %{{x:.2f}}<br>ML \u03c3: %{{y:.3f}}<br>%{{customdata[0]}}<extra></extra>',
        name=f'{face} (\u03c1={r_s:.3f})',
    ), row=1, col=i+1)
    
    z = np.polyfit(valid[col], valid[target], 1)
    x_line = np.linspace(valid[col].min(), valid[col].max(), 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line),
        mode='lines', line=dict(color='red', width=2, dash='dash'),
        showlegend=False
    ), row=1, col=i+1)
    
    fig.update_xaxes(title_text=f'{face} Profile Peak-to-Peak', row=1, col=i+1)
    fig.update_yaxes(title_text='ML \u03c3 [mm]' if i == 0 else '', row=1, col=i+1)

fig.update_layout(height=450, width=1100,
    title_text='Meniscus Profile Amplitude (from Chebyshev reconstruction) vs Mold Level \u03c3',
    title_font_size=14, title_x=0.5)
fig.show()

# =====================================================================
# 2. Individual sequence profiles: 5 most stable vs 5 most unstable
# =====================================================================
df_sorted = df.sort_values(target)
n_ex = 5
stable_idx = df_sorted.head(n_ex).index
unstable_idx = df_sorted.tail(n_ex).index

fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Fixed Face (BFF)', 'Loose Face (BFL)'],
    horizontal_spacing=0.08
)

# Stable sequences
for i in stable_idx:
    sigma = df.loc[i, target]
    seq = df.loc[i, 'Seq_Name']
    fig2.add_trace(go.Scatter(
        x=x_norm, y=bff_profiles[i],
        mode='lines', line=dict(color='green', width=1.5, dash='solid'),
        name=f'{seq} (\u03c3={sigma:.2f})', legendgroup='stable',
    ), row=1, col=1)
    fig2.add_trace(go.Scatter(
        x=x_norm, y=bfl_profiles[i],
        mode='lines', line=dict(color='green', width=1.5, dash='solid'),
        showlegend=False, legendgroup='stable',
    ), row=1, col=2)

# Unstable sequences
for i in unstable_idx:
    sigma = df.loc[i, target]
    seq = df.loc[i, 'Seq_Name']
    fig2.add_trace(go.Scatter(
        x=x_norm, y=bff_profiles[i],
        mode='lines', line=dict(color='red', width=1.5, dash='solid'),
        name=f'{seq} (\u03c3={sigma:.2f})', legendgroup='unstable',
    ), row=1, col=1)
    fig2.add_trace(go.Scatter(
        x=x_norm, y=bfl_profiles[i],
        mode='lines', line=dict(color='red', width=1.5, dash='solid'),
        showlegend=False, legendgroup='unstable',
    ), row=1, col=2)

fig2.update_xaxes(title_text='Normalized mold width x', row=1, col=1)
fig2.update_xaxes(title_text='Normalized mold width x', row=1, col=2)
fig2.update_yaxes(title_text='Meniscus height deviation (a.u.)', row=1, col=1)

fig2.update_layout(
    height=500, width=1200,
    title_text=f'Individual Meniscus Profiles: {n_ex} Most Stable (green) vs {n_ex} Most Unstable (red)',
    title_font_size=14, title_x=0.5,
    legend=dict(font_size=9),
)
fig2.show()

print(f"\nStable sequences (green): ML \u03c3 = {df.loc[stable_idx, target].min():.3f} \u2013 {df.loc[stable_idx, target].max():.3f} mm")
print(f"Unstable sequences (red): ML \u03c3 = {df.loc[unstable_idx, target].min():.3f} \u2013 {df.loc[unstable_idx, target].max():.3f} mm")

In [0]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =====================================================================
# Select one stable and one unstable sequence
# =====================================================================
df_s = df_seq_ext.sort_values('MOLD_LEVEL_std [mm]')
stable_seq_idx = df_s.head(30).sample(1, random_state=42).index[0]
unstable_seq_idx = df_s.tail(30).sample(1, random_state=42).index[0]

seq_pairs = [
    (stable_seq_idx, 'Stable'),
    (unstable_seq_idx, 'Unstable'),
]

for seq_idx, label in seq_pairs:
    row = df_seq_ext.iloc[seq_idx]
    grp = normal_groups[seq_idx]
    seq_data = df_fc_on.iloc[grp].copy()
    
    sigma = row['MOLD_LEVEL_std [mm]']
    seq_name = row['Seq_Name']
    
    # Reconstruct f(x) at each timestamp
    n_pts = 100
    x_norm = np.linspace(-1, 1, n_pts)
    
    profiles_bff = []
    timestamps = []
    ml_values = []
    
    for _, r in seq_data.iterrows():
        X1, X2, X3, X4 = r['cheb_X1_bff'], r['cheb_X2_bff'], r['cheb_X3_bff'], r['cheb_X4_bff']
        if np.isnan(X1) or np.isnan(X2):
            continue
        fx = X1*cheb_T1(x_norm) + X2*cheb_T2(x_norm) + X3*cheb_T3(x_norm) + X4*cheb_T4(x_norm)
        profiles_bff.append(fx)
        timestamps.append(r['plainTimeStamp'])
        ml_values.append(r['Mold Level'])
    
    profiles_bff = np.array(profiles_bff)
    
    # Compute global y-range for consistent axis
    y_min = profiles_bff.min() - 2
    y_max = profiles_bff.max() + 2
    
    # Subsample: every 5th frame
    step = 5
    frame_indices = list(range(0, len(profiles_bff), step))
    
    # Build animated figure
    fig = go.Figure()
    
    # Initial frame
    fig.add_trace(go.Scatter(
        x=x_norm, y=profiles_bff[0],
        mode='lines', line=dict(color='#1f77b4', width=3),
        name='BFF profile'
    ))
    # Reference: sequence-average profile
    avg_profile = profiles_bff.mean(axis=0)
    fig.add_trace(go.Scatter(
        x=x_norm, y=avg_profile,
        mode='lines', line=dict(color='gray', width=1, dash='dash'),
        name='Sequence avg'
    ))
    
    # Animation frames
    frames = []
    for fi in frame_indices:
        t_str = str(timestamps[fi])[11:19]  # HH:MM:SS
        ml_val = ml_values[fi]
        frames.append(go.Frame(
            data=[
                go.Scatter(x=x_norm, y=profiles_bff[fi],
                           mode='lines', line=dict(color='#1f77b4', width=3)),
                go.Scatter(x=x_norm, y=avg_profile,
                           mode='lines', line=dict(color='gray', width=1, dash='dash')),
            ],
            name=t_str,
            layout=go.Layout(
                title_text=f'{label} Sequence ({seq_name}, ML \u03c3={sigma:.2f} mm)<br>'
                           f'<sub>t = {t_str} | Mold Level = {ml_val:.1f} mm</sub>'
            )
        ))
    
    fig.frames = frames
    
    # Slider + play button
    slider_steps = [
        dict(args=[[f.name], dict(mode='immediate', frame=dict(duration=100, redraw=True))],
             label=f.name, method='animate')
        for f in frames
    ]
    
    fig.update_layout(
        height=450, width=900,
        title_text=f'{label} Sequence ({seq_name}, ML \u03c3={sigma:.2f} mm)<br>'
                   f'<sub>BFF meniscus profile evolution (every {step}s)</sub>',
        title_font_size=14,
        xaxis_title='Normalized mold width x [-1, +1]',
        yaxis_title='Meniscus height deviation (a.u.)',
        yaxis_range=[y_min, y_max],
        updatemenus=[dict(
            type='buttons', showactive=False, x=0.05, y=1.15,
            buttons=[
                dict(label='\u25B6 Play', method='animate',
                     args=[None, dict(frame=dict(duration=150, redraw=True),
                                      fromcurrent=True, transition=dict(duration=50))]),
                dict(label='\u23F8 Pause', method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate', transition=dict(duration=0))])
            ]
        )],
        sliders=[dict(active=0, steps=slider_steps, x=0.05, len=0.9,
                      currentvalue=dict(prefix='Time: ', font_size=12))]
    )
    fig.show()

print(f"\nStable: {df_seq_ext.iloc[stable_seq_idx]['Seq_Name']} (ML \u03c3 = {df_seq_ext.iloc[stable_seq_idx]['MOLD_LEVEL_std [mm]']:.3f} mm)")
print(f"Unstable: {df_seq_ext.iloc[unstable_seq_idx]['Seq_Name']} (ML \u03c3 = {df_seq_ext.iloc[unstable_seq_idx]['MOLD_LEVEL_std [mm]']:.3f} mm)")
print("\n\u25B6 Press Play to animate. The dashed gray line is the sequence-average profile.")

In [0]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import signal

# =====================================================================
# 1. Prepare continuous time series for lag analysis
#    Use the full FC-ON dataset, segmented to avoid crossing time gaps
# =====================================================================
df_lag = df_fc_on[['plainTimeStamp', 'Mold Level', 
                   'cheb_X1_bff', 'cheb_X2_bff', 
                   'cheb_X1_bfl', 'cheb_X2_bfl']].copy()
df_lag = df_lag.dropna().sort_values('plainTimeStamp').reset_index(drop=True)

# Segment by time gaps > 5s
dt = df_lag['plainTimeStamp'].diff().dt.total_seconds().fillna(0)
df_lag['segment'] = (dt > 5).cumsum()

# Use the longest continuous segment for best cross-correlation
seg_sizes = df_lag.groupby('segment').size()
best_seg = seg_sizes.idxmax()
df_seg = df_lag[df_lag['segment'] == best_seg].reset_index(drop=True)
print(f"Longest continuous segment: {len(df_seg):,} samples ({len(df_seg)/60:.1f} min)")
print(f"Time: {df_seg['plainTimeStamp'].iloc[0]} → {df_seg['plainTimeStamp'].iloc[-1]}")

# =====================================================================
# 2. Compute rolling statistics (10s window)
# =====================================================================
win = 10  # seconds (= samples at 1 Hz)

df_seg['ML_abs_dev'] = (df_seg['Mold Level'] - df_seg['Mold Level'].rolling(win, center=True).mean()).abs()
df_seg['X2_bff_rolling_std'] = df_seg['cheb_X2_bff'].rolling(win, center=True).std()
df_seg['X1_bff_rolling_std'] = df_seg['cheb_X1_bff'].rolling(win, center=True).std()
df_seg['X2_bff_abs'] = df_seg['cheb_X2_bff'].abs()
df_seg['X1_bff_abs'] = df_seg['cheb_X1_bff'].abs()

df_clean = df_seg.dropna(subset=['ML_abs_dev', 'X2_bff_rolling_std', 'X1_bff_rolling_std'])
print(f"Clean samples for cross-correlation: {len(df_clean):,}")

# =====================================================================
# 3. Cross-correlation at different lags
#    Positive lag = Chebyshev leads Mold Level (shape change PRECEDES ML excursion)
#    Negative lag = Mold Level leads Chebyshev
# =====================================================================
max_lag = 30  # seconds

def compute_xcorr(x, y, max_lag):
    """Normalized cross-correlation for lag range [-max_lag, +max_lag]."""
    x_n = (x - x.mean()) / x.std()
    y_n = (y - y.mean()) / y.std()
    n = len(x_n)
    lags = range(-max_lag, max_lag + 1)
    xcorr = []
    for lag in lags:
        if lag >= 0:
            r = np.corrcoef(x_n[:n-lag], y_n[lag:])[0, 1]
        else:
            r = np.corrcoef(x_n[-lag:], y_n[:n+lag])[0, 1]
        xcorr.append(r)
    return np.array(list(lags)), np.array(xcorr)

# Cross-correlations for key Chebyshev features
features = [
    ('X2_bff_rolling_std', 'X\u2082 BFF Rolling Std', '#9467bd'),
    ('X1_bff_rolling_std', 'X\u2081 BFF Rolling Std', '#2ca02c'),
    ('X2_bff_abs', '|X\u2082| BFF Instantaneous', '#d62728'),
    ('X1_bff_abs', '|X\u2081| BFF Instantaneous', '#ff7f0e'),
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f[1] for f in features],
    vertical_spacing=0.12, horizontal_spacing=0.08
)

results = []
for idx, (col, label, color) in enumerate(features):
    r = idx // 2 + 1
    c = idx % 2 + 1
    
    lags, xcorr = compute_xcorr(df_clean[col].values, df_clean['ML_abs_dev'].values, max_lag)
    
    fig.add_trace(go.Bar(
        x=lags, y=xcorr, marker_color=[
            color if l >= 0 else '#aaaaaa' for l in lags
        ],
        showlegend=False,
        hovertemplate='Lag: %{x}s<br>r: %{y:.4f}<extra></extra>'
    ), row=r, col=c)
    
    # Mark peak
    peak_lag = lags[np.argmax(xcorr)]
    peak_val = xcorr.max()
    fig.add_annotation(
        x=peak_lag, y=peak_val,
        text=f'Peak: lag={peak_lag}s, r={peak_val:.3f}',
        showarrow=True, arrowhead=2, font_size=10,
        row=r, col=c
    )
    
    fig.update_xaxes(title_text='Lag (s)' if r == 2 else '', row=r, col=c)
    fig.update_yaxes(title_text='Cross-correlation' if c == 1 else '', row=r, col=c)
    
    results.append({'Feature': label, 'Peak Lag (s)': peak_lag, 'Peak r': peak_val})

fig.update_layout(
    height=650, width=1000,
    title_text='Lagged Cross-Correlation: Chebyshev Shape \u2192 Mold Level |Deviation|<br>'
               '<sub>Positive lag = shape change PRECEDES mold level excursion (colored bars)</sub>',
    title_font_size=14, title_x=0.5
)
fig.show()

print("\nCross-Correlation Peak Summary:")
print(f"{'Feature':<35} {'Peak Lag (s)':>12} {'Peak r':>10}")
print("-"*60)
for res in results:
    direction = '\u2192 Shape LEADS ML' if res['Peak Lag (s)'] > 0 else ('\u2192 ML LEADS shape' if res['Peak Lag (s)'] < 0 else '\u2192 Simultaneous')
    print(f"{res['Feature']:<35} {res['Peak Lag (s)']:>+10d}s  {res['Peak r']:>+8.4f}   {direction}")

In [0]:
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

# =====================================================================
# Granger Causality Test
# H0: X does NOT Granger-cause Y
# If p < 0.05, reject H0 → X has predictive information for Y
# =====================================================================
max_test_lag = 15  # test lags 1..15 seconds

# Subsample to speed up (use every row, but cap at 50k for Granger)
df_gc = df_clean[['ML_abs_dev', 'X2_bff_rolling_std', 'X1_bff_rolling_std',
                   'X2_bff_abs', 'X1_bff_abs']].iloc[:50000].copy()

print("="*90)
print("GRANGER CAUSALITY TESTS")
print("H0: X does NOT Granger-cause Y. Reject if p < 0.05.")
print("="*90)

test_pairs = [
    # (cause, effect, label)
    ('X2_bff_rolling_std', 'ML_abs_dev', 'X\u2082 rolling std \u2192 ML deviation'),
    ('X1_bff_rolling_std', 'ML_abs_dev', 'X\u2081 rolling std \u2192 ML deviation'),
    ('X2_bff_abs', 'ML_abs_dev', '|X\u2082| instantaneous \u2192 ML deviation'),
    ('X1_bff_abs', 'ML_abs_dev', '|X\u2081| instantaneous \u2192 ML deviation'),
    # Reverse direction
    ('ML_abs_dev', 'X2_bff_rolling_std', 'ML deviation \u2192 X\u2082 rolling std'),
    ('ML_abs_dev', 'X1_bff_rolling_std', 'ML deviation \u2192 X\u2081 rolling std'),
]

granger_results = []

for cause_col, effect_col, label in test_pairs:
    data = df_gc[[effect_col, cause_col]].dropna()
    
    if len(data) < max_test_lag * 3:
        print(f"\n{label}: Not enough data")
        continue
    
    try:
        gc = grangercausalitytests(data, maxlag=max_test_lag, verbose=False)
        
        # Find best (most significant) lag
        best_lag = None
        best_p = 1.0
        all_lags = []
        
        for lag in range(1, max_test_lag + 1):
            p_val = gc[lag][0]['ssr_ftest'][1]  # F-test p-value
            f_stat = gc[lag][0]['ssr_ftest'][0]
            all_lags.append({'lag': lag, 'F': f_stat, 'p': p_val})
            if p_val < best_p:
                best_p = p_val
                best_lag = lag
        
        sig = '***' if best_p < 0.001 else ('**' if best_p < 0.01 else ('*' if best_p < 0.05 else 'n.s.'))
        granger_results.append({
            'Test': label, 'Best Lag (s)': best_lag,
            'F-stat': gc[best_lag][0]['ssr_ftest'][0],
            'p-value': best_p, 'Significance': sig,
            'all_lags': all_lags
        })
    except Exception as e:
        print(f"\n{label}: Error - {e}")

# Print summary table
print(f"\n{'Test':<45} {'Best Lag':>8} {'F-stat':>10} {'p-value':>12} {'Sig':>5}")
print("-"*90)
for res in granger_results:
    print(f"{res['Test']:<45} {res['Best Lag (s)']:>6d}s  {res['F-stat']:>10.1f}  {res['p-value']:>11.2e}  {res['Significance']:>5}")
print("="*90)

# =====================================================================
# Visualize Granger F-stat across lags for shape → ML tests
# =====================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

shape_to_ml = [r for r in granger_results if '\u2192 ML' in r['Test']]
ml_to_shape = [r for r in granger_results if '\u2192 X' in r['Test']]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Shape \u2192 ML Deviation (predictive?)', 'ML Deviation \u2192 Shape (reactive?)'],
    horizontal_spacing=0.08
)

colors_fwd = ['#9467bd', '#2ca02c', '#d62728', '#ff7f0e']
for i, res in enumerate(shape_to_ml):
    lags = [d['lag'] for d in res['all_lags']]
    fstats = [d['F'] for d in res['all_lags']]
    label_short = res['Test'].split(' \u2192')[0]
    fig.add_trace(go.Scatter(
        x=lags, y=fstats, mode='lines+markers',
        name=label_short, line=dict(color=colors_fwd[i], width=2),
        marker=dict(size=5),
    ), row=1, col=1)

colors_rev = ['#9467bd', '#2ca02c']
for i, res in enumerate(ml_to_shape):
    lags = [d['lag'] for d in res['all_lags']]
    fstats = [d['F'] for d in res['all_lags']]
    label_short = res['Test'].split(' \u2192')[0]
    fig.add_trace(go.Scatter(
        x=lags, y=fstats, mode='lines+markers',
        name=label_short + ' (rev)', line=dict(color=colors_rev[i], width=2, dash='dot'),
        marker=dict(size=5),
    ), row=1, col=2)

for col in [1, 2]:
    fig.update_xaxes(title_text='Lag (seconds)', row=1, col=col)
    fig.update_yaxes(title_text='F-statistic', row=1, col=col)

fig.update_layout(
    height=450, width=1100,
    title_text='Granger Causality F-statistics Across Lags<br>'
               '<sub>Higher F = stronger predictive relationship at that lag</sub>',
    title_font_size=14, title_x=0.5,
    legend=dict(font_size=10)
)
fig.show()

print("\nInterpretation:")
print("  Shape \u2192 ML: If F-stat is high, meniscus shape changes PREDICT mold level excursions")
print("  ML \u2192 Shape: If F-stat is high, mold level changes PREDICT meniscus shape changes")
print("  If both directions are significant, there is bidirectional feedback")

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.patches import Patch
from pyspark.sql import functions as F

target = 'MOLD_LEVEL_std [mm]'

# =====================================================================
# 1. Get casting campaign boundaries from metadata
# =====================================================================
casting_campaigns = df_23_6_spark.select(
    'Datetime start first heat', 'Datetime start last heat', 'Quality casting'
).filter(
    F.col('Datetime start first heat').isNotNull()
).distinct().toPandas()

casting_campaigns = casting_campaigns.sort_values('Datetime start first heat').reset_index(drop=True)
casting_campaigns['heat_number'] = range(1, len(casting_campaigns) + 1)
casting_campaigns.rename(columns={
    'Datetime start first heat': 'cast_start',
    'Datetime start last heat': 'cast_end'
}, inplace=True)
casting_campaigns['cast_end_ext'] = casting_campaigns['cast_end'] + pd.Timedelta(hours=2)

print(f"Found {len(casting_campaigns)} casting campaigns (heats)")

# =====================================================================
# 2. Assign each sequence to a casting
# =====================================================================
def assign_casting(seq_start, campaigns):
    for _, camp in campaigns.iterrows():
        if camp['cast_start'] <= seq_start <= camp['cast_end_ext']:
            return camp['heat_number'], camp['Quality casting']
    return None, None

df_seq_ext['heat_number'] = None
df_seq_ext['steel_grade'] = None

for idx in df_seq_ext.index:
    t = df_seq_ext.loc[idx, 'Seq_time_Start']
    hn, sg = assign_casting(t, casting_campaigns)
    df_seq_ext.loc[idx, 'heat_number'] = hn
    df_seq_ext.loc[idx, 'steel_grade'] = sg

df_seq_ext['heat_number'] = pd.to_numeric(df_seq_ext['heat_number'])

matched = df_seq_ext['heat_number'].notna().sum()
print(f"Sequences matched to a casting: {matched}/{len(df_seq_ext)} ({100*matched/len(df_seq_ext):.1f}%)")

# =====================================================================
# 3. Per-casting aggregates
# =====================================================================
df_heat = df_seq_ext[df_seq_ext['heat_number'].notna()].copy()

cast_stats = df_heat.groupby(['heat_number', 'steel_grade']).agg(
    ML_sigma_median=(target, 'median'),
    ML_sigma_mean=(target, 'mean'),
    ML_sigma_q75=(target, lambda x: x.quantile(0.75)),
    ML_sigma_q25=(target, lambda x: x.quantile(0.25)),
    n_seq=(target, 'count'),
    cheb_X2_std_med=('cheb_X2_bff_std', 'median'),
    ArFlow_med=('ArFlow_SEN_avg', 'median'),
    Vc_med=('CASTING_SPEED_avg [m/min]', 'median'),
    Width_med=('MOLD_WIDTH_avg [m]', 'median'),
).reset_index()

# Color by steel grade family
grade_family_map = {g: (g[0] if isinstance(g, str) else '?') for g in cast_stats['steel_grade'].unique()}
cast_stats['family'] = cast_stats['steel_grade'].map(grade_family_map)
family_colors = {'1': '#2ecc71', '2': '#3498db', '3': '#e67e22', '5': '#9b59b6', '?': 'gray'}

# =====================================================================
# 4. Figure: 3-panel heat progression analysis
# =====================================================================
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1.5, 1.5]})

# --- Panel 1: ML σ per casting with IQR bands ---
ax = axes[0]
for _, row in cast_stats.iterrows():
    hn = row['heat_number']
    color = family_colors.get(row['family'], 'gray')
    ax.plot([hn, hn], [row['ML_sigma_q25'], row['ML_sigma_q75']],
            color=color, linewidth=6, alpha=0.3, solid_capstyle='round')
    ax.scatter(hn, row['ML_sigma_median'], color=color, s=100, zorder=5,
               edgecolors='black', linewidth=0.5)

# Overlay individual sequences as strip
for _, row in df_heat.iterrows():
    hn = row['heat_number']
    color = family_colors.get(grade_family_map.get(row['steel_grade'], '?'), 'gray')
    ax.scatter(hn + np.random.normal(0, 0.12), row[target],
               color=color, alpha=0.12, s=8, zorder=2)

# Trend line
x_trend = cast_stats['heat_number'].values
y_trend = cast_stats['ML_sigma_median'].values
slope, intercept, r, p, se = stats.linregress(x_trend, y_trend)
ax.plot(x_trend, intercept + slope * x_trend, 'k--', linewidth=1.5, alpha=0.6,
        label=f'Trend: slope={slope:+.03f} mm/heat, r={r:.2f}, p={p:.3f}')

ax.axhline(2.0, color='red', linewidth=1, linestyle=':', alpha=0.7, label='±2 mm target')
ax.set_ylabel('ML σ [mm]', fontsize=12)
ax.set_title('Mold Level Stability Across Casting Campaign (Heat Progression)\n'
             'Does quality degrade over successive heats?', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Grade labels
for _, row in cast_stats.iterrows():
    ax.text(row['heat_number'], ax.get_ylim()[1] * 0.95, row['steel_grade'],
            ha='center', va='top', fontsize=7, rotation=45, alpha=0.7)

legend_handles = [Patch(facecolor=c, edgecolor='black', linewidth=0.5, label=f'Family {f}')
                  for f, c in family_colors.items() if f != '?']
ax.legend(handles=legend_handles + ax.get_legend_handles_labels()[0][-2:],
          fontsize=8, loc='upper left', ncol=3)

# --- Panel 2: Process context per casting ---
ax2 = axes[1]
ax2b = ax2.twinx()
ax2.bar(cast_stats['heat_number'] - 0.15, cast_stats['Vc_med'], 0.3,
        color='#3498db', alpha=0.6, label='Casting Speed [m/min]')
ax2b.bar(cast_stats['heat_number'] + 0.15, cast_stats['Width_med'] * 1000, 0.3,
         color='#e74c3c', alpha=0.6, label='Mold Width [mm]')
ax2.set_ylabel('Vc [m/min]', fontsize=10, color='#3498db')
ax2b.set_ylabel('Width [mm]', fontsize=10, color='#e74c3c')
ax2.legend(loc='upper left', fontsize=8)
ax2b.legend(loc='upper right', fontsize=8)
ax2.spines['top'].set_visible(False)

# --- Panel 3: Chebyshev X2 std per casting ---
ax3 = axes[2]
for _, row in cast_stats.iterrows():
    color = family_colors.get(row['family'], 'gray')
    ax3.scatter(row['heat_number'], row['cheb_X2_std_med'], color=color, s=80,
                edgecolors='black', linewidth=0.5, zorder=5)

slope2, intercept2, r2, p2, _ = stats.linregress(x_trend, cast_stats['cheb_X2_std_med'].values)
ax3.plot(x_trend, intercept2 + slope2 * x_trend, 'k--', linewidth=1.5, alpha=0.6,
         label=f'Trend: slope={slope2:+.04f}/heat, r={r2:.2f}, p={p2:.3f}')
ax3.set_xlabel('Heat Number (chronological order)', fontsize=12)
ax3.set_ylabel('Cheb X\u2082 BFF Std\n(meniscus shape var.)', fontsize=10)
ax3.legend(fontsize=8)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/heat_progression_analysis.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Statistical summary
print(f"\n{'='*70}")
print(f"HEAT PROGRESSION SUMMARY ({len(casting_campaigns)} castings, April 13-30 2025)")
print(f"{'='*70}")
print(f"ML σ trend across heats:  slope = {slope:+.03f} mm/heat, r = {r:.3f}, p = {p:.3f}")
print(f"X\u2082 std trend across heats: slope = {slope2:+.04f}/heat, r = {r2:.3f}, p = {p2:.3f}")
if p < 0.05:
    direction = 'degrading' if slope > 0 else 'improving'
    print(f"\u2192 SIGNIFICANT trend: ML stability is {direction} over the campaign")
else:
    print(f"\u2192 No significant trend: ML stability does NOT systematically change across heats")
print(f"\nVariation between castings is dominated by steel grade & SEN type, not mold wear.")

## Regime Discovery: Unsupervised Clustering of Operating Conditions

Rather than manual 2×2 bins (Vc/Width), we use **HDBSCAN** to discover *natural* operating regimes from the full feature space, then characterise what distinguishes the “always-stable” cluster from the “always-unstable” one.

**Approach:**
1. Select physics-relevant features (exclude confounded EMBR absolute currents)
2. StandardScaler → HDBSCAN (density-based, no *k* required)
3. UMAP 2D projection for visualisation
4. Per-cluster profiling → actionable operating envelopes

In [0]:
%pip install hdbscan umap-learn --quiet

In [0]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import hdbscan
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

target = 'MOLD_LEVEL_std [mm]'

# =====================================================================
# 1. Feature selection for clustering
#    Exclude ALL EMBR features (absolute + L-R), metadata, target
# =====================================================================
exclude_cols = [
    'Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'Seq_Condition',
    'Quality casting', 'quality_family', 'steel_grade', 'heat_number',
    target, 'MOLD_LEVEL_avg [mm]',
    'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
    'has_high_variability', 'has_disturbance', 'disturbance_type',
    # All EMBR features (confounded setpoints + regime-dependent L-R)
    'EMBR_DC_Bottom_L_avg', 'EMBR_AC_Master_L_avg', 'EMBR_DC_Master_L_avg',
    'EMBR_DC_Bottom_LR_mean', 'EMBR_DC_Bottom_LR_std',
    'EMBR_AC_Master_LR_mean', 'EMBR_AC_Master_LR_std',
    'EMBR_DC_Master_LR_mean', 'EMBR_DC_Master_LR_std',
    # Constant / low-info
    'castMode',
    # Clustering artefacts from previous runs
    'cluster', 'cluster_prob', 'umap_1', 'umap_2',
]

feature_cols_cluster = [c for c in df_seq_ext.columns
                        if c not in exclude_cols
                        and df_seq_ext[c].dtype in ['float64', 'int64']]

print(f"Clustering features ({len(feature_cols_cluster)}):")
for i, c in enumerate(feature_cols_cluster):
    print(f"  {i+1:>2}. {c}")

# =====================================================================
# 2. Prepare matrix, handle NaNs
# =====================================================================
X_clust = df_seq_ext[feature_cols_cluster].copy()
nan_mask = X_clust.isna().any(axis=1)
print(f"\nRows with NaN: {nan_mask.sum()} / {len(X_clust)} \u2192 dropping")

X_clean = X_clust[~nan_mask].reset_index(drop=True)
df_cluster = df_seq_ext[~nan_mask].reset_index(drop=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
print(f"Scaled feature matrix: {X_scaled.shape}")

# =====================================================================
# 3. UMAP 2D projection
# =====================================================================
reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1,
                    random_state=42, metric='euclidean')
embedding = reducer.fit_transform(X_scaled)
df_cluster['umap_1'] = embedding[:, 0]
df_cluster['umap_2'] = embedding[:, 1]

# =====================================================================
# 4. HDBSCAN clustering (re-tuned for finer regime discovery)
# =====================================================================
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    cluster_selection_method='eom',
    prediction_data=True
)
df_cluster['cluster'] = clusterer.fit_predict(X_scaled)
df_cluster['cluster_prob'] = clusterer.probabilities_

n_clusters = df_cluster['cluster'].nunique() - (1 if -1 in df_cluster['cluster'].values else 0)
n_noise = (df_cluster['cluster'] == -1).sum()
print(f"\nHDBSCAN results: {n_clusters} clusters, {n_noise} noise points ({100*n_noise/len(df_cluster):.1f}%)")
print(df_cluster['cluster'].value_counts().sort_index())

# =====================================================================
# 5. Plotly visualisation: 3 panels
# =====================================================================
df_plot = df_cluster.copy()
df_plot['cluster_label'] = df_plot['cluster'].apply(
    lambda c: 'Noise' if c == -1 else f'C{c}'
)

# --- Panel 1: UMAP by cluster ---
fig1 = px.scatter(
    df_plot, x='umap_1', y='umap_2', color='cluster_label',
    hover_data=['Seq_Name', target, 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]'],
    title='HDBSCAN Clusters in UMAP Space (EMBR-free)',
    labels={'umap_1': 'UMAP-1', 'umap_2': 'UMAP-2', 'cluster_label': 'Cluster'},
    opacity=0.6, width=900, height=600,
)
fig1.update_traces(marker=dict(size=6))
fig1.update_layout(legend=dict(font=dict(size=9), itemsizing='constant'))
fig1.show()

# --- Panel 2: UMAP colored by ML \u03c3 ---
fig2 = px.scatter(
    df_plot, x='umap_1', y='umap_2', color=target,
    hover_data=['Seq_Name', 'cluster_label', 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]'],
    title='ML \u03c3 in UMAP Space',
    labels={'umap_1': 'UMAP-1', 'umap_2': 'UMAP-2', target: 'ML \u03c3 [mm]'},
    color_continuous_scale='Viridis',
    range_color=[0, df_plot[target].quantile(0.95)],
    opacity=0.7, width=900, height=600,
)
fig2.update_traces(marker=dict(size=6))
fig2.show()

# --- Panel 3: UMAP colored by top SHAP feature ---
top_feat = 'meniscus_FF_LF_asym_std'
if top_feat in df_plot.columns:
    fig3 = px.scatter(
        df_plot, x='umap_1', y='umap_2', color=top_feat,
        hover_data=['Seq_Name', 'cluster_label', target],
        title=f'Top SHAP Feature ({top_feat}) in UMAP Space',
        labels={'umap_1': 'UMAP-1', 'umap_2': 'UMAP-2'},
        color_continuous_scale='Inferno',
        range_color=[0, df_plot[top_feat].quantile(0.95)],
        opacity=0.7, width=900, height=600,
    )
    fig3.update_traces(marker=dict(size=6))
    fig3.show()

# =====================================================================
# 6. Per-cluster profiling: median of key process variables
# =====================================================================
profile_cols = [
    target, 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]', 'SEN_avg [mm]',
    'ArFlow_SEN_avg', 'castingLength_avg',
    'meniscus_FF_LF_asym_std', 'cheb_X2_bfl_std', 'cheb_X2_bff_std',
    'ML_LR_asym_std',
]
profile_cols = [c for c in profile_cols if c in df_cluster.columns]

cluster_profile = df_cluster.groupby('cluster')[profile_cols].median().round(3)
cluster_profile.insert(0, 'n_sequences', df_cluster.groupby('cluster').size())

print("\n" + "="*100)
print("PER-CLUSTER PROFILES (median values)")
print("="*100)
with pd.option_context('display.max_columns', 20, 'display.width', 120):
    print(cluster_profile.to_string())

# Rank clusters by ML stability
print("\nClusters ranked by ML \u03c3 (most stable first):")
stability_rank = cluster_profile[[target, 'n_sequences']].sort_values(target)
for cid, row in stability_rank.iterrows():
    label = 'Noise' if cid == -1 else f'Cluster {cid}'
    print(f"  {label:<12}  ML \u03c3 = {row[target]:.3f} mm  (n={int(row['n_sequences'])})")

## Predictive Model + SHAP Interaction Maps

Using **LightGBM** gradient boosting to predict mold level instability (ML σ) from the per-sequence feature set, then **SHAP TreeExplainer** for:
1. **Global feature importance** — which features matter most?
2. **SHAP dependence plots** — non-linear effects and interaction detection
3. **SHAP interaction heatmap** — pairwise feature interaction strengths

Dataset: `df_seq_ext` with 1,212 normal sequences, 47 numeric features, target = `MOLD_LEVEL_std [mm]`

In [0]:
%pip install shap --quiet

In [0]:
import numpy as np
import pandas as pd
import xgboost as xgb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

target = 'MOLD_LEVEL_std [mm]'

# =====================================================================
# 1. Prepare features & target
# =====================================================================
exclude = [
    'Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'Seq_Condition',
    'Quality casting', 'quality_family', 'steel_grade', 'heat_number',
    target, 'MOLD_LEVEL_avg [mm]',
    'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
    'has_high_variability', 'has_disturbance', 'disturbance_type',
    'cluster', 'cluster_prob', 'umap_1', 'umap_2',
    # --- Confounded EMBR absolute currents (controlled setpoints) ---
    'EMBR_DC_Bottom_L_avg', 'EMBR_AC_Master_L_avg', 'EMBR_DC_Master_L_avg',
    # --- All EMBR L-R features (also regime-dependent) ---
    'EMBR_DC_Bottom_LR_mean', 'EMBR_DC_Bottom_LR_std',
    'EMBR_AC_Master_LR_mean', 'EMBR_AC_Master_LR_std',
    'EMBR_DC_Master_LR_mean', 'EMBR_DC_Master_LR_std',
]

feature_cols = [c for c in df_seq_ext.columns
                if c not in exclude
                and df_seq_ext[c].dtype in ['float64', 'int64', 'int32', 'float32']]

print(f"Features ({len(feature_cols)}):")
for i, c in enumerate(feature_cols):
    print(f"  {i+1:>2}. {c}")

df_model = df_seq_ext[feature_cols + [target]].dropna().reset_index(drop=True)
X = df_model[feature_cols].values
y = df_model[target].values
print(f"\nModel dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Target stats \u2014 mean: {y.mean():.3f}, median: {np.median(y):.3f}, std: {y.std():.3f}")

# =====================================================================
# 2. XGBoost model with 5-fold cross-validation
# =====================================================================
xgb_params = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    tree_method='hist',
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(y))
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    preds = model.predict(X_val)
    oof_preds[val_idx] = preds
    r2 = r2_score(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    fold_scores.append({'fold': fold, 'R2': r2, 'RMSE': rmse})
    print(f"  Fold {fold}: R\u00b2 = {r2:.4f}, RMSE = {rmse:.4f} mm")

# Overall OOF performance
oof_r2 = r2_score(y, oof_preds)
oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
oof_mae = mean_absolute_error(y, oof_preds)
print(f"\n{'='*60}")
print(f"OOF Performance (5-fold):")
print(f"  R\u00b2   = {oof_r2:.4f}")
print(f"  RMSE = {oof_rmse:.4f} mm")
print(f"  MAE  = {oof_mae:.4f} mm")
print(f"{'='*60}")

# =====================================================================
# 3. Train final model on ALL data (for SHAP)
# =====================================================================
final_model = xgb.XGBRegressor(**xgb_params)
final_model.fit(X, y, verbose=False)

# =====================================================================
# 4. Plotly: Actual vs Predicted + Feature Importance
# =====================================================================
# --- Panel 1: Actual vs Predicted (OOF) ---
fig_pred = go.Figure()
fig_pred.add_trace(go.Scatter(
    x=y, y=oof_preds, mode='markers',
    marker=dict(size=5, color='#3498db', opacity=0.4),
    name='Predictions',
    hovertemplate='Actual: %{x:.3f} mm<br>Predicted: %{y:.3f} mm<extra></extra>',
))
lim_max = max(y.max(), oof_preds.max()) * 1.05
fig_pred.add_trace(go.Scatter(
    x=[0, lim_max], y=[0, lim_max], mode='lines',
    line=dict(color='black', dash='dash', width=1),
    name='Perfect prediction', showlegend=True,
))
fig_pred.update_layout(
    title=f'Actual vs Predicted (5-fold OOF)<br><sup>R\u00b2 = {oof_r2:.3f}, RMSE = {oof_rmse:.3f} mm</sup>',
    xaxis_title='Actual ML \u03c3 [mm]',
    yaxis_title='Predicted ML \u03c3 [mm] (OOF)',
    width=750, height=600,
    xaxis=dict(range=[0, lim_max]),
    yaxis=dict(range=[0, lim_max]),
)
fig_pred.show()

# --- Panel 2: XGBoost Feature Importance (top 20) ---
importances = final_model.feature_importances_
imp_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True).tail(20)

fig_imp = px.bar(
    imp_df, x='importance', y='feature', orientation='h',
    title='XGBoost Feature Importance (Top 20, Gain-based)',
    labels={'importance': 'Gain-based Importance', 'feature': ''},
    color='importance', color_continuous_scale='Greens',
    width=800, height=600,
)
fig_imp.update_layout(coloraxis_showscale=False)
fig_imp.show()

In [0]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# =====================================================================
# 1. Compute SHAP values with TreeExplainer
# =====================================================================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

# Create a DataFrame version for readability
X_df = pd.DataFrame(X, columns=feature_cols)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base prediction): {explainer.expected_value:.3f} mm")

# =====================================================================
# 2. SHAP Summary Plot (global importance + direction)
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 12))
shap.summary_plot(shap_values, X_df, max_display=25, show=False)
plt.title('SHAP Summary: Feature Impact on ML \u03c3 Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/shap_summary.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =====================================================================
# 3. SHAP Bar Plot (mean |SHAP|)
# =====================================================================
fig, ax = plt.subplots(figsize=(10, 10))
shap.summary_plot(shap_values, X_df, plot_type='bar', max_display=25, show=False)
plt.title('Mean |SHAP| — Global Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig('/dbfs/FileStore/TATAIjmulden_FCMoldG5/figures/shap_bar.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# =====================================================================
# 4. Top feature importance ranking
# =====================================================================
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_rank = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("\nTop 15 features by mean |SHAP|:")
print("="*55)
for i, (_, row) in enumerate(shap_rank.head(15).iterrows(), 1):
    print(f"  {i:>2}. {row['feature']:<40} {row['mean_abs_shap']:.4f}")

In [0]:
import shap
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# =====================================================================
# 1. SHAP Dependence Plots for top 6 features (matplotlib — shap API)
# =====================================================================
top_features = shap_rank.head(6)['feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for i, feat in enumerate(top_features):
    ax = axes[i // 3, i % 3]
    feat_idx = feature_cols.index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, X_df,
        interaction_index='auto',
        ax=ax, show=False,
        alpha=0.5, dot_size=15,
    )
    ax.set_title(f'{feat}', fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('SHAP Dependence Plots \u2014 Top 6 Features\n'
             '(colour = auto-detected strongest interaction partner)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# =====================================================================
# 2. SHAP Interaction Values (pairwise)
# =====================================================================
print("Computing SHAP interaction values (this may take a few minutes)...")
shap_interaction = explainer.shap_interaction_values(X)
print(f"SHAP interaction tensor shape: {shap_interaction.shape}")

# =====================================================================
# 3. SHAP Interaction Heatmap (Plotly)
# =====================================================================
n_features = len(feature_cols)
interaction_matrix = np.zeros((n_features, n_features))

for i in range(n_features):
    for j in range(n_features):
        interaction_matrix[i, j] = np.abs(shap_interaction[:, i, j]).mean()

# Zero out diagonal (main effects)
np.fill_diagonal(interaction_matrix, 0)

# Top 20 features by total interaction strength
total_interaction = interaction_matrix.sum(axis=1)
top_idx = np.argsort(total_interaction)[-20:][::-1]
top_names = [feature_cols[i] for i in top_idx]
interaction_sub = interaction_matrix[np.ix_(top_idx, top_idx)]

fig_heat = px.imshow(
    interaction_sub,
    x=top_names, y=top_names,
    color_continuous_scale='YlOrRd',
    labels=dict(color='Mean |SHAP Interaction|'),
    title='SHAP Pairwise Interaction Heatmap (Top 20 Features)<br>'
          '<sup>Off-diagonal = how much two features jointly influence ML \u03c3</sup>',
    width=900, height=800,
    aspect='equal',
)
fig_heat.update_xaxes(tickangle=45, tickfont=dict(size=9))
fig_heat.update_yaxes(tickfont=dict(size=9))
fig_heat.show()

# =====================================================================
# 4. Top interaction pairs
# =====================================================================
interaction_pairs = []
for i in range(n_features):
    for j in range(i+1, n_features):
        interaction_pairs.append({
            'Feature A': feature_cols[i],
            'Feature B': feature_cols[j],
            'Mean |Interaction|': interaction_matrix[i, j]
        })

df_interactions = pd.DataFrame(interaction_pairs).sort_values('Mean |Interaction|', ascending=False)
print("\nTop 15 Feature Interaction Pairs:")
print("="*75)
for i, (_, row) in enumerate(df_interactions.head(15).iterrows(), 1):
    print(f"  {i:>2}. {row['Feature A']:<30} \u00d7 {row['Feature B']:<30} = {row['Mean |Interaction|']:.4f}")

In [0]:
import sys, importlib

repo_root = "/Workspace/Repos/dieudonne.nkulikiyimfura@se.abb.com/fcMoldG5_data_analysis_TATA"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Force-reimport all package modules (picks up any edits)
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("fcmold_analysis"):
        del sys.modules[mod_name]

from fcmold_analysis import STRAND_CONFIGS, StrandAnalysisPipeline
from fcmold_analysis.config import CONFIG

# =====================================================================
# Run pipeline for STRAND 5
# =====================================================================
pipeline_5 = StrandAnalysisPipeline(STRAND_CONFIGS["23_5"], CONFIG)
results_5 = pipeline_5.run(generate_visualizations=False)

if results_5["success"]:
    df_raw_5 = results_5["df_raw"]
    df_fc_5 = results_5["df_fc_mold"]
    df_seq_5 = results_5["df_seq"]
    normal_groups_5 = results_5["normal_groups"]
    abnormal_groups_5 = results_5["abnormal_groups"]
    print(f"\n{'='*60}")
    print(f"STRAND 5 SUMMARY")
    print(f"{'='*60}")
    print(f"  Raw rows (steady-state):  {len(df_raw_5):,}")
    print(f"  FC Mold ON rows:          {len(df_fc_5):,}")
    print(f"  Normal sequences:         {len(df_seq_5)}")
    print(f"  Abnormal sequences:       {len(abnormal_groups_5)}")
    print(f"  ML σ median:              {df_seq_5['MOLD_LEVEL_std [mm]'].median():.3f} mm")
    print(f"  ML σ mean:                {df_seq_5['MOLD_LEVEL_std [mm]'].mean():.3f} mm")
    print(f"  Columns in df_seq:        {len(df_seq_5.columns)}")
else:
    print(f"ERROR: {results_5['error']}")
    print(results_5.get('traceback', ''))

In [0]:
# Re-run Strand 6 through the SAME pipeline so both have identical schemas
pipeline_6 = StrandAnalysisPipeline(STRAND_CONFIGS["23_6"], CONFIG)
results_6 = pipeline_6.run(generate_visualizations=False)

if results_6["success"]:
    df_raw_6 = results_6["df_raw"]
    df_fc_6 = results_6["df_fc_mold"]
    df_seq_6 = results_6["df_seq"]
    normal_groups_6 = results_6["normal_groups"]
    abnormal_groups_6 = results_6["abnormal_groups"]
    print(f"\n{'='*60}")
    print(f"STRAND 6 SUMMARY")
    print(f"{'='*60}")
    print(f"  Raw rows (steady-state):  {len(df_raw_6):,}")
    print(f"  FC Mold ON rows:          {len(df_fc_6):,}")
    print(f"  Normal sequences:         {len(df_seq_6)}")
    print(f"  Abnormal sequences:       {len(abnormal_groups_6)}")
    print(f"  ML σ median:              {df_seq_6['MOLD_LEVEL_std [mm]'].median():.3f} mm")
    print(f"  ML σ mean:                {df_seq_6['MOLD_LEVEL_std [mm]'].mean():.3f} mm")
    print(f"  Columns in df_seq:        {len(df_seq_6.columns)}")
    print(f"\n  Schema match with S5:     {set(df_seq_6.columns) == set(df_seq_5.columns)}")
else:
    print(f"ERROR: {results_6['error']}")
    print(results_6.get('traceback', ''))

In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

target = 'MOLD_LEVEL_std [mm]'

# Tag and combine
df_seq_5['strand'] = 'Strand 5'
df_seq_6['strand'] = 'Strand 6'
df_cmp = pd.concat([df_seq_5, df_seq_6], ignore_index=True)

# =====================================================================
# 1. Distribution overlays for key process parameters
# =====================================================================
panels = [
    (target, 'ML \u03c3 [mm]'),
    ('CASTING_SPEED_avg [m/min]', 'Casting Speed [m/min]'),
    ('MOLD_WIDTH_avg [m]', 'Mold Width [m]'),
    ('SEN_avg [mm]', 'SEN Depth [mm]'),
    ('ArFlow_SEN_avg', 'Argon Flow SEN [NL/min]'),
    ('ptp_mm', 'ML peak-to-peak [mm]'),
]

fig = make_subplots(rows=2, cols=3,
    subplot_titles=[p[1] for p in panels],
    vertical_spacing=0.14, horizontal_spacing=0.07)

colors = {'Strand 5': '#e74c3c', 'Strand 6': '#3498db'}

for i, (col, label) in enumerate(panels):
    row, c = i // 3 + 1, i % 3 + 1
    if col not in df_cmp.columns:
        continue
    for strand, color in colors.items():
        vals = df_cmp.loc[df_cmp['strand'] == strand, col].dropna()
        fig.add_trace(go.Histogram(
            x=vals, name=strand, marker_color=color,
            opacity=0.55, nbinsx=50, showlegend=(i == 0),
        ), row=row, col=c)
    fig.update_xaxes(title_text=label, row=row, col=c)

fig.update_layout(
    title='Process Parameter Distributions: Strand 5 vs Strand 6',
    height=650, width=1300, barmode='overlay',
    legend=dict(x=0.85, y=0.98),
)
fig.show()

# =====================================================================
# 2. Summary statistics table
# =====================================================================
print('\n' + '='*80)
print('COMPARATIVE SUMMARY')
print('='*80)
header = f"{'Metric':<35} {'Strand 5':>15} {'Strand 6':>15} {'\u0394 (5-6)':>12}"
print(header)
print('-'*80)

stats_pairs = [
    ('Raw rows (steady-state)', len(df_raw_5), len(df_raw_6)),
    ('FC Mold ON rows', len(df_fc_5), len(df_fc_6)),
    ('Normal sequences', len(df_seq_5), len(df_seq_6)),
    ('ML \u03c3 median [mm]', df_seq_5[target].median(), df_seq_6[target].median()),
    ('ML \u03c3 mean [mm]', df_seq_5[target].mean(), df_seq_6[target].mean()),
    ('% stable (\u03c3<2mm)', 100*(df_seq_5[target]<2).mean(), 100*(df_seq_6[target]<2).mean()),
    ('Vc avg [m/min]', df_seq_5['CASTING_SPEED_avg [m/min]'].mean(), df_seq_6['CASTING_SPEED_avg [m/min]'].mean()),
    ('Width avg [m]', df_seq_5['MOLD_WIDTH_avg [m]'].mean(), df_seq_6['MOLD_WIDTH_avg [m]'].mean()),
    ('SEN avg [mm]', df_seq_5['SEN_avg [mm]'].mean(), df_seq_6['SEN_avg [mm]'].mean()),
]

for label, v5, v6 in stats_pairs:
    if isinstance(v5, int):
        print(f"  {label:<33} {v5:>15,} {v6:>15,} {v5-v6:>+12,}")
    else:
        print(f"  {label:<33} {v5:>15.3f} {v6:>15.3f} {v5-v6:>+12.3f}")

In [0]:
# =====================================================================
# Compare engineered feature distributions
# =====================================================================
feat_panels = [
    ('meniscus_FF_LF_asym_std', 'Meniscus FF-LF Asym \u03c3'),
    ('cheb_X2_bfl_std', 'Cheb X\u2082 BFL \u03c3 (standing wave)'),
    ('cheb_X2_bff_std', 'Cheb X\u2082 BFF \u03c3'),
    ('cheb_X1_bfl_std', 'Cheb X\u2081 BFL \u03c3 (slope)'),
    ('ML_LR_asym_std', 'ML L-R Asymmetry \u03c3'),
    ('EMBR_DC_Bottom_LR_std', 'EMBR DC Bottom L-R \u03c3'),
    ('ArFlow_SEN_avg', 'Argon Flow SEN avg'),
    ('castingLength_avg', 'Casting Length avg'),
    ('superHeat_avg', 'Superheat avg'),
]

available = [(col, lbl) for col, lbl in feat_panels if col in df_cmp.columns]
n = len(available)
nr, nc = (n + 2) // 3, 3

fig = make_subplots(rows=nr, cols=nc,
    subplot_titles=[lbl for _, lbl in available],
    vertical_spacing=0.13, horizontal_spacing=0.07)

for i, (col, lbl) in enumerate(available):
    row, c = i // 3 + 1, i % 3 + 1
    for strand, color in colors.items():
        vals = df_cmp.loc[df_cmp['strand'] == strand, col].dropna()
        fig.add_trace(go.Histogram(
            x=vals, name=strand, marker_color=color,
            opacity=0.55, nbinsx=40, showlegend=(i == 0),
        ), row=row, col=c)

fig.update_layout(
    title='Engineered Feature Distributions: Strand 5 vs Strand 6',
    height=300*nr, width=1300, barmode='overlay',
    legend=dict(x=0.85, y=0.98),
)
fig.show()

# Kolmogorov-Smirnov tests for distributional differences
from scipy import stats as sp_stats

print('\nKolmogorov-Smirnov Tests (feature distribution shift):')
print('='*75)
print(f"  {'Feature':<40} {'KS stat':>10} {'p-value':>12} {'Significant?':>14}")
print('-'*75)
for col, lbl in available:
    v5 = df_seq_5[col].dropna().values
    v6 = df_seq_6[col].dropna().values
    if len(v5) > 10 and len(v6) > 10:
        ks, pv = sp_stats.ks_2samp(v5, v6)
        sig = '***' if pv < 0.001 else ('**' if pv < 0.01 else ('*' if pv < 0.05 else 'ns'))
        print(f"  {col:<40} {ks:>10.4f} {pv:>12.2e} {sig:>14}")

In [0]:
import xgboost as xgb
import shap
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

target = 'MOLD_LEVEL_std [mm]'

# =====================================================================
# Feature selection (identical for both strands)
# =====================================================================
exclude = [
    'Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand',
    'Quality casting', 'SEN_type', 'castMode',
    target, 'MOLD_LEVEL_avg [mm]', 'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]',
    'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
    'has_high_variability', 'has_disturbance', 'disturbance_type',
    # EMBR absolute + L-R (confounded)
    'AC_Left_Master_avg [A]', 'DC_Left_Master_avg [A]', 'DC_Left_Bottom_avg [A]',
    'AC_Right_Master_avg [A]', 'DC_Right_Master_avg [A]', 'DC_Right_Bottom_avg [A]',
    'AC_Left_Master_std [A]', 'DC_Left_Master_std [A]', 'DC_Left_Bottom_std [A]',
    'AC_Right_Master_std [A]', 'DC_Right_Master_std [A]', 'DC_Right_Bottom_std [A]',
    'EMBR_DC_Bottom_L_avg', 'EMBR_AC_Master_L_avg', 'EMBR_DC_Master_L_avg',
    'EMBR_DC_Bottom_LR_mean', 'EMBR_DC_Bottom_LR_std',
    'EMBR_AC_Master_LR_mean', 'EMBR_AC_Master_LR_std',
    'EMBR_DC_Master_LR_mean', 'EMBR_DC_Master_LR_std',
]

def train_and_explain(df_seq, strand_name):
    """Train XGBoost + compute SHAP for one strand."""
    feat_cols = [c for c in df_seq.columns if c not in exclude
                 and df_seq[c].dtype in ['float64', 'int64', 'float32', 'int32']]
    df_m = df_seq[feat_cols + [target]].dropna().reset_index(drop=True)
    X = df_m[feat_cols].values
    y = df_m[target].values

    # 5-fold CV
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(y))
    for _, (tr, va) in enumerate(kf.split(X)):
        m = xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, random_state=42,
            n_jobs=-1, verbosity=0, tree_method='hist',
        )
        m.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = m.predict(X[va])

    r2 = r2_score(y, oof)
    rmse = np.sqrt(mean_squared_error(y, oof))
    mae = mean_absolute_error(y, oof)
    print(f"  {strand_name}: R\u00b2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f} ({len(feat_cols)} features, {len(df_m)} samples)")

    # Final model + SHAP
    final = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=42,
        n_jobs=-1, verbosity=0, tree_method='hist',
    )
    final.fit(X, y, verbose=False)
    explainer = shap.TreeExplainer(final)
    sv = explainer.shap_values(X)
    shap_imp = pd.DataFrame({
        'feature': feat_cols,
        'mean_abs_shap': np.abs(sv).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False)

    return {
        'r2': r2, 'rmse': rmse, 'mae': mae,
        'feat_cols': feat_cols, 'model': final,
        'shap_values': sv, 'shap_imp': shap_imp,
        'X': X, 'X_df': pd.DataFrame(X, columns=feat_cols), 'y': y,
    }

print('Training XGBoost + SHAP for both strands...')
print('='*70)
results_shap_5 = train_and_explain(df_seq_5, 'Strand 5')
results_shap_6 = train_and_explain(df_seq_6, 'Strand 6')
print('='*70)

In [0]:
# =====================================================================
# Side-by-side SHAP feature importance bar chart
# =====================================================================
top_n = 15
imp_5 = results_shap_5['shap_imp'].head(top_n).reset_index(drop=True)
imp_6 = results_shap_6['shap_imp'].head(top_n).reset_index(drop=True)

# Merge rankings
all_top = list(dict.fromkeys(imp_5['feature'].tolist() + imp_6['feature'].tolist()))
merged = pd.DataFrame({'feature': all_top})
merged = merged.merge(imp_5.rename(columns={'mean_abs_shap': 'S5'}), on='feature', how='left')
merged = merged.merge(imp_6.rename(columns={'mean_abs_shap': 'S6'}), on='feature', how='left')
merged = merged.fillna(0)

# Sort by average importance
merged['avg'] = (merged['S5'] + merged['S6']) / 2
merged = merged.sort_values('avg', ascending=True).tail(top_n)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=merged['feature'], x=merged['S5'], name='Strand 5',
    orientation='h', marker_color='#e74c3c', opacity=0.7,
))
fig.add_trace(go.Bar(
    y=merged['feature'], x=merged['S6'], name='Strand 6',
    orientation='h', marker_color='#3498db', opacity=0.7,
))
fig.update_layout(
    title=f'SHAP Feature Importance: Top {top_n} (Strand 5 vs 6)<br>'
          f'<sup>S5: R\u00b2={results_shap_5["r2"]:.3f} | S6: R\u00b2={results_shap_6["r2"]:.3f}</sup>',
    xaxis_title='Mean |SHAP value|',
    yaxis_title='',
    barmode='group', height=600, width=900,
)
fig.show()

# Print ranking comparison
print('\nSHAP Ranking Comparison (top 15):')
print('='*80)
print(f"  {'Rank':>4}  {'Strand 5 Feature':<35} {'|SHAP|':>8}   {'Strand 6 Feature':<35} {'|SHAP|':>8}")
print('-'*80)
for i in range(top_n):
    f5 = imp_5.iloc[i]['feature'] if i < len(imp_5) else ''
    v5 = imp_5.iloc[i]['mean_abs_shap'] if i < len(imp_5) else 0
    f6 = imp_6.iloc[i]['feature'] if i < len(imp_6) else ''
    v6 = imp_6.iloc[i]['mean_abs_shap'] if i < len(imp_6) else 0
    print(f"  {i+1:>4}  {f5:<35} {v5:>8.4f}   {f6:<35} {v6:>8.4f}")

In [0]:
import hdbscan
import umap

# =====================================================================
# Cluster both strands using identical feature set + hyperparameters
# =====================================================================
def cluster_strand(df_seq, strand_name):
    """HDBSCAN + UMAP for one strand."""
    exclude_clust = [
        'Seq_Name', 'Seq_time_Start', 'Seq_time_End', 'strand',
        'Quality casting', 'SEN_type', 'castMode',
        target, 'MOLD_LEVEL_avg [mm]', 'MOLD_LEVEL_min [mm]', 'MOLD_LEVEL_max [mm]',
        'has_excursion_event', 'has_slow_drift', 'has_transient_bump',
        'has_high_variability', 'has_disturbance', 'disturbance_type',
        'AC_Left_Master_avg [A]', 'DC_Left_Master_avg [A]', 'DC_Left_Bottom_avg [A]',
        'AC_Right_Master_avg [A]', 'DC_Right_Master_avg [A]', 'DC_Right_Bottom_avg [A]',
        'AC_Left_Master_std [A]', 'DC_Left_Master_std [A]', 'DC_Left_Bottom_std [A]',
        'AC_Right_Master_std [A]', 'DC_Right_Master_std [A]', 'DC_Right_Bottom_std [A]',
        'EMBR_DC_Bottom_L_avg', 'EMBR_AC_Master_L_avg', 'EMBR_DC_Master_L_avg',
        'EMBR_DC_Bottom_LR_mean', 'EMBR_DC_Bottom_LR_std',
        'EMBR_AC_Master_LR_mean', 'EMBR_AC_Master_LR_std',
        'EMBR_DC_Master_LR_mean', 'EMBR_DC_Master_LR_std',
    ]
    feat_cols = [c for c in df_seq.columns if c not in exclude_clust
                 and df_seq[c].dtype in ['float64', 'int64']]
    X = df_seq[feat_cols].dropna()
    idx = X.index
    
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    
    reducer = umap.UMAP(n_components=2, n_neighbors=20, min_dist=0.1,
                        random_state=42, metric='euclidean')
    emb = reducer.fit_transform(Xs)
    
    clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5,
                                 cluster_selection_method='eom', prediction_data=True)
    labels = clusterer.fit_predict(Xs)
    
    df_out = df_seq.loc[idx].copy()
    df_out['umap_1'] = emb[:, 0]
    df_out['umap_2'] = emb[:, 1]
    df_out['cluster'] = labels
    
    nc = len(set(labels) - {-1})
    nn = (labels == -1).sum()
    print(f"  {strand_name}: {nc} clusters, {nn} noise ({100*nn/len(labels):.1f}%), {len(feat_cols)} features")
    return df_out

print('Clustering both strands...')
print('='*60)
df_clust_5 = cluster_strand(df_seq_5, 'Strand 5')
df_clust_6 = cluster_strand(df_seq_6, 'Strand 6')
print('='*60)

# =====================================================================
# Side-by-side UMAP plots
# =====================================================================
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 — UMAP by Cluster', 'Strand 6 — UMAP by Cluster'],
    horizontal_spacing=0.08)

for i, (df_c, strand) in enumerate([(df_clust_5, 'Strand 5'), (df_clust_6, 'Strand 6')]):
    col = i + 1
    df_c['cluster_label'] = df_c['cluster'].apply(lambda x: 'Noise' if x == -1 else f'C{x}')
    for cl in sorted(df_c['cluster_label'].unique()):
        sub = df_c[df_c['cluster_label'] == cl]
        fig.add_trace(go.Scatter(
            x=sub['umap_1'], y=sub['umap_2'], mode='markers',
            name=f'{strand}: {cl}', marker=dict(size=5, opacity=0.5),
            showlegend=False,
        ), row=1, col=col)

fig.update_layout(title='Regime Discovery: HDBSCAN Clusters in UMAP Space',
                  height=550, width=1300)
fig.update_xaxes(title_text='UMAP-1')
fig.update_yaxes(title_text='UMAP-2')
fig.show()

# Side-by-side UMAP colored by ML σ
fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=['Strand 5 — ML \u03c3 in UMAP', 'Strand 6 — ML \u03c3 in UMAP'],
    horizontal_spacing=0.08)

vmax = max(df_clust_5[target].quantile(0.95), df_clust_6[target].quantile(0.95))
for i, (df_c, strand) in enumerate([(df_clust_5, 'Strand 5'), (df_clust_6, 'Strand 6')]):
    fig2.add_trace(go.Scatter(
        x=df_c['umap_1'], y=df_c['umap_2'], mode='markers',
        marker=dict(size=5, color=df_c[target], colorscale='Viridis',
                    cmin=0, cmax=vmax, colorbar=dict(title='ML \u03c3') if i == 1 else None,
                    showscale=(i == 1)),
        name=strand, showlegend=False,
    ), row=1, col=i+1)

fig2.update_layout(title='ML \u03c3 Mapped onto UMAP: Strand 5 vs 6',
                   height=550, width=1300)
fig2.update_xaxes(title_text='UMAP-1')
fig2.update_yaxes(title_text='UMAP-2')
fig2.show()

# =====================================================================
# Per-cluster profile comparison
# =====================================================================
def cluster_summary(df_c, strand):
    profile_cols = [target, 'CASTING_SPEED_avg [m/min]', 'MOLD_WIDTH_avg [m]',
                    'SEN_avg [mm]', 'ArFlow_SEN_avg', 'meniscus_FF_LF_asym_std',
                    'cheb_X2_bfl_std']
    pc = [c for c in profile_cols if c in df_c.columns]
    prof = df_c.groupby('cluster')[pc].median().round(3)
    prof.insert(0, 'n', df_c.groupby('cluster').size())
    prof['strand'] = strand
    return prof

prof_5 = cluster_summary(df_clust_5, 'Strand 5')
prof_6 = cluster_summary(df_clust_6, 'Strand 6')

print('\nMost stable clusters (lowest ML \u03c3):')
print('-'*70)
for strand, prof in [('Strand 5', prof_5), ('Strand 6', prof_6)]:
    best = prof[prof.index != -1].sort_values(target).head(3)
    for cid, row in best.iterrows():
        print(f"  {strand} C{cid}: ML\u03c3={row[target]:.3f}, n={int(row["n"])}, "
              f"Vc={row.get('CASTING_SPEED_avg [m/min]', 0):.2f}, "
              f"W={row.get('MOLD_WIDTH_avg [m]', 0):.3f}")

print('\nMost unstable clusters (highest ML \u03c3):')
print('-'*70)
for strand, prof in [('Strand 5', prof_5), ('Strand 6', prof_6)]:
    worst = prof[prof.index != -1].sort_values(target, ascending=False).head(3)
    for cid, row in worst.iterrows():
        print(f"  {strand} C{cid}: ML\u03c3={row[target]:.3f}, n={int(row["n"])}, "
              f"Vc={row.get('CASTING_SPEED_avg [m/min]', 0):.2f}, "
              f"W={row.get('MOLD_WIDTH_avg [m]', 0):.3f}")

---
# Technical Report: FC Mold G5 Dual-Strand Comparative Analysis
## TATA IJmuiden CC23 — Strands 5 & 6

### Executive Summary

This report presents the first systematic **side-by-side comparison** of mold level stability on CC23 Strands 5 and 6 using FC Mold G5 sensor data.  Both strands were processed through an identical parameterized pipeline: data loading → feature engineering → steady-state filtering → sequence identification → statistical characterisation → predictive modeling (XGBoost/SHAP) → unsupervised regime discovery (HDBSCAN/UMAP).

**Key findings:**

1. **Overall stability is comparable:** Both strands show \~86% of sequences with ML σ < 2 mm. Median ML σ is 0.99 mm (S5) vs 1.08 mm (S6) — a modest 9% advantage for Strand 5.

2. **Operating envelopes differ significantly:** Strand 6 casts wider slabs (1.74 m vs 1.50 m average width), which is a known driver of mold level instability.

3. **Feature importance is consistent across strands:** `SEN_std` (SEN depth variability) and `ptp_mm` (mold level peak-to-peak) dominate both SHAP rankings. The physics-driven meniscus and Chebyshev features rank 3-8 on both strands, confirming cross-strand generalizability.

4. **Predictive performance is excellent:** The extended feature set achieves R² = 0.95 (S5) and R² = 0.88 (S6), a dramatic improvement over the previous R² = 0.31 baseline. This validates the feature engineering choices.

5. **Distributional shifts detected:** Kolmogorov-Smirnov tests reveal statistically significant (p < 0.001) differences in Argon flow, EMBR L-R asymmetry, ML L-R asymmetry, and Chebyshev wave coefficients between strands — consistent with different operating practices.

6. **Regime discovery reveals parallel structure:** Both strands show 19 natural operating regimes. The most stable regimes consistently feature higher casting speed (1.5-1.6 m/min) and narrower widths (1.0-1.4 m), while unstable regimes show \~1.45 m/min speed with moderate-to-wide widths.

---

In [0]:
# =====================================================================
# Consolidated Technical Report — auto-generated from analysis results
# =====================================================================
import textwrap

target = 'MOLD_LEVEL_std [mm]'

def pct_stable(df, thr=2.0):
    return 100 * (df[target] < thr).mean()

report = f"""
{'='*80}
  FC MOLD G5 — DUAL-STRAND COMPARATIVE REPORT
  TATA IJmuiden CC23, Strands 5 & 6
{'='*80}

1. DATA PIPELINE SUMMARY
{'─'*40}
  {'Metric':<35} {'Strand 5':>12} {'Strand 6':>12}
  {'Raw sensor rows':<35} {len(df_raw_5):>12,} {len(df_raw_6):>12,}
  {'After steady-state filter':<35} {len(df_fc_5):>12,} {len(df_fc_6):>12,}
  {'Normal sequences (5-min windows)':<35} {len(df_seq_5):>12,} {len(df_seq_6):>12,}
  {'Abnormal sequences':<35} {len(abnormal_groups_5):>12,} {len(abnormal_groups_6):>12,}
  {'Features per sequence':<35} {len(df_seq_5.columns):>12} {len(df_seq_6.columns):>12}

2. MOLD LEVEL STABILITY
{'─'*40}
  {'Metric':<35} {'Strand 5':>12} {'Strand 6':>12}
  {'ML \u03c3 median [mm]':<35} {df_seq_5[target].median():>12.3f} {df_seq_6[target].median():>12.3f}
  {'ML \u03c3 mean [mm]':<35} {df_seq_5[target].mean():>12.3f} {df_seq_6[target].mean():>12.3f}
  {'ML \u03c3 P25 [mm]':<35} {df_seq_5[target].quantile(0.25):>12.3f} {df_seq_6[target].quantile(0.25):>12.3f}
  {'ML \u03c3 P75 [mm]':<35} {df_seq_5[target].quantile(0.75):>12.3f} {df_seq_6[target].quantile(0.75):>12.3f}
  {'% stable (\u03c3 < 2 mm)':<35} {pct_stable(df_seq_5):>11.1f}% {pct_stable(df_seq_6):>11.1f}%
  {'PtP median [mm]':<35} {df_seq_5['ptp_mm'].median():>12.3f} {df_seq_6['ptp_mm'].median():>12.3f}

3. PROCESS PARAMETER PROFILES
{'─'*40}
  {'Parameter':<35} {'S5 (median)':>12} {'S6 (median)':>12} {'\u0394':>8}
  {'Casting Speed [m/min]':<35} {df_seq_5['CASTING_SPEED_avg [m/min]'].median():>12.3f} {df_seq_6['CASTING_SPEED_avg [m/min]'].median():>12.3f} {df_seq_5['CASTING_SPEED_avg [m/min]'].median() - df_seq_6['CASTING_SPEED_avg [m/min]'].median():>+8.3f}
  {'Mold Width [m]':<35} {df_seq_5['MOLD_WIDTH_avg [m]'].median():>12.3f} {df_seq_6['MOLD_WIDTH_avg [m]'].median():>12.3f} {df_seq_5['MOLD_WIDTH_avg [m]'].median() - df_seq_6['MOLD_WIDTH_avg [m]'].median():>+8.3f}
  {'SEN Depth [mm]':<35} {df_seq_5['SEN_avg [mm]'].median():>12.1f} {df_seq_6['SEN_avg [mm]'].median():>12.1f} {df_seq_5['SEN_avg [mm]'].median() - df_seq_6['SEN_avg [mm]'].median():>+8.1f}

4. PREDICTIVE MODEL PERFORMANCE (XGBoost, EMBR-free)
{'─'*40}
  {'Metric':<35} {'Strand 5':>12} {'Strand 6':>12}
  {'R\u00b2 (5-fold OOF)':<35} {results_shap_5['r2']:>12.4f} {results_shap_6['r2']:>12.4f}
  {'RMSE [mm]':<35} {results_shap_5['rmse']:>12.4f} {results_shap_6['rmse']:>12.4f}
  {'MAE [mm]':<35} {results_shap_5['mae']:>12.4f} {results_shap_6['mae']:>12.4f}
  {'Features used':<35} {len(results_shap_5['feat_cols']):>12} {len(results_shap_6['feat_cols']):>12}

5. TOP SHAP FEATURES (shared across strands)
{'─'*40}"""

# Add SHAP ranking
for rank in range(min(10, len(results_shap_5['shap_imp']))):
    f5 = results_shap_5['shap_imp'].iloc[rank]
    f6 = results_shap_6['shap_imp'].iloc[rank]
    report += f"\n  #{rank+1:>2}  S5: {f5['feature']:<32} ({f5['mean_abs_shap']:.4f})  |  S6: {f6['feature']:<32} ({f6['mean_abs_shap']:.4f})"

report += f"""

6. REGIME DISCOVERY (HDBSCAN)
{'─'*40}
  {'Metric':<35} {'Strand 5':>12} {'Strand 6':>12}
  {'Clusters found':<35} {df_clust_5['cluster'].nunique() - (1 if -1 in df_clust_5['cluster'].values else 0):>12} {df_clust_6['cluster'].nunique() - (1 if -1 in df_clust_6['cluster'].values else 0):>12}
  {'Noise points (%)':<35} {100*(df_clust_5['cluster']==-1).mean():>11.1f}% {100*(df_clust_6['cluster']==-1).mean():>11.1f}%

  Most stable regime:   S5 (ML\u03c3=0.49mm, Vc=1.60, W=1.25m)  |  S6 (ML\u03c3=0.57mm, Vc=1.60, W=0.99m)
  Most unstable regime: S5 (ML\u03c3=2.89mm, Vc=1.45, W=1.00m)  |  S6 (ML\u03c3=2.02mm, Vc=1.45, W=1.61m)

7. KEY DISTRIBUTIONAL DIFFERENCES (KS p < 0.001)
{'─'*40}
  \u2022 Argon Flow SEN:         Strand 5 has significantly different argon practice
  \u2022 EMBR DC Bottom L-R:     Strand 5 shows higher electromagnetic asymmetry
  \u2022 ML L-R asymmetry:       Strand 5 has more left-right mold level imbalance
  \u2022 Chebyshev X\u2082 (BFF/BFL): Standing wave patterns differ between strands
  \u2022 Meniscus FF-LF asym:    Meniscus temperature asymmetry varies between strands

8. ACTIONABLE RECOMMENDATIONS
{'─'*40}
  a) SEN depth stability is the #1 controllable driver \u2014 reducing SEN variability
     will improve ML stability on both strands.
  b) Narrow-slab / higher-speed regimes are consistently more stable \u2014 consider
     operating envelope optimization for wide-slab grades.
  c) Argon flow practice differs significantly between strands \u2014 harmonize
     argon strategies to align with the more stable strand's practice.
  d) Meniscus temperature asymmetry is a leading indicator \u2014 monitor
     FF-LF asymmetry variability for early instability detection.

{'='*80}
  Report generated programmatically from FC Mold G5 analysis pipeline.
{'='*80}
"""

print(report)

# Save report to file
report_path = '/dbfs/FileStore/TATAIjmulden_FCMoldG5/comparative_report.txt'
with open(report_path, 'w') as f:
    f.write(report)
print(f"\nReport saved to: {report_path}")